# SurfaceGuard AI: Industrial Steel Surface Defect Detection

**Project:** Deep Learning for Automated Quality Inspection of Steel Manufacturing Surfaces  
**Objective:** Develop and compare three semantic segmentation architectures for pixel-level detection of four defect classes (A, B, C, D) on industrial steel sheets.  
**Competition Metric:** Composite Score = 0.70 * Foreground Dice + 0.20 * FP Silence Score + 0.10 * Empty Accuracy  

**Models Compared:**
1. **ScratchUNet** -- Baseline U-Net trained entirely from scratch
2. **ResAttentionUNet** -- State-of-the-art scratch segmenter with Residual scSE blocks, ASPP bottleneck, and Attention Gates
3. **PretrainedResNet34UNet** -- Transfer learning benchmark using ImageNet-pretrained ResNet-34 encoder with 2-phase progressive fine-tuning

---

## 1. Environment Setup and Dependencies

All required libraries are installed and imported below. This notebook is entirely self-contained
and does not depend on any external project source files.

In [ ]:
import os
import sys

# Configure Linux GNU C Library (glibc) memory allocator to prevent multi-arena heap fragmentation
# By default, glibc spawns 8 * num_cores (32 arenas), which hoarded 28 GB of VmData across epochs.
# MALLOC_ARENA_MAX=2 strictly limits glibc to 2 arenas, and 64KB thresholds force immediate munmap.
import ctypes
try:
    libc = ctypes.CDLL("libc.so.6")
    # M_MMAP_THRESHOLD = -3, M_TRIM_THRESHOLD = -1, M_ARENA_MAX = -8
    libc.mallopt(-3, 65536)
    libc.mallopt(-1, 65536)
    libc.mallopt(-8, 2)
except Exception:
    pass

os.environ["MALLOC_ARENA_MAX"] = "2"
os.environ["MALLOC_TRIM_THRESHOLD_"] = "65536"
os.environ["MALLOC_MMAP_THRESHOLD_"] = "65536"

import gc
import cv2
import time
import math
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.colors import ListedColormap
from collections import Counter, OrderedDict

# Prevent OpenCV internal thread pool leakage in multi-worker DataLoaders
cv2.setNumThreads(0)
try:
    cv2.ocl.setUseOpenCL(False)
except Exception:
    pass

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

import albumentations as A
from tqdm import tqdm

try:
    import wandb
except ImportError:
    wandb = None

warnings.filterwarnings('ignore')
print(f"PyTorch version: {torch.__version__} | OpenCV version: {cv2.__version__}")
print(f"glibc memory allocator tuned: MALLOC_ARENA_MAX=2 (prevents 28 GB VmData arena bloat)")


## 2. Global Configuration

Central configuration class that governs all hyperparameters, paths, and reproducibility settings.
All random seeds are fixed to ensure deterministic, reproducible results across runs.

In [ ]:
class CFG:
    # -- Reproducibility --
    SEED = 42

    # -- Data Paths (auto-resolved for Kaggle / local environments) --
    DATA_CSV = 'data/train.csv'
    IMAGE_DIR = 'data/train'
    SAMPLE_SUB_CSV = 'data/sample_submission.csv'
    TEST_IMAGE_DIR = 'data/test'

    # -- Image Dimensions (native steel sheet aspect ratio: 128 x 800) --
    IMG_HEIGHT = 128
    IMG_WIDTH = 800
    IN_CHANNELS = 3
    NUM_CLASSES = 4

    # -- Training Hyperparameters --
    BATCH_SIZE = 16
    EPOCHS = 40
    LR = 3e-4
    WEIGHT_DECAY = 1e-4
    VAL_SIZE = 0.20
    NUM_WORKERS = 0

    # -- Multi-GPU DataParallel Configuration --
    # Set to False to train on single GPU (cuda:0). Disabling nn.DataParallel completely
    # eliminates POSIX worker thread spawning, cross-GPU PCI-e scatter/gather buffers,
    # thread-local autocast mismatches, and the ~2 GB/epoch host RAM accumulation.
    # Single T4 (16 GB VRAM) easily accommodates batch size 16 (only ~3.5 GB required).
    USE_DATA_PARALLEL = False
    RESUME_IF_CHECKPOINT_EXISTS = True

    # -- Defect Class Names --
    DEFECT_CLASSES = ['defect_a', 'defect_b', 'defect_c', 'defect_d']
    DEFECT_LABELS = ['Defect A (Inclusions/Pitting)', 'Defect B (Edge Cracks/Tears)',
                     'Defect C (Crazing/Scaling)', 'Defect D (Surface Patches)']

    # -- W&B Configuration --
    WANDB_PROJECT = '24F2004781-t32026'
    WANDB_ENTITY = 'dineshwar-developer-indian-institute-of-technology'

    # -- Model Checkpoint & W&B Download Settings --
    # Set to True to allow downloading pre-trained artifacts from W&B
    # Set to False to strictly train models from scratch inside the notebook
    USE_WANDB_CHECKPOINTS = True

    # Set to True to force re-training all models even if checkpoints/artifacts exist
    FORCE_RETRAIN = True

    # Artifact Candidate Names for Each Architecture
    MODEL_ARTIFACTS = {
        'SurfaceDefectClassifier': {
            'artifact': 'classifier-best',
            'default_version': 'latest',
            'fallback_artifacts': [
                'surfaceguard-stage1-classifier:latest',
                'best_classifier:latest',
                'classifier-best:v0'
            ],
            'local_fallback': 'models/best_classifier.pth'
        },
        'ScratchUNet': {
            'artifact': 'scratch-unet-best',
            'default_version': 'latest',
            'fallback_artifacts': [
                'res-attention-unet-scratch-v1:latest',
                'scratch-unet-best:v0'
            ],
            'local_fallback': 'models/best_scratch_unet.pth'
        },
        'ResAttentionUNet': {
            'artifact': 'resattunet-best',
            'default_version': 'latest',
            'fallback_artifacts': [
                'res-attention-unet-fold1:latest',
                'resattentionunet-best:latest',
                'resattunet-best:v0'
            ],
            'local_fallback': 'models/best_resattunet.pth'
        },
        'PretrainedResNet34UNet': {
            'artifact': 'pretrained-resnet34-best',
            'default_version': 'latest',
            'fallback_artifacts': [
                'resnet34-best:latest',
                'pretrained-resnet34-best:v0'
            ],
            'local_fallback': 'models/best_pretrained_resnet34.pth'
        },
    }

    # -- Per-Class Decision Thresholds (empirically calibrated) --
    CLASS_THRESHOLDS = {0: 0.32, 1: 0.35, 2: 0.30, 3: 0.30}

    # -- Post-Processing Area Constraints --
    MIN_AREAS = {0: 60, 1: 120, 2: 350, 3: 300}
    MAX_AREAS = {0: 25000, 1: 15000, 2: 35000, 3: 35000}

    # -- Classifier Gating Thresholds --
    CLS_GATES = {0: 0.25, 1: 0.20, 2: 0.35, 3: 0.35}


def seed_everything(seed=CFG.SEED):
    """Fix all random number generators for full reproducibility."""
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    os.environ['PYTHONHASHSEED'] = str(seed)

seed_everything()

# Auto-resolve data paths for Kaggle environment
def resolve_paths():
    """Resolve dataset paths across Kaggle, Colab, or local environments."""
    csv_candidates = [
        '/kaggle/input/competitions/industrial-surface-defect-detection/public/train.csv',
        '/kaggle/input/industrial-surface-defect-detection/public/train.csv',
        '/kaggle/input/industrial-surface-defect-detection/train.csv',
        'data/train.csv', '../data/train.csv',
    ]
    img_candidates = [
        '/kaggle/input/competitions/industrial-surface-defect-detection/public/train',
        '/kaggle/input/industrial-surface-defect-detection/public/train',
        '/kaggle/input/industrial-surface-defect-detection/train',
        'data/train', '../data/train',
    ]
    sub_candidates = [
        '/kaggle/input/competitions/industrial-surface-defect-detection/public/sample_submission.csv',
        '/kaggle/input/industrial-surface-defect-detection/public/sample_submission.csv',
        '/kaggle/input/industrial-surface-defect-detection/sample_submission.csv',
        'data/sample_submission.csv', '../data/sample_submission.csv',
    ]
    test_candidates = [
        '/kaggle/input/competitions/industrial-surface-defect-detection/public/test',
        '/kaggle/input/industrial-surface-defect-detection/public/test',
        '/kaggle/input/industrial-surface-defect-detection/test',
        'data/test', '../data/test',
    ]

    for c in csv_candidates:
        if os.path.exists(c):
            CFG.DATA_CSV = c
            break
    for c in img_candidates:
        if os.path.exists(c):
            CFG.IMAGE_DIR = c
            break
    for c in sub_candidates:
        if os.path.exists(c):
            CFG.SAMPLE_SUB_CSV = c
            break
    for c in test_candidates:
        if os.path.exists(c):
            CFG.TEST_IMAGE_DIR = c
            break

    print(f"Data CSV: {CFG.DATA_CSV}")
    print(f"Image Dir: {CFG.IMAGE_DIR}")
    print(f"Sample Submission: {CFG.SAMPLE_SUB_CSV}")
    print(f"Test Image Dir: {CFG.TEST_IMAGE_DIR}")

resolve_paths()

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"\nCompute Device: {DEVICE}")


## 3. Core Utility Functions

Run-Length Encoding (RLE) conversion functions used throughout the pipeline for mask serialization
and deserialization. RLE is the standard format for competition submissions.

In [ ]:
def rle_to_mask(rle_string, height=128, width=800):
    """Decode a run-length encoded string into a 2D binary mask.

    Vectorized implementation using NumPy C-level string parsing (np.fromstring)
    to completely eliminate Python string/int object allocations and prevent
    heap fragmentation across thousands of dataset batches.

    Args:
        rle_string: Space-separated string of start-length pairs.
        height: Image height in pixels.
        width: Image width in pixels.

    Returns:
        Binary mask of shape (height, width) with dtype uint8.
    """
    if pd.isna(rle_string) or not str(rle_string).strip():
        return np.zeros((height, width), dtype=np.uint8)
    try:
        rle_values = np.fromstring(str(rle_string), sep=' ', dtype=np.int32)
        if len(rle_values) == 0:
            return np.zeros((height, width), dtype=np.uint8)
        starts = rle_values[0::2] - 1
        lengths = rle_values[1::2]
        mask_1d = np.zeros(height * width, dtype=np.uint8)
        for s, l in zip(starts, lengths):
            mask_1d[s:s + l] = 1
        return mask_1d.reshape((height, width), order='F')
    except Exception:
        return np.zeros((height, width), dtype=np.uint8)


def mask_to_rle(mask):
    """Encode a 2D binary mask into a run-length encoded string.

    Args:
        mask: 2D numpy array with values in {0, 1}.

    Returns:
        RLE string in 1-indexed, column-major (Fortran) order.
    """
    pixels = mask.flatten(order='F')
    pixels = np.concatenate([[0], pixels, [0]])
    runs = np.where(pixels[1:] != pixels[:-1])[0] + 1
    runs[1::2] -= runs[::2]
    return ' '.join(str(x) for x in runs)


# Visualization color palette for the four defect classes
DEFECT_COLORS = {
    0: (255, 0, 0),      # Defect A: Red
    1: (0, 255, 0),      # Defect B: Green
    2: (0, 100, 255),    # Defect C: Blue
    3: (255, 165, 0),    # Defect D: Orange
}

def overlay_masks_on_image(image, mask_4ch, alpha=0.45):
    """Overlay a 4-channel defect mask onto a steel sheet image.

    Args:
        image: RGB image array of shape (H, W, 3).
        mask_4ch: Binary mask of shape (4, H, W) or (H, W, 4).
        alpha: Transparency factor for the overlay.

    Returns:
        Blended RGB image with colored defect overlays.
    """
    if mask_4ch.ndim == 3 and mask_4ch.shape[0] == 4:
        mask_4ch = mask_4ch.transpose(1, 2, 0)
    overlay = image.copy().astype(np.float32)
    for c in range(4):
        if mask_4ch[:, :, c].sum() > 0:
            color = np.array(DEFECT_COLORS[c], dtype=np.float32)
            region = mask_4ch[:, :, c] > 0
            overlay[region] = overlay[region] * (1 - alpha) + color * alpha
    return overlay.astype(np.uint8)


## 4. Weights and Biases Authentication

Authenticate with Weights and Biases for experiment tracking. The API key is retrieved from
Kaggle secrets or environment variables.

In [ ]:
if wandb is not None:
    try:
        from kaggle_secrets import UserSecretsClient
        secret = UserSecretsClient()
        wandb_key = secret.get_secret("WANDB_API_KEY")
        wandb.login(key=wandb_key)
        print("Successfully authenticated with Weights & Biases via Kaggle Secrets.")
    except Exception:
        try:
            wandb.login()
            print("Authenticated with Weights & Biases via stored credentials.")
        except Exception as e:
            print(f"W&B authentication skipped: {e}")
            wandb = None
else:
    print("W&B not installed. Experiment tracking is disabled.")

def init_wandb(run_name, config=None):
    """Safely initialize or reinitialize a Weights & Biases run.

    Args:
        run_name: Display name for the run in W&B dashboard.
        config: Optional configuration dictionary.
    Returns:
        wandb.Run instance if successful, else None.
    """
    if wandb is None:
        return None
    try:
        run = wandb.init(
            project=CFG.WANDB_PROJECT,
            entity=CFG.WANDB_ENTITY,
            name=run_name,
            config=config,
            reinit=True
        )
        return run
    except Exception as e:
        print(f"[W&B] Notice: wandb.init failed ({e}). Proceeding without remote logging.")
        return None

def fetch_model_from_wandb(artifact_name, version='latest', project=CFG.WANDB_PROJECT,
                           entity=CFG.WANDB_ENTITY, target_dir='models', fallback_artifacts=None):
    """Download a model checkpoint artifact directly from Weights & Biases.

    Args:
        artifact_name: Name of the W&B model artifact (e.g., 'scratch-unet-best').
        version: Specific version tag (e.g., 'latest', 'v0', 'v1'). Defaults to 'latest'.
        project: W&B project name.
        entity: W&B entity/username.
        target_dir: Local destination directory for downloaded weights.
        fallback_artifacts: Optional list of fallback artifact identifiers.

    Returns:
        Local path to the downloaded .pth checkpoint file.
    """
    if wandb is None:
        print(f"[W&B] wandb not available. Cannot fetch artifact '{artifact_name}'.")
        return None

    os.makedirs(target_dir, exist_ok=True)
    try:
        api = wandb.Api()
    except Exception as e:
        print(f"[W&B] Could not initialize W&B API: {e}")
        return None

    tag = f":{version}" if not str(version).startswith(":") else version
    primary = f"{artifact_name}{tag}"
    candidates = [primary]
    if fallback_artifacts:
        candidates.extend(fallback_artifacts)

    for cand in candidates:
        if ":" not in cand:
            cand = f"{cand}:latest"
        full_paths = [
            f"{entity}/{project}/{cand}" if entity else f"{project}/{cand}",
            f"{project}/{cand}"
        ]
        for p in full_paths:
            try:
                print(f"[W&B] Fetching artifact: '{p}'...")
                art = api.artifact(p, type='model')
                dl_dir = art.download(root=target_dir)
                pths = [f for f in os.listdir(dl_dir) if f.endswith('.pth')]
                if pths:
                    ckpt_path = os.path.join(dl_dir, pths[0])
                    print(f"[W&B] Successfully downloaded '{art.name}' (Version: {art.version}) -> {ckpt_path}")
                    return ckpt_path
            except Exception:
                continue

    # Fallback: Search logged artifacts across recent runs matching display name
    try:
        clean_name = artifact_name.split(':')[0]
        runs = api.runs(f"{entity}/{project}" if entity else project)
        for r in runs:
            if clean_name in r.name:
                for a in r.logged_artifacts():
                    if a.type == 'model':
                        dl_dir = a.download(root=target_dir)
                        pths = [f for f in os.listdir(dl_dir) if f.endswith('.pth')]
                        if pths:
                            ckpt_path = os.path.join(dl_dir, pths[0])
                            print(f"[W&B] Resolved artifact from run '{r.name}': {ckpt_path}")
                            return ckpt_path
    except Exception as e:
        print(f"[W&B] Run search note: {e}")

    print(f"[W&B] Warning: Could not download artifact for '{artifact_name}'.")
    return None


---

## 5. Exploratory Data Analysis (EDA)

A thorough exploration of the Severstal steel surface defect dataset to understand class distributions,
defect morphology, co-occurrence patterns, and the physical characteristics that inform our model design.

### 5.1 Dataset Overview and Class Distribution

In [ ]:
# Load the raw training annotations
df = pd.read_csv(CFG.DATA_CSV)
print(f"Raw annotation rows: {len(df):,}")
print(f"Columns: {list(df.columns)}")
print(f"\nFirst 10 rows:")
df.head(10)

In [ ]:
## Count unique images and defect annotations
# total_images = df['image_id'].nunique()
# total_annotations = df['mask_rle'].notna().sum()
# total_empty = len(df) - total_annotations
# 
# print(f"Total unique steel sheet images: {total_images:,}")
# print(f"Total defect annotations (non-empty RLE): {total_annotations:,}")
# print(f"Total empty annotation slots: {total_empty:,}")
# print(f"Annotation density: {total_annotations / len(df) * 100:.1f}%")

### 5.2 Class Frequencies and Clean Sheet Mass

The dataset exhibits severe class imbalance. The majority of steel sheets are completely defect-free
(clean sheets), and among defective images, certain defect types are extremely rare. This imbalance
directly motivates our use of Focal Loss, defect-aware weighted sampling, and per-channel loss weighting.

In [ ]:
## Pivot to one row per image for analysis
# pivoted = df.pivot(index='image_id', columns='defect_type', values='mask_rle').reset_index()
# 
## Count images with each defect type
# class_counts = {}
# for cls in CFG.DEFECT_CLASSES:
#     count = pivoted[cls].notna().sum()
#     class_counts[cls] = count
#     print(f"{cls}: {count:,} images ({count/total_images*100:.1f}%)")
# 
## Clean sheet count: images with zero defects across all four channels
# has_any_defect = pivoted[CFG.DEFECT_CLASSES].notna().any(axis=1)
# clean_count = (~has_any_defect).sum()
# defective_count = has_any_defect.sum()
# clean_sheet_mass = clean_count / total_images * 100
# 
# print(f"\nClean sheets (zero defects): {clean_count:,} ({clean_sheet_mass:.1f}%)")
# print(f"Defective sheets (>= 1 defect): {defective_count:,} ({100 - clean_sheet_mass:.1f}%)")
# 
## Visualization: Class frequency bar chart
# fig, axes = plt.subplots(1, 2, figsize=(14, 5))
# 
## Bar chart of class frequencies
# colors = ['#e74c3c', '#2ecc71', '#3498db', '#e67e22']
# bars = axes[0].bar(CFG.DEFECT_LABELS, list(class_counts.values()), color=colors, edgecolor='black', linewidth=0.8)
# axes[0].set_title('Defect Class Frequency Distribution', fontsize=13, fontweight='bold')
# axes[0].set_ylabel('Number of Images', fontsize=11)
# axes[0].set_xlabel('Defect Class', fontsize=11)
# for bar, count in zip(bars, class_counts.values()):
#     axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 15,
#                  f'{count:,}', ha='center', va='bottom', fontsize=10, fontweight='bold')
# axes[0].tick_params(axis='x', rotation=15)
# 
## Pie chart: clean vs defective
# pie_labels = ['Clean Sheets', 'Defective Sheets']
# pie_sizes = [clean_count, defective_count]
# pie_colors = ['#95a5a6', '#e74c3c']
# wedges, texts, autotexts = axes[1].pie(
#     pie_sizes, labels=pie_labels, colors=pie_colors,
#     autopct='%1.1f%%', startangle=90, textprops={'fontsize': 11}
# )
# axes[1].set_title('Clean vs. Defective Sheet Distribution', fontsize=13, fontweight='bold')
# 
# plt.tight_layout()
# plt.show()

### 5.3 Defect Co-occurrence and Combination Signatures

Understanding which defect types co-occur on the same steel sheet is critical for designing the
stratified train/validation split and for tuning the multi-label classification head.

In [ ]:
## Generate combination signatures
# def make_signature(row):
#     """Create a string signature representing which defect types are present on this image."""
#     active = [c for c in CFG.DEFECT_CLASSES if pd.notna(row.get(c)) and str(row.get(c)).strip() != '']
#     return '+'.join(sorted(active)) if active else 'clean'
# 
# pivoted['signature'] = pivoted.apply(make_signature, axis=1)
# sig_counts = pivoted['signature'].value_counts()
# 
# print("Defect Combination Signatures (sorted by frequency):")
# print("-" * 55)
# for sig, count in sig_counts.items():
#     pct = count / total_images * 100
#     print(f"  {sig:30s} : {count:5,} images ({pct:5.1f}%)")
# 
## Visualization
# fig, ax = plt.subplots(figsize=(12, 5))
# sig_df = sig_counts.reset_index()
# sig_df.columns = ['Signature', 'Count']
# bar_colors = ['#95a5a6' if s == 'clean' else '#3498db' for s in sig_df['Signature']]
# bars = ax.barh(sig_df['Signature'][::-1], sig_df['Count'][::-1], color=bar_colors[::-1], edgecolor='black', linewidth=0.5)
# ax.set_xlabel('Number of Images', fontsize=11)
# ax.set_title('Defect Combination Signatures Across Training Set', fontsize=13, fontweight='bold')
# for bar, count in zip(bars, sig_df['Count'][::-1]):
#     ax.text(bar.get_width() + 5, bar.get_y() + bar.get_height()/2,
#             f'{count:,}', va='center', fontsize=9)
# plt.tight_layout()
# plt.show()

### 5.4 Pixel-Level Defect Area Statistics and Aspect Ratio Analysis

Each defect type exhibits distinct morphological characteristics grounded in industrial steel metallurgy:
- Defect A (Inclusions / Pitting): Small, isolated pinhole clusters with moderate aspect ratio (Median H/W: 1.68).
- Defect B (Edge Cracks / Tears): Extremely tall, narrow tears concentrated along strip edges (Median H/W: 8.45).
- Defect C (Crazing / Scaling): Widespread multi-scale surface crazing networks with long vertical elongation tails (Median H/W: 3.20).
- Defect D (Surface Patches): Broad diffuse texture anomalies and oxidation stains spreading horizontally (Median H/W: 0.75, wider than tall).

Understanding these empirical shapes informs our post-processing thresholds and architecture design (e.g., anisotropic strip convolutions in ASPP).

In [ ]:
## Compute pixel-level statistics for each defect class
# area_stats = {cls: [] for cls in CFG.DEFECT_CLASSES}
# aspect_ratios = {cls: [] for cls in CFG.DEFECT_CLASSES}
# 
# sample_images = pivoted[pivoted['signature'] != 'clean'].head(500)
# 
# for _, row in tqdm(sample_images.iterrows(), total=len(sample_images), desc="Computing defect morphology"):
#     img_id = str(row['image_id'])
#     for c, cls in enumerate(CFG.DEFECT_CLASSES):
#         rle = row.get(cls)
#         if pd.notna(rle) and str(rle).strip() != '':
#             mask = rle_to_mask(str(rle), height=CFG.IMG_HEIGHT, width=CFG.IMG_WIDTH)
#             total_area = int(mask.sum())
#             if total_area > 0:
#                 area_stats[cls].append(total_area)
#                 # Connected component analysis for aspect ratio
#                 num_labels, labels, stats, _ = cv2.connectedComponentsWithStats(mask, connectivity=8)
#                 for label_idx in range(1, num_labels):
#                     comp_h = stats[label_idx, cv2.CC_STAT_HEIGHT]
#                     comp_w = stats[label_idx, cv2.CC_STAT_WIDTH]
#                     if comp_w > 0:
#                         aspect_ratios[cls].append(comp_h / comp_w)
# 
## Print summary statistics
# print("Per-Class Defect Area Statistics (in pixels):")
# print("=" * 70)
# for cls, label in zip(CFG.DEFECT_CLASSES, CFG.DEFECT_LABELS):
#     areas = area_stats[cls]
#     if areas:
#         print(f"  {label}:")
#         print(f"    Count: {len(areas):,} | Mean: {np.mean(areas):,.0f} | "
#               f"Median: {np.median(areas):,.0f} | Min: {np.min(areas):,} | Max: {np.max(areas):,}")
#     else:
#         print(f"  {label}: No annotations found in sample")
# 
## Visualization: Defect area distributions (log scale)
# fig, axes = plt.subplots(2, 2, figsize=(14, 10))
# colors = ['#e74c3c', '#2ecc71', '#3498db', '#e67e22']
# for idx, (cls, label) in enumerate(zip(CFG.DEFECT_CLASSES, CFG.DEFECT_LABELS)):
#     ax = axes[idx // 2][idx % 2]
#     areas = area_stats[cls]
#     if areas:
#         ax.hist(areas, bins=50, color=colors[idx], alpha=0.7, edgecolor='black', linewidth=0.5)
#         ax.axvline(np.median(areas), color='black', linestyle='--', linewidth=1.5, label=f'Median: {np.median(areas):,.0f}')
#         ax.set_title(f'{label} - Area Distribution', fontsize=12, fontweight='bold')
#         ax.set_xlabel('Defect Area (pixels)', fontsize=10)
#         ax.set_ylabel('Frequency', fontsize=10)
#         ax.legend(fontsize=9)
#     else:
#         ax.set_title(f'{label} - No Data', fontsize=12)
# plt.suptitle('Defect Area Distributions Across Classes', fontsize=14, fontweight='bold', y=1.01)
# plt.tight_layout()
# plt.show()

In [ ]:
## Aspect Ratio Distribution per Defect Class
# fig, axes = plt.subplots(2, 2, figsize=(14, 10))
# colors = ['#e74c3c', '#2ecc71', '#3498db', '#e67e22']
# for idx, (cls, label) in enumerate(zip(CFG.DEFECT_CLASSES, CFG.DEFECT_LABELS)):
#     ax = axes[idx // 2][idx % 2]
#     ratios = aspect_ratios[cls]
#     if ratios:
#         clipped = [min(r, 15.0) for r in ratios]
#         ax.hist(clipped, bins=50, color=colors[idx], alpha=0.7, edgecolor='black', linewidth=0.5)
#         median_ar = np.median(ratios)
#         ax.axvline(median_ar, color='black', linestyle='--', linewidth=1.5,
#                    label=f'Median H/W: {median_ar:.2f}')
#         ax.axvline(1.0, color='gray', linestyle=':', linewidth=1.0, label='Square (H/W=1)')
#         ax.set_title(f'{label} - Aspect Ratio (H/W)', fontsize=12, fontweight='bold')
#         ax.set_xlabel('Height / Width Ratio', fontsize=10)
#         ax.set_ylabel('Frequency', fontsize=10)
#         ax.legend(fontsize=9)
#     else:
#         ax.set_title(f'{label} - No Data', fontsize=12)
# plt.suptitle('Connected Component Aspect Ratios: Defect Shape Anisotropy', fontsize=14, fontweight='bold', y=1.01)
# plt.tight_layout()
# plt.show()
# 
# print("\nKey Morphological Observations:")
# print("  - Defect A (Inclusions/Pitting): Compact pinhole clusters with moderate aspect ratio (Median H/W: 1.68)")
# print("  - Defect B (Edge Cracks/Tears): Highly anisotropic tall vertical tears along strip edges (Median H/W: 8.45)")
# print("  - Defect C (Crazing/Scaling): Widespread multi-scale crazing networks with long vertical tails (Median H/W: 3.20)")
# print("  - Defect D (Surface Patches): Broad diffuse areas with horizontal spread, wider than tall (Median H/W: 0.75)")


### 5.5 Sample Defective Sheet Visualization

Visual inspection of actual defective steel sheets with their ground-truth segmentation masks overlaid.
Each defect class is color-coded: Red (A), Green (B), Blue (C), Orange (D).

In [ ]:
## Visualize sample defective images with their ground truth masks
# defective_samples = pivoted[pivoted['signature'] != 'clean'].sample(8, random_state=CFG.SEED)
# 
# fig, axes = plt.subplots(4, 2, figsize=(18, 14))
# legend_patches = [
#     mpatches.Patch(color=np.array(DEFECT_COLORS[i])/255, label=CFG.DEFECT_LABELS[i])
#     for i in range(4)
# ]
# 
# for idx, (_, row) in enumerate(defective_samples.iterrows()):
#     ax = axes[idx // 2][idx % 2]
#     img_id = str(row['image_id'])
#     img_path = os.path.join(CFG.IMAGE_DIR, img_id if img_id.endswith('.jpg') else img_id + '.jpg')
#     image = cv2.imread(img_path)
#     if image is None:
#         ax.set_title(f"Image not found: {img_id}")
#         continue
#     image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
#     image = cv2.resize(image, (CFG.IMG_WIDTH, CFG.IMG_HEIGHT))
# 
#     mask_4ch = np.zeros((4, CFG.IMG_HEIGHT, CFG.IMG_WIDTH), dtype=np.uint8)
#     active_classes = []
#     for c, cls in enumerate(CFG.DEFECT_CLASSES):
#         rle = row.get(cls)
#         if pd.notna(rle) and str(rle).strip() != '':
#             mask_4ch[c] = rle_to_mask(str(rle), height=CFG.IMG_HEIGHT, width=CFG.IMG_WIDTH)
#             active_classes.append(cls.replace('defect_', '').upper())
# 
#     overlay = overlay_masks_on_image(image, mask_4ch)
#     ax.imshow(overlay)
#     ax.set_title(f'{img_id} | Active: {", ".join(active_classes)}', fontsize=10, fontweight='bold')
#     ax.axis('off')
# 
# fig.legend(handles=legend_patches, loc='lower center', ncol=4, fontsize=11, frameon=True)
# plt.suptitle('Sample Defective Steel Sheets with Ground Truth Segmentation Masks',
#              fontsize=14, fontweight='bold')
# plt.tight_layout(rect=[0, 0.04, 1, 0.97])
# plt.show()

---

## 6. Data Pipeline

### 6.1 Dataset Class and Stratified Splitting

The dataset class handles RLE decoding, synchronized image-mask augmentations via Albumentations,
and tensor conversion. The stratified split ensures that rare defect combinations (like isolated
Defect B) are proportionally represented in both training and validation sets.

In [ ]:
class SurfaceDefectDataset(Dataset):
    """PyTorch Dataset for loading steel surface images with multi-class defect masks.

    Memory-optimized implementation: pre-extracts image IDs and RLE strings into flat
    NumPy arrays/lists during initialization. Completely eliminates Pandas DataFrame
    indexing (.iloc) inside __getitem__, preventing Linux multiprocessing Copy-on-Write (CoW)
    page duplication and memory leaks across training epochs.
    """
    def __init__(self, df, image_dir, transforms=None):
        self.image_dir = image_dir
        self.transforms = transforms
        if 'defect_type' in df.columns:
            pivoted = df.pivot(
                index='image_id', columns='defect_type', values='mask_rle'
            ).reset_index()
        else:
            pivoted = df.copy().reset_index(drop=True)

        # Pre-extract flat arrays: zero Pandas operations in __getitem__
        self.image_ids = pivoted['image_id'].astype(str).values
        self.rles = {
            c: np.array([str(x) if (pd.notna(x) and str(x).strip() != '') else '' for x in pivoted[c].values], dtype=object)
            for c in CFG.DEFECT_CLASSES
        }

    def __len__(self):
        return len(self.image_ids)

    def __getitem__(self, idx):
        image_id = self.image_ids[idx]
        if not image_id.endswith('.jpg'):
            image_path = os.path.join(self.image_dir, image_id + '.jpg')
        else:
            image_path = os.path.join(self.image_dir, image_id)

        image = cv2.imread(image_path)
        if image is None:
            raise FileNotFoundError(f"Image not found at path: {image_path}")
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        h, w, _ = image.shape

        # Build 4-channel binary mask: shape (H, W, 4)
        mask = np.zeros((h, w, len(CFG.DEFECT_CLASSES)), dtype=np.uint8)
        for c, defect in enumerate(CFG.DEFECT_CLASSES):
            rle_str = self.rles[defect][idx]
            if rle_str:
                mask[:, :, c] = rle_to_mask(rle_str, height=h, width=w)

        # Synchronized augmentations applied to both image and mask
        if self.transforms:
            augmented = self.transforms(image=image, mask=mask)
            image = augmented['image']
            mask = augmented['mask']

        # Convert to PyTorch tensors (zero-copy from contiguous NumPy buffer)
        image_np = np.ascontiguousarray(image.transpose(2, 0, 1), dtype=np.float32) / 255.0
        mask_np = np.ascontiguousarray(mask.transpose(2, 0, 1), dtype=np.float32)
        return torch.from_numpy(image_np), torch.from_numpy(mask_np)


def get_stratified_split(df, val_size=0.20, random_state=42):
    """Create an 80/20 train/validation split stratified by multi-label defect combination signature.

    This ensures that rare defect types (especially Defect B, which appears in only a small fraction
    of images) are proportionally represented in both the training and validation sets.

    Returns:
        train_df, val_df in pivoted format (1 row per unique image).
    """
    if 'defect_type' in df.columns:
        pivoted = df.pivot(
            index='image_id', columns='defect_type', values='mask_rle'
        ).reset_index()
    else:
        pivoted = df.copy()

    def make_sig(row):
        active = [c for c in CFG.DEFECT_CLASSES if pd.notna(row.get(c)) and str(row.get(c)).strip() != '']
        return '+'.join(sorted(active)) if active else 'clean'

    pivoted['strat_sig'] = pivoted.apply(make_sig, axis=1)
    train_df = (
        pivoted.groupby('strat_sig', group_keys=False)
        .apply(lambda g: g.sample(frac=1.0 - val_size, random_state=random_state))
    )
    val_df = pivoted.drop(train_df.index)
    train_df = train_df.drop(columns=['strat_sig'], errors='ignore').reset_index(drop=True)
    val_df = val_df.drop(columns=['strat_sig'], errors='ignore').reset_index(drop=True)
    return train_df, val_df


def get_defect_aware_sampler(df, clean_weight=0.2, a_weight=4.0, b_weight=12.0,
                              c_weight=1.2, d_weight=1.5, num_samples=None):
    """Create a WeightedRandomSampler that oversamples rare defects and downsamples clean sheets.

    Sampling weights ensure that every training batch contains examples of rare defect classes
    (Defect A at 4x, Defect B at 12x), preventing gradient dilution from the dominant clean class.
    """
    if 'defect_type' in df.columns:
        pivoted = df.pivot(
            index='image_id', columns='defect_type', values='mask_rle'
        ).reset_index()
    else:
        pivoted = df.copy()

    weights = np.full(len(pivoted), clean_weight, dtype=np.float32)
    for idx in range(len(pivoted)):
        row = pivoted.iloc[idx]
        w = clean_weight
        if pd.notna(row.get('defect_c')) and str(row.get('defect_c')).strip() != '':
            w = max(w, c_weight)
        if pd.notna(row.get('defect_d')) and str(row.get('defect_d')).strip() != '':
            w = max(w, d_weight)
        if pd.notna(row.get('defect_a')) and str(row.get('defect_a')).strip() != '':
            w = max(w, a_weight)
        if pd.notna(row.get('defect_b')) and str(row.get('defect_b')).strip() != '':
            w = max(w, b_weight)
        weights[idx] = w

    if num_samples is None:
        num_samples = len(pivoted)
    return torch.utils.data.WeightedRandomSampler(
        weights=torch.tensor(weights, dtype=torch.double),
        num_samples=num_samples, replacement=True
    )

print("Dataset, stratified split, and defect-aware sampler defined successfully.")


### 6.2 Data Augmentation Pipeline

The augmentation strategy is designed specifically for industrial steel inspection:
- **Spatial transforms** (flip, affine) simulate different strip orientations on the production line
- **Photometric transforms** (brightness, contrast, CLAHE) address varying lighting conditions
- **Regularization** (CoarseDropout, GaussNoise) prevents overfitting to surface texture patterns

In [ ]:
def get_transforms():
    """Build Albumentations augmentation pipelines for training and validation.

    Training pipeline includes spatial, photometric, and regularization augmentations.
    Validation pipeline only resizes to the target dimensions (no augmentation).
    """
    # Build augmentation list with backward/forward compatible API calls
    train_augs = [
        A.Resize(height=CFG.IMG_HEIGHT, width=CFG.IMG_WIDTH),
        A.HorizontalFlip(p=0.5),
        A.VerticalFlip(p=0.5),
        A.Affine(
            scale=(0.9, 1.1),
            translate_percent=(-0.05, 0.05),
            rotate=(-15, 15),
            border_mode=cv2.BORDER_REFLECT,
            p=0.5,
        ),
        A.RandomBrightnessContrast(brightness_limit=0.25, contrast_limit=0.25, p=0.5),
        A.CLAHE(clip_limit=2.0, tile_grid_size=(8, 8), p=0.5),
    ]

    # CoarseDropout: Backward-compatible across Albumentations v1 and v2
    try:
        train_augs.append(A.CoarseDropout(
            num_holes_range=(1, 6), hole_height_range=(12, 24),
            hole_width_range=(12, 48), fill=0, p=0.3))
    except TypeError:
        train_augs.append(A.CoarseDropout(
            max_holes=6, max_height=24, max_width=48,
            min_holes=1, min_height=12, min_width=12, fill_value=0, p=0.3))

    # GaussNoise: Backward-compatible
    try:
        train_augs.append(A.GaussNoise(std_range=(0.01, 0.04), p=0.2))
    except TypeError:
        train_augs.append(A.GaussNoise(var_limit=(10.0, 50.0), p=0.2))

    train_tf = A.Compose(train_augs)
    val_tf = A.Compose([A.Resize(height=CFG.IMG_HEIGHT, width=CFG.IMG_WIDTH)])
    return train_tf, val_tf

train_tf, val_tf = get_transforms()
print("Augmentation pipelines created successfully.")

### 6.3 Data Loading and Split Verification

In [ ]:
# Perform stratified split
train_df, val_df = get_stratified_split(df, val_size=CFG.VAL_SIZE, random_state=CFG.SEED)
print(f"Training set: {len(train_df):,} images")
print(f"Validation set: {len(val_df):,} images")

# Create datasets
train_dataset = SurfaceDefectDataset(train_df, image_dir=CFG.IMAGE_DIR, transforms=train_tf)
val_dataset = SurfaceDefectDataset(val_df, image_dir=CFG.IMAGE_DIR, transforms=val_tf)

# Clean up large exploratory DataFrames to prevent Linux memory retention
for _var in ['sample_images', 'pivoted', 'df_eda', 'defective_samples']:
    if _var in locals():
        try:
            del globals()[_var]
        except Exception:
            pass
gc.collect()

# Create data loaders:
# - num_workers=0: loads batches synchronously in the main thread, completely eliminating
#   multiprocessing /dev/shm shared memory buffers and Linux fork Copy-on-Write (CoW) page accumulation.
# - persistent_workers=False: required by PyTorch when num_workers=0.
# - pin_memory=False: prevents host page-locked memory accumulation in the CUDA pinned memory thread.
num_workers = 0
train_sampler = get_defect_aware_sampler(train_df)
train_loader = DataLoader(
    train_dataset,
    batch_size=CFG.BATCH_SIZE,
    sampler=train_sampler,
    num_workers=0,
    pin_memory=False,
    persistent_workers=False
)
val_loader = DataLoader(
    val_dataset,
    batch_size=CFG.BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=False,
    persistent_workers=False
)

print(f"\nTraining batches per epoch: {len(train_loader)}")
print(f"Validation batches: {len(val_loader)}")
print("Using Defect-Aware Weighted Sampler (Defect B oversampled 12x, Defect A 4x)")
print(f"DataLoaders configured with num_workers=0, persistent_workers=False (zero multiprocessing overhead & zero shared memory creep)")


---

## 7. Loss Functions

### Multi-Task Composite Loss Architecture

The loss function combines three complementary objectives:

1. **Focal Loss** -- Pixel-level classification loss that down-weights easy background pixels,
   preventing the dominant clean steel class from drowning out rare defect gradients.

2. **Soft Dice Loss** -- Region-level overlap loss that directly optimizes the Dice coefficient,
   the primary component of our competition metric. Empty-slice masking ensures that
   channels without ground-truth defects do not push predictions toward negative infinity.

3. **Auxiliary Classification BCE** -- Binary cross-entropy loss on the auxiliary classifier head
   that predicts defect presence/absence per channel, providing additional gradient signal
   at training time.

In [ ]:
class SoftDiceLoss(nn.Module):
    """Soft Dice Loss with per-channel weighting and empty-slice masking.

    When mask_empty=True, Dice is computed only on (batch, channel) slices that contain
    actual ground-truth defect pixels. This prevents rare classes (A, B) from being forced
    to suppress all predictions simply because 98% of images are clean for those channels.
    """
    def __init__(self, smooth=1.0, channel_weights=None, mask_empty=True):
        super().__init__()
        self.smooth = smooth
        self.mask_empty = mask_empty
        if channel_weights is not None:
            if not isinstance(channel_weights, torch.Tensor):
                channel_weights = torch.tensor(channel_weights, dtype=torch.float32)
        self.register_buffer('channel_weights', channel_weights)

    def forward(self, logits, targets):
        probs = torch.sigmoid(logits)
        probs_flat = probs.flatten(2)
        targets_flat = targets.flatten(2)
        intersection = (probs_flat * targets_flat).sum(dim=2)
        target_sum = targets_flat.sum(dim=2)
        union = probs_flat.sum(dim=2) + target_sum
        dice = (2.0 * intersection + self.smooth) / (union + self.smooth)

        if self.mask_empty:
            has_gt = target_sum > 0
            if has_gt.any():
                if self.channel_weights is not None:
                    cw = self.channel_weights.to(device=dice.device, dtype=dice.dtype)
                    cw_expanded = cw.unsqueeze(0).expand_as(dice)
                    valid_weights = cw_expanded * has_gt.float()
                    return 1.0 - (dice * valid_weights).sum() / (valid_weights.sum() + 1e-7)
                return 1.0 - dice[has_gt].mean()
            return torch.tensor(0.0, device=logits.device, dtype=logits.dtype)

        if self.channel_weights is not None:
            cw = self.channel_weights.to(device=dice.device, dtype=dice.dtype)
            weighted_dice = (dice * cw).sum(dim=1) / (cw.sum() + 1e-7)
            return 1.0 - weighted_dice.mean()
        return 1.0 - dice.mean()


class FocalLoss(nn.Module):
    """Multi-label Binary Focal Loss for handling extreme class imbalance.

    Focal Loss down-weights well-classified (easy) examples by the factor (1 - p_t)^gamma,
    focusing the model's learning capacity on hard-to-classify defect pixels.
    """
    def __init__(self, alpha=None, gamma=2.0, reduction='mean'):
        super().__init__()
        self.gamma = gamma
        self.reduction = reduction
        if alpha is not None:
            if not isinstance(alpha, torch.Tensor):
                if isinstance(alpha, (int, float)):
                    alpha = torch.tensor([float(alpha)])
                else:
                    alpha = torch.tensor(alpha, dtype=torch.float32)
            if alpha.ndim == 1 and len(alpha) > 1:
                alpha = alpha.view(1, -1, 1, 1)
        self.register_buffer('alpha', alpha)

    def forward(self, logits, targets):
        probs = torch.sigmoid(logits)
        bce = F.binary_cross_entropy_with_logits(logits, targets, reduction='none')
        p_t = targets * probs + (1.0 - targets) * (1.0 - probs)
        focal_weight = (1.0 - p_t) ** self.gamma
        if self.alpha is not None:
            a = self.alpha.to(device=logits.device, dtype=logits.dtype)
            alpha_t = targets * a + (1.0 - targets) * (1.0 - a)
            focal_loss = alpha_t * focal_weight * bce
        else:
            focal_loss = focal_weight * bce
        if self.reduction == 'mean':
            return focal_loss.mean()
        elif self.reduction == 'sum':
            return focal_loss.sum()
        return focal_loss


class ComboLoss(nn.Module):
    """Combined Focal + Soft Dice Loss for multi-label segmentation.

    Merges pixel-level Focal Loss (handles class imbalance) with region-level Dice Loss
    (directly optimizes overlap metric). Focal loss is scaled by 5x to match the magnitude
    range of Dice loss (~0.5 to 1.0).
    """
    MultiTaskComboLoss = None  # Alias placeholder

    def __init__(self, bce_weight=1.0, dice_weight=1.0, smooth=1.0, use_focal=True,
                 gamma=2.0, alpha=(0.75, 0.85, 0.50, 0.60),
                 pos_weight=(12.0, 25.0, 3.0, 6.0),
                 channel_weights=(1.5, 2.5, 1.0, 1.2), mask_empty=True):
        super().__init__()
        self.bce_weight = bce_weight
        self.dice_weight = dice_weight
        self.use_focal = use_focal

        if use_focal:
            self.pixel_loss_fn = FocalLoss(alpha=alpha, gamma=gamma)
        else:
            if pos_weight is not None:
                if not isinstance(pos_weight, torch.Tensor):
                    if isinstance(pos_weight, (int, float)):
                        pos_weight = torch.tensor([float(pos_weight)])
                    else:
                        pos_weight = torch.tensor(pos_weight, dtype=torch.float32)
                if pos_weight.ndim == 1 and len(pos_weight) > 1:
                    pos_weight = pos_weight.view(1, -1, 1, 1)
            self.register_buffer('pos_weight', pos_weight)
            self.pixel_loss_fn = None

        self.dice = SoftDiceLoss(smooth=smooth, channel_weights=channel_weights, mask_empty=mask_empty)

    def forward(self, logits, targets):
        if self.use_focal:
            pixel_loss = 5.0 * self.pixel_loss_fn(logits, targets)
        else:
            if self.pos_weight is not None:
                pw = self.pos_weight.to(device=logits.device, dtype=logits.dtype)
                pixel_loss = F.binary_cross_entropy_with_logits(logits, targets, pos_weight=pw)
            else:
                pixel_loss = F.binary_cross_entropy_with_logits(logits, targets)
        dice_loss = self.dice(logits, targets)
        return self.bce_weight * pixel_loss + self.dice_weight * dice_loss

# Alias for backward compatibility
MultiTaskComboLoss = ComboLoss
ComboLoss.MultiTaskComboLoss = ComboLoss

print("Loss functions defined: FocalLoss, SoftDiceLoss, ComboLoss")

## 8. Evaluation Metrics and Post-Processing

### 8.1 Competition Metrics

The official competition score combines three components:
- **Foreground Dice (70%):** Measures pixel-level overlap between predicted and true defect masks
- **FP Silence Score (20%):** Penalizes false positive predictions on clean steel sheets
- **Empty Accuracy (10%):** Rewards perfectly clean predictions on defect-free image-channel pairs

In [ ]:
def post_process_mask(mask_numpy, min_areas=CFG.MIN_AREAS, max_areas=CFG.MAX_AREAS,
                      filter_geometry=True, use_closing=True):
    """Apply morphological post-processing to binarized segmentation masks.

    Processing steps:
      1. Morphological closing (3x3 rectangle) to connect adjacent clusters and fill internal holes.
      2. Total area ceiling to eliminate whole-sheet false alarms (e.g., 49k shadow hallucinations).
      3. Minimum area filtering to remove small noise speckles per connected component.
      4. Geometric aspect-ratio filtering for elongated defect classes (A and B).
    """
    clean_mask = np.zeros_like(mask_numpy)
    num_classes = mask_numpy.shape[0]

    for cls_idx in range(num_classes):
        channel = mask_numpy[cls_idx].astype(np.uint8)
        if channel.sum() == 0:
            continue

        if use_closing:
            kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (3, 3))
            channel = cv2.morphologyEx(channel, cv2.MORPH_CLOSE, kernel)

        total_pixels = int(channel.sum())
        if total_pixels == 0:
            continue

        max_area = max_areas.get(cls_idx, None) if max_areas else None
        if max_area and total_pixels > max_area:
            continue

        min_area = min_areas.get(cls_idx, 0)
        if total_pixels < min_area:
            continue

        num_labels, labels, stats, _ = cv2.connectedComponentsWithStats(channel, connectivity=8)
        for label in range(1, num_labels):
            area = stats[label, cv2.CC_STAT_AREA]
            h = stats[label, cv2.CC_STAT_HEIGHT]
            w = stats[label, cv2.CC_STAT_WIDTH]

            if area < min_area:
                continue
            if filter_geometry:
                if cls_idx == 1:  # Defect B: edge cracks/tears are tall vertical features
                    aspect_ratio = h / max(w, 1)
                    if h < 25 or aspect_ratio < 2.0:
                        continue
                elif cls_idx == 0:  # Defect A: inclusions/pitting morphology
                    aspect_ratio = h / max(w, 1)
                    if h < 15 or aspect_ratio < 1.2:
                        continue
            clean_mask[cls_idx][labels == label] = 1

    return clean_mask


def compute_competition_metrics(pred_masks, gt_masks, threshold=CFG.CLASS_THRESHOLDS):
    """Compute the official competition composite score.

    Args:
        pred_masks: Tensor of shape (B, 4, H, W) containing logits, probabilities, or binary masks.
        gt_masks: Tensor of shape (B, 4, H, W) containing ground-truth binary masks.
        threshold: Per-channel binarization threshold (float, dict, or list).

    Returns:
        Dictionary containing foreground_dice, false_positive_score, empty_accuracy, and official_score.
    """
    N, C, H, W = pred_masks.shape
    canvas_pixels = H * W

    if pred_masks.dtype in (torch.float32, torch.float64):
        if pred_masks.min() < 0.0 or pred_masks.max() > 1.0:
            probs = torch.sigmoid(pred_masks)
        else:
            probs = pred_masks
        if isinstance(threshold, dict):
            th = torch.tensor([threshold.get(i, 0.5) for i in range(C)],
                              device=pred_masks.device, dtype=pred_masks.dtype).view(1, C, 1, 1)
        elif isinstance(threshold, (list, tuple)):
            th = torch.tensor(threshold, device=pred_masks.device, dtype=pred_masks.dtype).view(1, C, 1, 1)
        else:
            th = float(threshold)
        preds_bin = (probs > th).float()
    else:
        preds_bin = pred_masks.float()

    pred_flat = preds_bin.reshape(N * C, -1)
    gt_flat = gt_masks.float().reshape(N * C, -1)

    pos_mask = gt_flat.sum(dim=1) > 0
    neg_mask = ~pos_mask

    foreground_dice = 0.0
    if pos_mask.any():
        intersection = (pred_flat[pos_mask] * gt_flat[pos_mask]).sum(dim=1)
        union = pred_flat[pos_mask].sum(dim=1) + gt_flat[pos_mask].sum(dim=1)
        dice = (2.0 * intersection) / (union + 1e-7)
        foreground_dice = dice.mean().item()

    false_positive_score = 1.0
    empty_accuracy = 1.0
    if neg_mask.any():
        fp_pixels = pred_flat[neg_mask].sum(dim=1)
        fp_frac = fp_pixels / canvas_pixels
        false_positive_score = (1.0 - fp_frac.mean()).item()
        empty_accuracy = (fp_pixels == 0).float().mean().item()

    official_score = 0.70 * foreground_dice + 0.20 * false_positive_score + 0.10 * empty_accuracy
    return {
        'foreground_dice': foreground_dice,
        'false_positive_score': false_positive_score,
        'empty_accuracy': empty_accuracy,
        'official_score': official_score,
    }


def evaluate_model(model, val_loader, classifier=None, model_name="Model"):
    """Evaluate a segmentation model or TwoStagePipeline using scalar streaming metrics (0 bytes heap retention)."""
    model.eval()
    if classifier is not None:
        classifier.eval()
    cls_gate_tensor = torch.tensor([CFG.CLS_GATES[c] for c in range(4)], device=DEVICE).view(1, 4, 1, 1)
    th_tensor = torch.tensor([CFG.CLASS_THRESHOLDS[c] for c in range(4)], device=DEVICE).view(1, 4, 1, 1)

    dice_sum = 0.0
    n_pos = 0
    fp_sum = 0.0
    empty_correct = 0
    n_neg = 0

    print(f"\nEvaluating {model_name} on validation set ({len(val_loader.dataset)} images)...")
    with torch.no_grad():
        for images, masks in tqdm(val_loader, desc=f"Evaluating [{model_name}]"):
            images, masks = images.to(DEVICE), masks.to(DEVICE)
            with torch.amp.autocast('cuda', enabled=(DEVICE == 'cuda')):
                if isinstance(model, TwoStagePipeline):
                    mask_p, _ = model(images)
                else:
                    clf_p = None
                    if classifier is not None:
                        clf_logits = classifier(images)
                        clf_p = torch.sigmoid(clf_logits).unsqueeze(-1).unsqueeze(-1)

                    out = model(images, return_cls=True) if hasattr(model, 'cls_head') else model(images)
                    mask_logits, cls_logits = out if isinstance(out, tuple) else (out, None)
                    mask_p = torch.sigmoid(mask_logits)
                    cls_p = torch.sigmoid(cls_logits).unsqueeze(-1).unsqueeze(-1) if cls_logits is not None else None
                    effective_cls_p = clf_p if clf_p is not None else cls_p

                    if effective_cls_p is not None:
                        ratio = torch.clamp(effective_cls_p / (cls_gate_tensor + 1e-7), max=1.0)
                        mask_p = mask_p * (ratio ** 2)

            preds_bin = (mask_p > th_tensor).cpu().numpy().astype(np.uint8)
            gt_np = masks.cpu().numpy().astype(np.uint8)

            for b in range(preds_bin.shape[0]):
                cleaned = post_process_mask(preds_bin[b],
                                            min_areas=CFG.MIN_AREAS,
                                            max_areas=CFG.MAX_AREAS,
                                            filter_geometry=True,
                                            use_closing=True)
                for c in range(4):
                    p_c = cleaned[c]
                    g_c = gt_np[b, c]
                    p_sum = int(p_c.sum())
                    g_sum = int(g_c.sum())
                    if g_sum > 0:
                        inter = int(np.logical_and(p_c, g_c).sum())
                        dice_sum += (2.0 * inter) / (p_sum + g_sum + 1e-7)
                        n_pos += 1
                    else:
                        fp_sum += p_sum / (128.0 * 800.0)
                        empty_correct += int(p_sum == 0)
                        n_neg += 1

            del images, masks, mask_p, preds_bin, gt_np

    fg_dice = float(dice_sum / n_pos) if n_pos > 0 else 0.0
    fp_score = float(1.0 - (fp_sum / n_neg)) if n_neg > 0 else 1.0
    empty_acc = float(empty_correct / n_neg) if n_neg > 0 else 1.0
    official = 0.70 * fg_dice + 0.20 * fp_score + 0.10 * empty_acc

    metrics = {
        'foreground_dice': fg_dice,
        'false_positive_score': fp_score,
        'empty_accuracy': empty_acc,
        'official_score': official,
    }
    return metrics


print("Metrics, post-processing, and evaluation pipeline defined successfully.")


---

## 9. Stage 1: SurfaceDefectClassifier (Common Multi-Label Classifier)

### 9.1 Two-Stage Architecture Overview

Industrial steel sheet inspection presents an extreme class and foreground imbalance:
- **5,516 clean sheets** (approximately 48.4%) contain zero defects across all channels.
- In defective sheets, defect pixels occupy less than 1% to 2% of the canvas area.

Directly feeding pristine sheets to heavy segmentation models causes spurious false positive hallucinations,
severely penalizing the competition false-positive score.

To solve this, SurfaceGuard AI adopts a **Two-Stage Cascaded Architecture**:
1. **Stage 1 (Shared Gatekeeper)**: `SurfaceDefectClassifier` evaluates the entire image globally,
   outputting 4 independent defect probabilities.
2. **Stage 2 (Specialized Segmenter)**: If defects are detected, the sheet passes to the chosen segmentation
   model (ScratchUNet, ResAttentionUNet, or PretrainedResNet34UNet) with soft quadratic gating.

Architectural features of `SurfaceDefectClassifier`:
- 5-Stage Residual Backbone (64 -> 128 -> 256 -> 384 -> 512 channels) built 100% from scratch.
- Concurrent Spatial and Channel Squeeze-and-Excitation (scSE) blocks in every residual unit.
- Dual Global Pooling: Concatenates `AdaptiveAvgPool2d` (broad diffuse patches) and `AdaptiveMaxPool2d` (sharp 1-pixel scratch spikes).
- Regularized dense classification head with `Dropout(0.4)` and `BatchNorm1d`.

In [ ]:
class ChannelSE(nn.Module):
    """Channel Squeeze-and-Excitation block for global channel recalibration."""
    def __init__(self, in_channels, reduction=16):
        super().__init__()
        self.avg_pool = nn.AdaptiveAvgPool2d((1, 1))
        self.fc = nn.Sequential(
            nn.Linear(in_channels, max(in_channels // reduction, 4), bias=False),
            nn.ReLU(inplace=True),
            nn.Linear(max(in_channels // reduction, 4), in_channels, bias=False),
            nn.Sigmoid()
        )

    def forward(self, x):
        b, c, _, _ = x.size()
        return x * self.fc(self.avg_pool(x).view(b, c)).view(b, c, 1, 1)


class SpatialSE(nn.Module):
    """Spatial Squeeze-and-Excitation block for fine spatial feature weighting."""
    def __init__(self, in_channels):
        super().__init__()
        self.conv = nn.Conv2d(in_channels, 1, kernel_size=1)

    def forward(self, x):
        return x * torch.sigmoid(self.conv(x))


class Classifier_scSE(nn.Module):
    """Concurrent Spatial and Channel Squeeze-and-Excitation (scSE) block."""
    def __init__(self, in_channels, reduction=16):
        super().__init__()
        self.cse = ChannelSE(in_channels, reduction)
        self.sse = SpatialSE(in_channels)

    def forward(self, x):
        return self.cse(x) + self.sse(x)


class ClassifierResBlock(nn.Module):
    """Residual convolutional block with concurrent scSE attention."""
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(out_channels)
        self.relu = nn.ReLU(inplace=True)
        self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_channels)
        self.scse = Classifier_scSE(out_channels)
        self.shortcut = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=1, bias=False),
            nn.BatchNorm2d(out_channels)
        ) if in_channels != out_channels else nn.Identity()

    def forward(self, x):
        res = self.shortcut(x)
        out = self.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        out = self.scse(out)
        return self.relu(out + res)


class SurfaceDefectClassifier(nn.Module):
    """Dedicated 100% Scratch Multi-Label Classifier for Industrial Steel Sheets.

    Architectural Highlights:
      1. 5-Stage Residual Backbone (64 -> 128 -> 256 -> 384 -> 512) with scSE attention.
      2. Dual Global Pooling: Concatenates AdaptiveAvgPool2d and AdaptiveMaxPool2d.
      3. Regularized Dense Head with Dropout(0.4) and BatchNorm1d.
      4. Outputs 4 independent logits (Defects A, B, C, D) without softmax constraint.
    """
    def __init__(self, in_channels=3, num_classes=4):
        super().__init__()
        self.stage1 = ClassifierResBlock(in_channels, 64)
        self.pool1 = nn.MaxPool2d(kernel_size=2, stride=2)

        self.stage2 = ClassifierResBlock(64, 128)
        self.pool2 = nn.MaxPool2d(kernel_size=2, stride=2)

        self.stage3 = ClassifierResBlock(128, 256)
        self.pool3 = nn.MaxPool2d(kernel_size=2, stride=2)

        self.stage4 = ClassifierResBlock(256, 384)
        self.pool4 = nn.MaxPool2d(kernel_size=2, stride=2)

        self.stage5 = ClassifierResBlock(384, 512)
        self.pool5 = nn.MaxPool2d(kernel_size=2, stride=2)

        self.avg_pool = nn.AdaptiveAvgPool2d((1, 1))
        self.max_pool = nn.AdaptiveMaxPool2d((1, 1))
        pooled_dim = 512 * 2

        self.classifier = nn.Sequential(
            nn.Linear(pooled_dim, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(inplace=True),
            nn.Dropout(p=0.4),
            nn.Linear(256, num_classes)
        )

    def forward(self, x):
        x = self.pool1(self.stage1(x))
        x = self.pool2(self.stage2(x))
        x = self.pool3(self.stage3(x))
        x = self.pool4(self.stage4(x))
        x = self.pool5(self.stage5(x))

        avg_feat = self.avg_pool(x).flatten(1)
        max_feat = self.max_pool(x).flatten(1)
        combined_feat = torch.cat([avg_feat, max_feat], dim=1)
        return self.classifier(combined_feat)


# Verify Stage 1 SurfaceDefectClassifier architecture
model_clf = SurfaceDefectClassifier(in_channels=3, num_classes=4).to(DEVICE)
total_params = sum(p.numel() for p in model_clf.parameters())
trainable_params = sum(p.numel() for p in model_clf.parameters() if p.requires_grad)

dummy = torch.randn(2, 3, CFG.IMG_HEIGHT, CFG.IMG_WIDTH, device=DEVICE)
with torch.no_grad():
    clf_out = model_clf(dummy)

print("Stage 1: SurfaceDefectClassifier")
print(f"  Total parameters: {total_params:,}")
print(f"  Trainable parameters: {trainable_params:,}")
print(f"  Architecture: 5-Stage Residual Backbone with scSE + Dual Global Pooling")
print(f"  Output logits shape: {clf_out.shape} (expected: [2, 4])")
del model_clf, dummy, clf_out


class TwoStagePipeline(nn.Module):
    """End-to-End Two-Stage Architecture combining Stage 1 Classifier with a Stage 2 Segmenter.

    Replicates the exact Personal Best 0.65905 architecture:
      1. Stage 1: SurfaceDefectClassifier predicts image-level defect probabilities.
      2. Stage 2: Segmenter generates dense pixel-level masks and auxiliary classification logits.
      3. Classifier Ensemble: Blends standalone classifier probabilities with segmenter auxiliary head:
         cls_p = 0.5 * (clf_probs + seg_cls_probs)
      4. Soft Quadratic Gating: Attenuates mask probabilities smoothly:
         gated_mask = mask_probs * (clamp(cls_p / gate, max=1.0) ** 2)
    """
    def __init__(self, classifier, segmenter, cls_gates=CFG.CLS_GATES):
        super().__init__()
        self.classifier = classifier
        self.segmenter = segmenter
        self.register_buffer('cls_gate_tensor', torch.tensor([cls_gates[c] for c in range(4)]).view(1, 4, 1, 1))

    def forward(self, x):
        # Stage 1: Standalone Classifier
        clf_probs = None
        if self.classifier is not None:
            clf_logits = self.classifier(x)
            clf_probs = torch.sigmoid(clf_logits)

        # Stage 2: Segmenter with auxiliary classification
        seg_out = self.segmenter(x, return_cls=True) if hasattr(self.segmenter, 'cls_head') else self.segmenter(x)
        if isinstance(seg_out, tuple):
            mask_logits, seg_cls_logits = seg_out
            seg_cls_probs = torch.sigmoid(seg_cls_logits)
        else:
            mask_logits, seg_cls_probs = seg_out, None

        mask_probs = torch.sigmoid(mask_logits)

        # Ensemble Stage 1 Classifier with Segmenter Auxiliary Head (The 0.65905 Formula)
        if clf_probs is not None and seg_cls_probs is not None:
            combined_cls_p = 0.5 * (clf_probs + seg_cls_probs)
        elif clf_probs is not None:
            combined_cls_p = clf_probs
        elif seg_cls_probs is not None:
            combined_cls_p = seg_cls_probs
        else:
            combined_cls_p = None

        # Soft Quadratic Gating
        if combined_cls_p is not None:
            cls_p_4d = combined_cls_p.unsqueeze(-1).unsqueeze(-1)
            ratio = torch.clamp(cls_p_4d / (self.cls_gate_tensor + 1e-7), max=1.0)
            gated_mask_probs = mask_probs * (ratio ** 2)
        else:
            gated_mask_probs = mask_probs

        return gated_mask_probs, combined_cls_p


print("Defined TwoStagePipeline class successfully.")


### 9.2 SurfaceDefectClassifier Training Loop

Training configuration for the Stage 1 Classifier:
- **Weighted Binary Cross-Entropy Loss** with class positive weights (`[3.0, 8.0, 1.2, 1.8]`)
  to penalize missing rare defect instances (Defect B gets 8x weight, Defect A gets 3x weight).
- **AdamW Optimizer** with Cosine Annealing learning rate schedule.
- **Balanced Validation Metric**: 60% Macro F1 + 40% Clean Sheet Accuracy to simultaneously balance defect sensitivity with false alarm suppression.
- Automatically saves best checkpoint and logs artifact to Weights & Biases.

In [ ]:
# def train_surface_defect_classifier(train_loader, val_loader, epochs=CFG.EPOCHS, lr=CFG.LR):
#     """Pure training loop for the Stage 1 SurfaceDefectClassifier with live W&B tracking and runtime timing."""
#     print("=" * 70)
#     print("  STARTING STAGE 1: SURFACEDEFECTCLASSIFIER TRAINING")
#     print(f"  Device: {DEVICE} | Total Epochs: {epochs} | LR: {lr}")
#     print("=" * 70)
# 
#     model = SurfaceDefectClassifier(in_channels=3, num_classes=4).to(DEVICE)
#     if CFG.USE_DATA_PARALLEL and torch.cuda.is_available() and torch.cuda.device_count() > 1:
#         print(f"Utilizing {torch.cuda.device_count()} GPUs with DataParallel!")
#         model = nn.DataParallel(model)
# 
#     pos_weights = torch.tensor([3.0, 8.0, 1.2, 1.8], device=DEVICE)
#     criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weights)
# 
#     optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
#     scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs, eta_min=1e-6)
#     scaler = torch.amp.GradScaler('cuda', enabled=(DEVICE == 'cuda'))
# 
#     run = init_wandb(run_name="surfaceguard-stage1-classifier")
#     history = {'epoch': [], 'train_loss': [], 'val_loss': [], 'clean_acc': [], 'macro_f1': [], 'val_score': []}
#     best_val_score = 0.0
#     os.makedirs('models', exist_ok=True)
#     best_model_path = os.path.join('models', 'best_classifier.pth')
# 
#     run_start = time.time()
#     for epoch in range(1, epochs + 1):
#         epoch_start = time.time()
#         model.train()
#         train_loss = 0.0
# 
#         train_pbar = tqdm(train_loader, desc=f"Epoch {epoch:02d}/{epochs:02d} [Classifier Train]")
#         for batch_idx, (images, masks) in enumerate(train_pbar):
#             images = images.to(DEVICE)
#             targets = (masks.sum(dim=(2, 3)) > 0).float().to(DEVICE)
# 
#             optimizer.zero_grad()
#             with torch.amp.autocast('cuda', enabled=(DEVICE == 'cuda')):
#                 logits = model(images)
#                 loss = criterion(logits, targets)
# 
#             scaler.scale(loss).backward()
#             scaler.unscale_(optimizer)
#             torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=2.0)
#             scaler.step(optimizer)
#             scaler.update()
# 
#             train_loss += loss.item()
#             current_lr = optimizer.param_groups[0]['lr']
#             train_pbar.set_postfix({'loss': f"{loss.item():.4f}", 'lr': f"{current_lr:.2e}"})
# 
#             # In-epoch live W&B progress logging every 25 batches
#             loss_val = loss.item()
#             if (batch_idx + 1) % 25 == 0 or (batch_idx + 1) == len(train_loader):
#                 if run is not None:
#                     global_step = (epoch - 1) * len(train_loader) + (batch_idx + 1)
#                     epoch_progress = (epoch - 1) + (batch_idx + 1) / len(train_loader)
#                     run.log({
#                         'classifier/batch_loss': loss_val,
#                         'classifier/learning_rate': current_lr,
#                         'classifier/epoch_progress': epoch_progress,
#                         'classifier/step': global_step,
#                     }, commit=False)
# 
#             # CRITICAL: Free training batch tensors immediately
#             del images, targets, logits, loss
# 
#         scheduler.step()
#         avg_train_loss = train_loss / len(train_loader)
# 
#         # Validation Phase with scalar running confusion metrics (zero heap allocations)
#         model.eval()
#         val_loss = 0.0
#         total_clean = 0
#         correct_clean = 0
#         tp = np.zeros(4, dtype=np.int64)
#         fp = np.zeros(4, dtype=np.int64)
#         fn = np.zeros(4, dtype=np.int64)
# 
#         with torch.no_grad():
#             for images, masks in tqdm(val_loader, desc=f"Epoch {epoch:02d}/{epochs:02d} [Classifier Val]"):
#                 images = images.to(DEVICE)
#                 targets = (masks.sum(dim=(2, 3)) > 0).float().to(DEVICE)
# 
#                 with torch.amp.autocast('cuda', enabled=(DEVICE == 'cuda')):
#                     logits = model(images)
#                     loss = criterion(logits, targets)
# 
#                 val_loss += loss.item()
#                 probs = torch.sigmoid(logits)
#                 binary_b = (probs > 0.50).int()
#                 targets_b = targets.int()
# 
#                 # In-place clean sheet accounting
#                 is_clean = (targets_b.sum(dim=1) == 0)
#                 pred_clean = (binary_b.sum(dim=1) == 0)
#                 total_clean += is_clean.sum().item()
#                 correct_clean += (is_clean & pred_clean).sum().item()
# 
#                 # In-place per-class TP, FP, FN accounting
#                 tp += ((binary_b == 1) & (targets_b == 1)).sum(dim=0).cpu().numpy()
#                 fp += ((binary_b == 1) & (targets_b == 0)).sum(dim=0).cpu().numpy()
#                 fn += ((binary_b == 0) & (targets_b == 1)).sum(dim=0).cpu().numpy()
# 
#                 # CRITICAL: Free validation batch tensors immediately
#                 del images, targets, logits, probs, binary_b, targets_b, is_clean, pred_clean
# 
#         avg_val_loss = val_loss / len(val_loader)
#         clean_acc = float(correct_clean / total_clean) if total_clean > 0 else 1.0
# 
#         f1_scores = []
#         for c in range(4):
#             f1 = float((2 * tp[c]) / (2 * tp[c] + fp[c] + fn[c] + 1e-7))
#             f1_scores.append(f1)
#         macro_f1 = float(np.mean(f1_scores))
#         val_score = 0.60 * macro_f1 + 0.40 * clean_acc
# 
#         epoch_duration = time.time() - epoch_start
#         elapsed_min = (time.time() - run_start) / 60.0
#         avg_epoch_sec = (time.time() - run_start) / epoch
#         eta_min = ((epochs - epoch) * avg_epoch_sec) / 60.0
# 
#         history['epoch'].append(epoch)
#         history['train_loss'].append(avg_train_loss)
#         history['val_loss'].append(avg_val_loss)
#         history['clean_acc'].append(clean_acc)
#         history['macro_f1'].append(macro_f1)
#         history['val_score'].append(val_score)
# 
#         print(f"\n--- Epoch {epoch:02d}/{epochs:02d} Summary (Time: {epoch_duration:.1f}s | Elapsed: {elapsed_min:.1f}m | ETA: {eta_min:.1f}m) ---")
#         print(f"  Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f}")
#         print(f"  Clean Sheet Accuracy: {clean_acc * 100:.2f}% | Macro F1: {macro_f1:.4f} | Val Score: {val_score:.4f}")
#         print(f"  Per-Class F1 -> A: {f1_scores[0]:.3f} | B: {f1_scores[1]:.3f} | C: {f1_scores[2]:.3f} | D: {f1_scores[3]:.3f}")
# 
#         if run is not None:
#             metrics_dict = {
#                 'classifier/train_loss': avg_train_loss,
#                 'classifier/val_loss': avg_val_loss,
#                 'classifier/clean_accuracy': clean_acc,
#                 'classifier/macro_f1': macro_f1,
#                 'classifier/val_score': val_score,
#                 'classifier/epoch_duration_sec': epoch_duration,
#                 'classifier/elapsed_minutes': elapsed_min,
#                 'classifier/eta_minutes': eta_min,
#                 'classifier/epoch': epoch,
#             }
#             for idx, label in enumerate(CFG.DEFECT_CLASSES):
#                 metrics_dict[f'classifier/f1_{label}'] = f1_scores[idx]
#             run.log(metrics_dict)
# 
#         if val_score > best_val_score:
#             best_val_score = val_score
#             save_dict = model.module.state_dict() if hasattr(model, 'module') else model.state_dict()
#             torch.save({'state_dict': save_dict, 'epoch': epoch, 'val_score': val_score}, best_model_path)
#             del save_dict
#             print(f"  Saved new best classifier checkpoint to: {best_model_path} (Score: {best_val_score:.4f})")
# 
#         del train_pbar
#         gc.collect()
#         if DEVICE == 'cuda':
#             torch.cuda.empty_cache()
#         try:
#             import ctypes
#             ctypes.CDLL('libc.so.6').malloc_trim(0)
#         except Exception:
#             pass
# 
#     if run is not None and os.path.exists(best_model_path):
#         try:
#             artifact = wandb.Artifact(
#                 name='classifier-best',
#                 type='model',
#                 description='Best Stage 1 SurfaceDefectClassifier checkpoint',
#                 metadata={'val_score': best_val_score, 'epochs': epochs}
#             )
#             artifact.add_file(best_model_path)
#             run.log_artifact(artifact)
#             print("Logged artifact 'classifier-best' to W&B.")
#         except Exception as e:
#             print(f"Notice: Could not log artifact to W&B: {e}")
#         run.finish()
# 
#     print(f"\nStage 1 Classifier Training complete. Best validation score: {best_val_score:.4f}")
#     
#     # Explicit memory cleanup
#     del model, optimizer, scheduler, scaler
#     gc.collect()
#     if DEVICE == 'cuda':
#         torch.cuda.empty_cache()
# 
#     return history, best_val_score
# 
## Execute Stage 1 Classifier training if FORCE_RETRAIN or checkpoint missing
# if CFG.FORCE_RETRAIN or not os.path.exists('models/best_classifier.pth'):
#     history_clf, best_clf = train_surface_defect_classifier(train_loader, val_loader, epochs=CFG.EPOCHS, lr=CFG.LR)
# else:
#     print("Stage 1 Classifier checkpoint already exists. Set CFG.FORCE_RETRAIN = True to retrain.")
#     history_clf, best_clf = {'epoch': []}, None


### 9.3 SurfaceDefectClassifier Training Visualization

Visualization of training dynamics for the Stage 1 Classifier: BCE loss, clean sheet discrimination accuracy, and Macro F1 score.

In [ ]:
# def plot_classifier_history(history):
#     """Plot training and validation curves for Stage 1 SurfaceDefectClassifier."""
#     if not history or len(history.get('epoch', [])) <= 1:
#         print("Insufficient epoch history to generate training curves.")
#         return
# 
#     fig, axes = plt.subplots(1, 3, figsize=(18, 5))
#     epochs = history['epoch']
# 
#     axes[0].plot(epochs, history['train_loss'], 'b-o', label='Train Loss', linewidth=2)
#     axes[0].plot(epochs, history['val_loss'], 'r-s', label='Val Loss', linewidth=2)
#     axes[0].set_title('Stage 1 Classifier: Loss Curves', fontsize=12, fontweight='bold')
#     axes[0].set_xlabel('Epoch', fontsize=10)
#     axes[0].set_ylabel('BCE Loss', fontsize=10)
#     axes[0].legend(fontsize=9)
#     axes[0].grid(True, alpha=0.3)
# 
#     axes[1].plot(epochs, [a * 100 for a in history['clean_acc']], 'g-^', label='Clean Sheet Accuracy (%)', linewidth=2)
#     axes[1].set_title('Clean Sheet Discrimination', fontsize=12, fontweight='bold')
#     axes[1].set_xlabel('Epoch', fontsize=10)
#     axes[1].set_ylabel('Accuracy (%)', fontsize=10)
#     axes[1].legend(fontsize=9)
#     axes[1].grid(True, alpha=0.3)
# 
#     axes[2].plot(epochs, history['macro_f1'], 'm-d', label='Macro F1', linewidth=2)
#     axes[2].plot(epochs, history['val_score'], 'k-o', label='Balanced Score (60/40)', linewidth=2)
#     axes[2].set_title('Validation Performance', fontsize=12, fontweight='bold')
#     axes[2].set_xlabel('Epoch', fontsize=10)
#     axes[2].set_ylabel('Score', fontsize=10)
#     axes[2].legend(fontsize=9)
#     axes[2].grid(True, alpha=0.3)
# 
#     plt.suptitle('Stage 1: SurfaceDefectClassifier Training Dynamics', fontsize=14, fontweight='bold', y=1.02)
#     plt.tight_layout()
#     plt.show()
# 
# if 'history_clf' in locals() and len(history_clf.get('epoch', [])) > 0:
#     plot_classifier_history(history_clf)
# else:
#     print("No training history for Stage 1 Classifier. (Set CFG.FORCE_RETRAIN = True to train).")


---

## 10. Model Architecture 1: Two-Stage ScratchUNet (SurfaceDefectClassifier -> ScratchUNet)

### 10.1 Architecture Definition

A standard U-Net architecture trained entirely from scratch with no pre-trained weights.
The encoder consists of four stages of DoubleConv blocks (two 3x3 convolutions with BatchNorm
and ReLU), each followed by MaxPool2d downsampling. The bottleneck applies Dropout2d for
regularization. The decoder mirrors the encoder with ConvTranspose2d upsampling and skip
connections. An optional multi-scale auxiliary classification head fuses features from
encoder stages 1, 2, and the bottleneck for defect presence prediction.

| Stage | Operation | Feature Maps | Resolution |
|-------|-----------|-------------|------------|
| Enc1 | DoubleConv | 64 | 128 x 800 |
| Enc2 | DoubleConv | 128 | 64 x 400 |
| Enc3 | DoubleConv | 256 | 32 x 200 |
| Enc4 | DoubleConv | 512 | 16 x 100 |
| Bottleneck | DoubleConv + Dropout | 1024 | 8 x 50 |
| Dec4-1 | ConvTranspose2d + DoubleConv | 512 -> 64 | 16x100 -> 128x800 |
| Output | 1x1 Conv | 4 | 128 x 800 |

In [ ]:
# class DoubleConv(nn.Module):
#     """Two consecutive 3x3 convolutions each followed by BatchNorm and ReLU activation.
# 
#     This is the fundamental building block of the U-Net architecture. Bias is disabled
#     since BatchNorm already includes a learnable bias parameter.
#     """
#     def __init__(self, in_channels, out_channels):
#         super().__init__()
#         self.conv = nn.Sequential(
#             nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1, bias=False),
#             nn.BatchNorm2d(out_channels),
#             nn.ReLU(inplace=True),
#             nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1, bias=False),
#             nn.BatchNorm2d(out_channels),
#             nn.ReLU(inplace=True)
#         )
# 
#     def forward(self, x):
#         return self.conv(x)
# 
# 
# class AttentionBlock(nn.Module):
#     """Additive Attention Gate (Oktay et al., 2018).
# 
#     Gates low-level encoder skip features using coarse-level decoder features,
#     filtering out background surface reflections while preserving genuine defect activations.
#     """
#     def __init__(self, F_g, F_l, F_int):
#         super().__init__()
#         self.W_g = nn.Sequential(
#             nn.Conv2d(F_g, F_int, kernel_size=1, stride=1, padding=0, bias=True),
#             nn.BatchNorm2d(F_int)
#         )
#         self.W_x = nn.Sequential(
#             nn.Conv2d(F_l, F_int, kernel_size=1, stride=1, padding=0, bias=True),
#             nn.BatchNorm2d(F_int)
#         )
#         self.psi = nn.Sequential(
#             nn.Conv2d(F_int, 1, kernel_size=1, stride=1, padding=0, bias=True),
#             nn.BatchNorm2d(1),
#             nn.Sigmoid()
#         )
#         self.relu = nn.ReLU(inplace=True)
# 
#     def forward(self, g, x):
#         g1 = self.W_g(g)
#         x1 = self.W_x(x)
#         psi = self.relu(g1 + x1)
#         psi = self.psi(psi)
#         return x * psi
# 
# 
# class ScratchUNet(nn.Module):
#     """Baseline U-Net architecture trained 100% from scratch.
# 
#     Features:
#     - 4-stage encoder with DoubleConv blocks and MaxPool downsampling
#     - Bottleneck with Dropout2d regularization
#     - 4-stage decoder with ConvTranspose2d upsampling and skip connections
#     - Optional Attention Gates on skip connections
#     - Optional multi-scale auxiliary classification head (Enc1 MaxPool + Enc2 MaxPool + Bottleneck AvgPool)
#     """
#     def __init__(self, in_channels=3, out_channels=4, dropout_rate=0.3,
#                  with_classifier=True, with_attention=False):
#         super().__init__()
#         self.with_classifier = with_classifier
#         self.with_attention = with_attention
# 
#         # Encoder
#         self.enc1 = DoubleConv(in_channels, 64)
#         self.pool1 = nn.MaxPool2d(kernel_size=2, stride=2)
#         self.enc2 = DoubleConv(64, 128)
#         self.pool2 = nn.MaxPool2d(kernel_size=2, stride=2)
#         self.enc3 = DoubleConv(128, 256)
#         self.pool3 = nn.MaxPool2d(kernel_size=2, stride=2)
#         self.enc4 = DoubleConv(256, 512)
#         self.pool4 = nn.MaxPool2d(kernel_size=2, stride=2)
# 
#         # Bottleneck
#         self.bottleneck = DoubleConv(512, 1024)
#         self.dropout = nn.Dropout2d(p=dropout_rate) if dropout_rate > 0 else nn.Identity()
# 
#         # Multi-Scale Auxiliary Classification Head
#         if with_classifier:
#             self.max_pool_enc1 = nn.AdaptiveMaxPool2d((1, 1))
#             self.max_pool_enc2 = nn.AdaptiveMaxPool2d((1, 1))
#             self.avg_pool_b = nn.AdaptiveAvgPool2d((1, 1))
#             self.cls_head = nn.Sequential(
#                 nn.Linear(1216, 256), nn.BatchNorm1d(256), nn.ReLU(inplace=True),
#                 nn.Dropout(p=0.3), nn.Linear(256, out_channels),
#             )
# 
#         # Decoder
#         self.upconv4 = nn.ConvTranspose2d(1024, 512, kernel_size=2, stride=2)
#         self.dec4 = DoubleConv(1024, 512)
#         self.upconv3 = nn.ConvTranspose2d(512, 256, kernel_size=2, stride=2)
#         self.dec3 = DoubleConv(512, 256)
#         self.upconv2 = nn.ConvTranspose2d(256, 128, kernel_size=2, stride=2)
#         self.dec2 = DoubleConv(256, 128)
#         self.upconv1 = nn.ConvTranspose2d(128, 64, kernel_size=2, stride=2)
#         self.dec1 = DoubleConv(128, 64)
# 
#         # Optional Attention Gates
#         if with_attention:
#             self.att4 = AttentionBlock(F_g=512, F_l=512, F_int=256)
#             self.att3 = AttentionBlock(F_g=256, F_l=256, F_int=128)
#             self.att2 = AttentionBlock(F_g=128, F_l=128, F_int=64)
#             self.att1 = AttentionBlock(F_g=64, F_l=64, F_int=32)
# 
#         # Final 1x1 convolution
#         self.final_conv = nn.Conv2d(64, out_channels, kernel_size=1)
# 
#     def forward(self, x, return_cls=False):
#         # Encoder
#         e1 = self.enc1(x)
#         e2 = self.enc2(self.pool1(e1))
#         e3 = self.enc3(self.pool2(e2))
#         e4 = self.enc4(self.pool3(e3))
# 
#         # Bottleneck
#         b = self.dropout(self.bottleneck(self.pool4(e4)))
# 
#         # Auxiliary classification
#         cls_logits = None
#         if self.with_classifier and (return_cls or self.training):
#             feat = torch.cat([self.max_pool_enc1(e1).flatten(1),
#                               self.max_pool_enc2(e2).flatten(1),
#                               self.avg_pool_b(b).flatten(1)], dim=1)
#             cls_logits = self.cls_head(feat)
# 
#         # Decoder with skip connections
#         d4 = self.upconv4(b)
#         x4 = self.att4(g=d4, x=e4) if self.with_attention else e4
#         d4 = self.dec4(torch.cat((d4, x4), dim=1))
# 
#         d3 = self.upconv3(d4)
#         x3 = self.att3(g=d3, x=e3) if self.with_attention else e3
#         d3 = self.dec3(torch.cat((d3, x3), dim=1))
# 
#         d2 = self.upconv2(d3)
#         x2 = self.att2(g=d2, x=e2) if self.with_attention else e2
#         d2 = self.dec2(torch.cat((d2, x2), dim=1))
# 
#         d1 = self.upconv1(d2)
#         x1 = self.att1(g=d1, x=e1) if self.with_attention else e1
#         d1 = self.dec1(torch.cat((d1, x1), dim=1))
# 
#         mask_logits = self.final_conv(d1)
#         if return_cls or (self.training and cls_logits is not None):
#             return mask_logits, cls_logits
#         return mask_logits
# 
## Display architecture summary
# model_1 = ScratchUNet(in_channels=3, out_channels=4).to(DEVICE)
# total_params = sum(p.numel() for p in model_1.parameters())
# trainable_params = sum(p.numel() for p in model_1.parameters() if p.requires_grad)
# print(f"Model 1: ScratchUNet")
# print(f"  Total parameters: {total_params:,}")
# print(f"  Trainable parameters: {trainable_params:,}")
# print(f"  Architecture: 4-stage encoder-decoder with DoubleConv blocks")
# print(f"  Classifier: Multi-scale MaxPool auxiliary head")
# 
## Verify forward pass
# with torch.no_grad():
#     dummy = torch.randn(2, 3, 128, 800).to(DEVICE)
#     mask_out, cls_out = model_1(dummy, return_cls=True)
#     print(f"  Output mask shape: {mask_out.shape} (expected: [2, 4, 128, 800])")
#     print(f"  Output cls shape: {cls_out.shape} (expected: [2, 4])")
# del model_1, dummy, mask_out, cls_out
# gc.collect()
# if torch.cuda.is_available():
#     torch.cuda.empty_cache()

### 10.2 ScratchUNet Training Loop

In [ ]:
# def train_scratch_unet(train_loader, val_loader, epochs=CFG.EPOCHS, lr=CFG.LR):
#     """Train the baseline ScratchUNet with Focal+Dice ComboLoss and OneCycleLR scheduling."""
#     print("=" * 70)
#     print("  STARTING MODEL 1: BASELINE SCRATCH U-NET TRAINING")
#     print(f"  Device: {DEVICE} | Epochs: {epochs} | LR: {lr}")
#     print("=" * 70)
# 
#     model = ScratchUNet(in_channels=3, out_channels=4).to(DEVICE)
# 
#     if CFG.USE_DATA_PARALLEL and torch.cuda.is_available() and torch.cuda.device_count() > 1:
#         print(f"Utilizing {torch.cuda.device_count()} GPUs with DataParallel!")
#         model = nn.DataParallel(model)
# 
#     run = init_wandb(run_name='surfaceguard-scratch-unet')
# 
#     criterion = ComboLoss(bce_weight=1.0, dice_weight=1.0, use_focal=True, gamma=2.0,
#                           alpha=(0.75, 0.85, 0.50, 0.60),
#                           channel_weights=(1.5, 2.5, 1.0, 1.2))
#     optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
#     total_steps = epochs * len(train_loader)
#     scheduler = torch.optim.lr_scheduler.OneCycleLR(
#         optimizer, max_lr=lr, total_steps=total_steps,
#         pct_start=0.10, anneal_strategy='cos', div_factor=10.0, final_div_factor=100.0)
#     scaler = torch.amp.GradScaler('cuda', enabled=(DEVICE == 'cuda'))
# 
#     cls_gate_tensor = torch.tensor([CFG.CLS_GATES[c] for c in range(4)], device=DEVICE).view(1, 4, 1, 1)
#     th_tensor = torch.tensor([CFG.CLASS_THRESHOLDS[c] for c in range(4)], device=DEVICE).view(1, 4, 1, 1)
# 
#     os.makedirs('models', exist_ok=True)
#     best_score = -1.0
#     history = {'epoch': [], 'train_loss': [], 'val_loss': [],
#                'foreground_dice': [], 'false_positive_score': [], 'official_score': []}
# 
#     run_start = time.time()
#     for epoch in range(1, epochs + 1):
#         epoch_start = time.time()
#         model.train()
#         train_loss = 0.0
#         train_pbar = tqdm(train_loader, desc=f"Epoch {epoch:02d}/{epochs:02d} [ScratchUNet Train]")
#         for batch_idx, (images, masks) in enumerate(train_pbar):
#             images, masks = images.to(DEVICE), masks.to(DEVICE)
#             optimizer.zero_grad()
#             with torch.amp.autocast('cuda', enabled=(DEVICE == 'cuda')):
#                 mask_logits, cls_logits = model(images, return_cls=True)
#                 mask_loss = criterion(mask_logits, masks)
#                 cls_targets = (masks.flatten(2).sum(dim=2) > 0).float()
#                 cls_loss = F.binary_cross_entropy_with_logits(cls_logits, cls_targets)
#                 total_loss = mask_loss + 0.3 * cls_loss
# 
#             scaler.scale(total_loss).backward()
#             scaler.unscale_(optimizer)
#             torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
#             scaler.step(optimizer)
#             scaler.update()
#             scheduler.step()
#             train_loss += total_loss.item()
# 
#             current_lr = optimizer.param_groups[0]['lr']
#             train_pbar.set_postfix({'loss': f"{total_loss.item():.4f}", 'lr': f"{current_lr:.2e}"})
# 
#             # In-epoch live W&B progress logging every 25 batches
#             loss_val = total_loss.item()
#             if (batch_idx + 1) % 25 == 0 or (batch_idx + 1) == len(train_loader):
#                 if run is not None:
#                     global_step = (epoch - 1) * len(train_loader) + (batch_idx + 1)
#                     epoch_progress = (epoch - 1) + (batch_idx + 1) / len(train_loader)
#                     run.log({
#                         'scratch_unet/batch_loss': loss_val,
#                         'scratch_unet/learning_rate': current_lr,
#                         'scratch_unet/epoch_progress': epoch_progress,
#                         'scratch_unet/step': global_step,
#                     }, commit=False)
# 
#             # CRITICAL: Free batch tensors immediately to prevent RAM leak
#             del images, masks, mask_logits, cls_logits, mask_loss, cls_targets, cls_loss, total_loss
# 
#         avg_train_loss = train_loss / len(train_loader)
# 
#         # Validation phase with scalar streaming metrics (zero heap allocations)
#         model.eval()
#         val_loss = 0.0
#         dice_sum = 0.0
#         n_pos = 0
#         fp_sum = 0.0
#         empty_correct = 0
#         n_neg = 0
# 
#         with torch.no_grad():
#             with torch.amp.autocast('cuda', enabled=(DEVICE == 'cuda')):
#                 for images, masks in tqdm(val_loader, desc=f"Epoch {epoch:02d}/{epochs:02d} [ScratchUNet Val]"):
#                     images, masks = images.to(DEVICE), masks.to(DEVICE)
#                     mask_logits, cls_logits = model(images, return_cls=True)
#                     v_loss = criterion(mask_logits, masks)
#                     cls_targets = (masks.flatten(2).sum(dim=2) > 0).float()
#                     cls_loss_v = F.binary_cross_entropy_with_logits(cls_logits, cls_targets)
#                     val_loss += (v_loss + 0.3 * cls_loss_v).item()
# 
#                     mask_probs = torch.sigmoid(mask_logits)
#                     cls_probs = torch.sigmoid(cls_logits).unsqueeze(-1).unsqueeze(-1)
#                     ratio = torch.clamp(cls_probs / (cls_gate_tensor + 1e-7), max=1.0)
#                     gated = mask_probs * (ratio ** 2)
#                     preds_bin = (gated > th_tensor).float()
# 
#                     pf = preds_bin.flatten(2)
#                     mf = masks.flatten(2)
#                     inter = (pf * mf).sum(dim=2)
#                     pred_s = pf.sum(dim=2)
#                     gt_s = mf.sum(dim=2)
# 
#                     pos_m = gt_s > 0
#                     neg_m = ~pos_m
# 
#                     if pos_m.any():
#                         b_dice = (2.0 * inter[pos_m]) / (pred_s[pos_m] + gt_s[pos_m] + 1e-7)
#                         dice_sum += b_dice.sum().item()
#                         n_pos += pos_m.sum().item()
# 
#                     if neg_m.any():
#                         b_fp = pred_s[neg_m] / (128.0 * 800.0)
#                         fp_sum += b_fp.sum().item()
#                         empty_correct += (pred_s[neg_m] == 0).sum().item()
#                         n_neg += neg_m.sum().item()
# 
#                     # Free validation batch tensors immediately
#                     del images, masks, mask_logits, cls_logits, v_loss, cls_targets, cls_loss_v
#                     del mask_probs, cls_probs, ratio, gated, preds_bin, pf, mf, inter, pred_s, gt_s, pos_m, neg_m
# 
#         avg_val_loss = val_loss / len(val_loader)
#         fg_dice = (dice_sum / n_pos) if n_pos > 0 else 0.0
#         fp_score = 1.0 - (fp_sum / n_neg) if n_neg > 0 else 1.0
#         empty_acc = (empty_correct / n_neg) if n_neg > 0 else 1.0
#         official = 0.70 * fg_dice + 0.20 * fp_score + 0.10 * empty_acc
# 
#         epoch_duration = time.time() - epoch_start
#         elapsed_min = (time.time() - run_start) / 60.0
#         avg_epoch_sec = (time.time() - run_start) / epoch
#         eta_min = ((epochs - epoch) * avg_epoch_sec) / 60.0
# 
#         history['epoch'].append(epoch)
#         history['train_loss'].append(avg_train_loss)
#         history['val_loss'].append(avg_val_loss)
#         history['foreground_dice'].append(fg_dice)
#         history['false_positive_score'].append(fp_score)
#         history['official_score'].append(official)
# 
#         print(f"Epoch {epoch:02d}/{epochs:02d} | Time: {epoch_duration:.1f}s (Elapsed: {elapsed_min:.1f}m | ETA: {eta_min:.1f}m) | "
#               f"Train: {avg_train_loss:.4f} | Val: {avg_val_loss:.4f} | "
#               f"Dice: {fg_dice:.4f} | FP: {fp_score:.4f} | Score: {official:.4f}")
# 
#         if run is not None:
#             run.log({
#                 'scratch_unet/epoch': epoch,
#                 'scratch_unet/train_loss': avg_train_loss,
#                 'scratch_unet/val_loss': avg_val_loss,
#                 'scratch_unet/foreground_dice': fg_dice,
#                 'scratch_unet/false_positive_score': fp_score,
#                 'scratch_unet/official_score': official,
#                 'scratch_unet/epoch_duration_sec': epoch_duration,
#                 'scratch_unet/elapsed_minutes': elapsed_min,
#                 'scratch_unet/eta_minutes': eta_min,
#             })
# 
#         if official > best_score:
#             best_score = official
#             save_dict = model.module.state_dict() if hasattr(model, 'module') else model.state_dict()
#             torch.save(save_dict, 'models/best_scratch_unet.pth')
#             del save_dict
#             print(f"  >> Saved new best model! Score: {best_score:.4f}")
# 
#         del train_pbar
#         gc.collect()
#         if DEVICE == 'cuda':
#             torch.cuda.empty_cache()
#         try:
#             import ctypes
#             ctypes.CDLL('libc.so.6').malloc_trim(0)
#         except Exception:
#             pass
# 
#     if run is not None:
#         if os.path.exists('models/best_scratch_unet.pth'):
#             try:
#                 artifact = wandb.Artifact(name='scratch-unet-best', type='model',
#                                           description='Best ScratchUNet checkpoint',
#                                           metadata={'official_score': best_score, 'epochs': epochs})
#                 artifact.add_file('models/best_scratch_unet.pth')
#                 run.log_artifact(artifact)
#                 print("Logged artifact 'scratch-unet-best' to W&B.")
#             except Exception as e:
#                 print(f"Notice: Could not log artifact to W&B: {e}")
#         run.finish()
# 
#     print(f"\nTraining complete. Best official score: {best_score:.4f}")
#     
#     # Explicit memory cleanup
#     del model, optimizer, scheduler, scaler
#     gc.collect()
#     if DEVICE == 'cuda':
#         torch.cuda.empty_cache()
# 
#     return history, best_score
# 
## Execute ScratchUNet training if FORCE_RETRAIN or checkpoint missing
# if CFG.FORCE_RETRAIN or not os.path.exists('models/best_scratch_unet.pth'):
#     history_m1, best_m1 = train_scratch_unet(train_loader, val_loader, epochs=CFG.EPOCHS, lr=CFG.LR)
# else:
#     print("ScratchUNet checkpoint already exists. Set CFG.FORCE_RETRAIN = True to retrain.")
#     history_m1, best_m1 = {'epoch': []}, None


### 10.3 ScratchUNet Training Visualization

In [ ]:
def plot_training_history(history, model_name):
    """Plot training and validation loss curves alongside metric progression."""
    if len(history.get('epoch', [])) <= 1:
        print(f"[{model_name}] Evaluated from checkpoint.")
        print(f"  Official Score:   {history['official_score'][-1]:.5f}")
        print(f"  Foreground Dice:  {history['foreground_dice'][-1]:.5f}")
        print(f"  FP Silence Score: {history['false_positive_score'][-1]:.5f}")
        print("  (Epoch-by-epoch training curves are generated during live training runs.)")
        return
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))

    # Loss curves
    axes[0].plot(history['epoch'], history['train_loss'], 'b-o', markersize=4, label='Train Loss')
    axes[0].plot(history['epoch'], history['val_loss'], 'r-o', markersize=4, label='Val Loss')
    axes[0].set_title(f'{model_name} - Loss Curves', fontsize=12, fontweight='bold')
    axes[0].set_xlabel('Epoch')
    axes[0].set_ylabel('Loss')
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)

    # Dice score
    axes[1].plot(history['epoch'], history['foreground_dice'], 'g-o', markersize=4, label='Foreground Dice')
    axes[1].plot(history['epoch'], history['false_positive_score'], 'm-o', markersize=4, label='FP Silence Score')
    axes[1].set_title(f'{model_name} - Metric Progression', fontsize=12, fontweight='bold')
    axes[1].set_xlabel('Epoch')
    axes[1].set_ylabel('Score')
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)

    # Official score
    axes[2].plot(history['epoch'], history['official_score'], 'k-o', markersize=5, linewidth=2, label='Official Score')
    best_epoch = history['epoch'][np.argmax(history['official_score'])]
    best_val = max(history['official_score'])
    axes[2].axhline(y=best_val, color='red', linestyle='--', alpha=0.5, label=f'Best: {best_val:.4f} (Epoch {best_epoch})')
    axes[2].set_title(f'{model_name} - Official Competition Score', fontsize=12, fontweight='bold')
    axes[2].set_xlabel('Epoch')
    axes[2].set_ylabel('Official Score')
    axes[2].legend()
    axes[2].grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()

if 'history_m1' in locals() and len(history_m1.get('epoch', [])) > 0:
    plot_training_history(history_m1, 'Model 1: ScratchUNet')
else:
    print("No training history for ScratchUNet. (Set CFG.FORCE_RETRAIN = True to train).")


---

## 11. Model Architecture 2: Two-Stage ResAttentionUNet (SurfaceDefectClassifier -> ResAttentionUNet)

### 11.1 Architecture Definition

A next-generation segmentation architecture designed specifically for industrial surface inspection,
incorporating three key innovations over the baseline ScratchUNet:

1. **Residual scSE Blocks:** Each encoder and decoder stage uses residual connections with
   concurrent Spatial and Channel Squeeze-and-Excitation (scSE), dynamically recalibrating
   feature importance between defect-relevant and background channels.

2. **ASPP Bottleneck with Anisotropic Strip Convolutions:** The bottleneck uses Atrous Spatial
   Pyramid Pooling with multi-rate dilated convolutions plus strip convolutions (1x7 and 7x1)
   specifically designed to capture elongated horizontal scratch continuity.

3. **Additive Attention Gates:** All four skip connections are gated to filter out clean-sheet
   surface reflections while preserving genuine defect activations.

Trained 100% from scratch with ZERO external pre-trained weights.

In [ ]:
class scSEBlock(nn.Module):
    """Concurrent Spatial and Channel Squeeze-and-Excitation (scSE) Block.

    Combines channel attention (cSE: squeezes spatial context to recalibrate channel dependencies)
    with spatial attention (sSE: highlights defect spatial regions via 1x1 convolution).
    Reference: Roy et al., IEEE TMI 2018.
    """
    def __init__(self, channels, reduction=16):
        super().__init__()
        self.cSE = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Conv2d(channels, max(channels // reduction, 4), kernel_size=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(max(channels // reduction, 4), channels, kernel_size=1),
            nn.Sigmoid()
        )
        self.sSE = nn.Sequential(nn.Conv2d(channels, 1, kernel_size=1), nn.Sigmoid())

    def forward(self, x):
        return x * self.cSE(x) + x * self.sSE(x)


class SEResBlock(nn.Module):
    """Residual Convolutional Block with scSE attention.

    Identity shortcut solves gradient decay across deep architectures.
    The scSE block dynamically boosts defect features while suppressing steel surface grain.
    """
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels), nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels)
        )
        self.shortcut = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=1, bias=False),
            nn.BatchNorm2d(out_channels)
        ) if in_channels != out_channels else nn.Identity()
        self.scse = scSEBlock(out_channels)
        self.relu = nn.ReLU(inplace=True)

    def forward(self, x):
        residual = self.shortcut(x)
        out = self.scse(self.conv(x))
        return self.relu(out + residual)


class ASPPBottleneck(nn.Module):
    """Atrous Spatial Pyramid Pooling (ASPP) Bottleneck with Anisotropic Strip Convolutions.

    Captures multi-scale context specifically engineered for elongated surface defects:
    - Branch 0: 1x1 conv (local context)
    - Branch 1: 3x3 dilated conv at rate 2 (medium receptive field)
    - Branch 2: 3x3 dilated conv at rate 4 (wide receptive field)
    - Branch 3: Strip convolutions (1x7 then 7x1) for horizontal scratch continuity
    - Branch 4: Global average pooling (sheet-wide context)
    """
    def __init__(self, in_channels, out_channels):
        super().__init__()
        mid = out_channels // 4
        self.b0 = nn.Sequential(nn.Conv2d(in_channels, mid, 1, bias=False), nn.BatchNorm2d(mid), nn.ReLU(True))
        self.b1 = nn.Sequential(nn.Conv2d(in_channels, mid, 3, padding=2, dilation=2, bias=False), nn.BatchNorm2d(mid), nn.ReLU(True))
        self.b2 = nn.Sequential(nn.Conv2d(in_channels, mid, 3, padding=4, dilation=4, bias=False), nn.BatchNorm2d(mid), nn.ReLU(True))
        self.b3 = nn.Sequential(
            nn.Conv2d(in_channels, mid, (1, 7), padding=(0, 3), bias=False), nn.BatchNorm2d(mid), nn.ReLU(True),
            nn.Conv2d(mid, mid, (7, 1), padding=(3, 0), bias=False), nn.BatchNorm2d(mid), nn.ReLU(True))
        self.b4 = nn.Sequential(nn.AdaptiveAvgPool2d(1), nn.Conv2d(in_channels, mid, 1, bias=False), nn.BatchNorm2d(mid), nn.ReLU(True))
        self.project = nn.Sequential(nn.Conv2d(mid * 5, out_channels, 1, bias=False), nn.BatchNorm2d(out_channels), nn.ReLU(True), nn.Dropout2d(0.3))

    def forward(self, x):
        h, w = x.shape[2], x.shape[3]
        out4 = F.interpolate(self.b4(x), size=(h, w), mode='bilinear', align_corners=True)
        return self.project(torch.cat([self.b0(x), self.b1(x), self.b2(x), self.b3(x), out4], dim=1))

class AttentionBlock(nn.Module):
    """Additive Attention Gate (Oktay et al., 2018).

    Gates low-level encoder skip features using coarse-level decoder features,
    filtering out background surface reflections while preserving genuine defect activations.
    """
    def __init__(self, F_g, F_l, F_int):
        super().__init__()
        self.W_g = nn.Sequential(
            nn.Conv2d(F_g, F_int, kernel_size=1, stride=1, padding=0, bias=True),
            nn.BatchNorm2d(F_int)
        )
        self.W_x = nn.Sequential(
            nn.Conv2d(F_l, F_int, kernel_size=1, stride=1, padding=0, bias=True),
            nn.BatchNorm2d(F_int)
        )
        self.psi = nn.Sequential(
            nn.Conv2d(F_int, 1, kernel_size=1, stride=1, padding=0, bias=True),
            nn.BatchNorm2d(1),
            nn.Sigmoid()
        )
        self.relu = nn.ReLU(inplace=True)

    def forward(self, g, x):
        g1 = self.W_g(g)
        x1 = self.W_x(x)
        psi = self.relu(g1 + x1)
        psi = self.psi(psi)
        return x * psi

class ResAttentionUNet(nn.Module):
    """State-of-the-art scratch surface inspection architecture.

    Combines Residual SE-Blocks, ASPP Bottleneck with anisotropic strip convolutions,
    Additive Attention Gates, and a multi-scale auxiliary classification head.
    Trained 100% from scratch with ZERO external pre-trained weights.
    """
    def __init__(self, in_channels=3, out_channels=4, deep_supervision=False):
        super().__init__()
        self.with_classifier = True
        self.with_attention = True
        self.deep_supervision = deep_supervision

        # Encoder with Residual scSE Blocks
        self.enc1 = SEResBlock(in_channels, 64)
        self.pool1 = nn.MaxPool2d(2)
        self.enc2 = SEResBlock(64, 128)
        self.pool2 = nn.MaxPool2d(2)
        self.enc3 = SEResBlock(128, 256)
        self.pool3 = nn.MaxPool2d(2)
        self.enc4 = SEResBlock(256, 512)
        self.pool4 = nn.MaxPool2d(2)

        # ASPP Bottleneck
        self.bottleneck = ASPPBottleneck(512, 1024)

        # Auxiliary classification head
        self.max_pool_enc1 = nn.AdaptiveMaxPool2d(1)
        self.max_pool_enc2 = nn.AdaptiveMaxPool2d(1)
        self.avg_pool_b = nn.AdaptiveAvgPool2d(1)
        self.cls_head = nn.Sequential(
            nn.Linear(64 + 128 + 1024, 256), nn.BatchNorm1d(256), nn.ReLU(True),
            nn.Dropout(0.3), nn.Linear(256, out_channels))

        # Decoder with Attention Gates
        self.upconv4 = nn.ConvTranspose2d(1024, 512, 2, stride=2)
        self.att4 = AttentionBlock(512, 512, 256)
        self.dec4 = SEResBlock(1024, 512)
        self.upconv3 = nn.ConvTranspose2d(512, 256, 2, stride=2)
        self.att3 = AttentionBlock(256, 256, 128)
        self.dec3 = SEResBlock(512, 256)
        self.upconv2 = nn.ConvTranspose2d(256, 128, 2, stride=2)
        self.att2 = AttentionBlock(128, 128, 64)
        self.dec2 = SEResBlock(256, 128)
        self.upconv1 = nn.ConvTranspose2d(128, 64, 2, stride=2)
        self.att1 = AttentionBlock(64, 64, 32)
        self.dec1 = SEResBlock(128, 64)

        self.final_conv = nn.Conv2d(64, out_channels, kernel_size=1)

        if deep_supervision:
            self.ds3 = nn.Conv2d(256, out_channels, 1)
            self.ds2 = nn.Conv2d(128, out_channels, 1)

    def forward(self, x, return_cls=False):
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool1(e1))
        e3 = self.enc3(self.pool2(e2))
        e4 = self.enc4(self.pool3(e3))
        b = self.bottleneck(self.pool4(e4))

        # Auxiliary classifier
        cls_logits = None
        if return_cls or self.training:
            feat = torch.cat([self.max_pool_enc1(e1).flatten(1),
                              self.max_pool_enc2(e2).flatten(1),
                              self.avg_pool_b(b).flatten(1)], dim=1)
            cls_logits = self.cls_head(feat)

        # Decoder with attention-gated skip connections
        d4 = self.upconv4(b)
        d4 = self.dec4(torch.cat([d4, self.att4(g=d4, x=e4)], dim=1))
        d3 = self.upconv3(d4)
        d3 = self.dec3(torch.cat([d3, self.att3(g=d3, x=e3)], dim=1))
        d2 = self.upconv2(d3)
        d2 = self.dec2(torch.cat([d2, self.att2(g=d2, x=e2)], dim=1))
        d1 = self.upconv1(d2)
        d1 = self.dec1(torch.cat([d1, self.att1(g=d1, x=e1)], dim=1))

        mask_logits = self.final_conv(d1)

        if self.training and self.deep_supervision:
            ds3_out = F.interpolate(self.ds3(d3), size=x.shape[2:], mode='bilinear', align_corners=True)
            ds2_out = F.interpolate(self.ds2(d2), size=x.shape[2:], mode='bilinear', align_corners=True)
            return mask_logits, cls_logits, ds3_out, ds2_out

        if return_cls or (self.training and cls_logits is not None):
            return mask_logits, cls_logits
        return mask_logits

# Display architecture summary
model_2 = ResAttentionUNet(in_channels=3, out_channels=4).to(DEVICE)
total_params_2 = sum(p.numel() for p in model_2.parameters())
trainable_params_2 = sum(p.numel() for p in model_2.parameters() if p.requires_grad)
print(f"Model 2: ResAttentionUNet")
print(f"  Total parameters: {total_params_2:,}")
print(f"  Trainable parameters: {trainable_params_2:,}")
print(f"  Key innovations: Residual scSE, ASPP + Strip Conv, Attention Gates")

with torch.no_grad():
    dummy = torch.randn(2, 3, 128, 800).to(DEVICE)
    mask_out, cls_out = model_2(dummy, return_cls=True)
    print(f"  Output mask shape: {mask_out.shape}")
    print(f"  Output cls shape: {cls_out.shape}")
del model_2, dummy, mask_out, cls_out
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

### 11.2 ResAttentionUNet Training Loop

In [ ]:
def train_res_attention_unet(train_loader, val_loader, epochs=CFG.EPOCHS, lr=CFG.LR, resume_from=None):
    """Single-GPU fallback training loop for ResAttentionUNet."""
    print("=" * 70)
    print("  STARTING MODEL 2: RESATTENTION U-NET TRAINING (SINGLE GPU FALLBACK)")
    print(f"  Device: {DEVICE} | Target Epochs: {epochs} | LR: {lr}")
    print("=" * 70)

    model = ResAttentionUNet(in_channels=3, out_channels=4).to(DEVICE)
    run = init_wandb(run_name='surfaceguard-resattunet')

    criterion = ComboLoss(bce_weight=1.0, dice_weight=1.0, use_focal=True, gamma=2.0,
                          alpha=(0.75, 0.85, 0.50, 0.60),
                          channel_weights=(1.5, 2.5, 1.0, 1.2))
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    total_steps = epochs * len(train_loader)
    scheduler = torch.optim.lr_scheduler.OneCycleLR(
        optimizer, max_lr=lr, total_steps=total_steps,
        pct_start=0.10, anneal_strategy='cos', div_factor=10.0, final_div_factor=100.0)
    scaler = torch.amp.GradScaler('cuda', enabled=(DEVICE == 'cuda'))

    cls_gate_tensor = torch.tensor([CFG.CLS_GATES[c] for c in range(4)], device=DEVICE).view(1, 4, 1, 1)
    th_tensor = torch.tensor([CFG.CLASS_THRESHOLDS[c] for c in range(4)], device=DEVICE).view(1, 4, 1, 1)

    os.makedirs('models', exist_ok=True)
    best_score = -1.0
    start_epoch = 1

    if resume_from is None and getattr(CFG, 'RESUME_IF_CHECKPOINT_EXISTS', False):
        if os.path.exists('models/best_resattunet.pth'):
            resume_from = 'models/best_resattunet.pth'

    if resume_from and os.path.exists(resume_from):
        print(f"  >> Resuming ResAttentionUNet from existing checkpoint: {resume_from}")
        try:
            ckpt = torch.load(resume_from, map_location=DEVICE)
            state_dict = ckpt['state_dict'] if (isinstance(ckpt, dict) and 'state_dict' in ckpt) else ckpt
            state_dict = {k.replace('module.', ''): v for k, v in state_dict.items()}
            target_model = model.module if hasattr(model, 'module') else model
            target_model.load_state_dict(state_dict, strict=False)
            if isinstance(ckpt, dict):
                if 'epoch' in ckpt:
                    start_epoch = int(ckpt['epoch']) + 1
                if 'official_score' in ckpt:
                    best_score = float(ckpt['official_score'])
            print(f"  >> Resumed weights successfully! Starting from Epoch {start_epoch:02d} (Best Score: {best_score:.4f})")
        except Exception as e:
            print(f"  >> Notice: Could not resume from checkpoint ({e}). Starting fresh from Epoch 01.")

    history = {'epoch': [], 'train_loss': [], 'val_loss': [],
               'foreground_dice': [], 'false_positive_score': [], 'official_score': []}

    run_start = time.time()
    for epoch in range(start_epoch, epochs + 1):
        epoch_start = time.time()
        model.train()
        train_loss = 0.0
        train_pbar = tqdm(train_loader, desc=f"Epoch {epoch:02d}/{epochs:02d} [ResAttUNet Train]")
        for batch_idx, (images, masks) in enumerate(train_pbar):
            images, masks = images.to(DEVICE), masks.to(DEVICE)
            optimizer.zero_grad()
            with torch.amp.autocast('cuda', enabled=(DEVICE == 'cuda')):
                mask_logits, cls_logits = model(images, return_cls=True)
                mask_loss = criterion(mask_logits, masks)
                cls_targets = (masks.flatten(2).sum(dim=2) > 0).float()
                cls_loss = F.binary_cross_entropy_with_logits(cls_logits, cls_targets)
                total_loss = mask_loss + 0.3 * cls_loss

            scaler.scale(total_loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()
            train_loss += total_loss.item()

            current_lr = optimizer.param_groups[0]['lr']
            train_pbar.set_postfix({'loss': f"{total_loss.item():.4f}", 'lr': f"{current_lr:.2e}"})

            loss_val = total_loss.item()
            if (batch_idx + 1) % 25 == 0 or (batch_idx + 1) == len(train_loader):
                if run is not None:
                    global_step = (epoch - 1) * len(train_loader) + (batch_idx + 1)
                    run.log({
                        'resattunet/batch_loss': loss_val,
                        'resattunet/learning_rate': current_lr,
                        'resattunet/step': global_step,
                    }, commit=False)

            del images, masks, mask_logits, cls_logits, mask_loss, cls_targets, cls_loss, total_loss

        avg_train_loss = train_loss / len(train_loader)

        model.eval()
        val_loss = 0.0
        dice_sum = 0.0
        n_pos = 0
        fp_sum = 0.0
        empty_correct = 0
        n_neg = 0

        with torch.no_grad():
            with torch.amp.autocast('cuda', enabled=(DEVICE == 'cuda')):
                for images, masks in tqdm(val_loader, desc=f"Epoch {epoch:02d}/{epochs:02d} [ResAttUNet Val]"):
                    images, masks = images.to(DEVICE), masks.to(DEVICE)
                    mask_logits, cls_logits = model(images, return_cls=True)
                    v_loss = criterion(mask_logits, masks)
                    cls_targets = (masks.flatten(2).sum(dim=2) > 0).float()
                    cls_loss_v = F.binary_cross_entropy_with_logits(cls_logits, cls_targets)
                    val_loss += (v_loss + 0.3 * cls_loss_v).item()

                    mask_probs = torch.sigmoid(mask_logits)
                    cls_probs = torch.sigmoid(cls_logits).unsqueeze(-1).unsqueeze(-1)
                    ratio = torch.clamp(cls_probs / (cls_gate_tensor + 1e-7), max=1.0)
                    gated = mask_probs * (ratio ** 2)
                    preds_bin = (gated > th_tensor).float()

                    pf = preds_bin.flatten(2)
                    mf = masks.flatten(2)
                    inter = (pf * mf).sum(dim=2)
                    pred_s = pf.sum(dim=2)
                    gt_s = mf.sum(dim=2)

                    pos_m = gt_s > 0
                    neg_m = ~pos_m

                    if pos_m.any():
                        b_dice = (2.0 * inter[pos_m]) / (pred_s[pos_m] + gt_s[pos_m] + 1e-7)
                        dice_sum += b_dice.sum().item()
                        n_pos += pos_m.sum().item()

                    if neg_m.any():
                        b_fp = pred_s[neg_m] / (128.0 * 800.0)
                        fp_sum += b_fp.sum().item()
                        empty_correct += (pred_s[neg_m] == 0).sum().item()
                        n_neg += neg_m.sum().item()

                    del images, masks, mask_logits, cls_logits, v_loss, cls_targets, cls_loss_v
                    del mask_probs, cls_probs, ratio, gated, preds_bin, pf, mf, inter, pred_s, gt_s, pos_m, neg_m

        avg_val_loss = val_loss / len(val_loader)
        fg_dice = (dice_sum / n_pos) if n_pos > 0 else 0.0
        fp_score = 1.0 - (fp_sum / n_neg) if n_neg > 0 else 1.0
        empty_acc = (empty_correct / n_neg) if n_neg > 0 else 1.0
        official = 0.70 * fg_dice + 0.20 * fp_score + 0.10 * empty_acc

        epoch_duration = time.time() - epoch_start
        elapsed_min = (time.time() - run_start) / 60.0
        avg_epoch_sec = (time.time() - run_start) / (epoch - start_epoch + 1)
        eta_min = ((epochs - epoch) * avg_epoch_sec) / 60.0

        history['epoch'].append(epoch)
        history['train_loss'].append(avg_train_loss)
        history['val_loss'].append(avg_val_loss)
        history['foreground_dice'].append(fg_dice)
        history['false_positive_score'].append(fp_score)
        history['official_score'].append(official)

        print(f"Epoch {epoch:02d}/{epochs:02d} | Time: {epoch_duration:.1f}s (Elapsed: {elapsed_min:.1f}m | ETA: {eta_min:.1f}m) | "
              f"Train: {avg_train_loss:.4f} | Val: {avg_val_loss:.4f} | "
              f"Dice: {fg_dice:.4f} | FP: {fp_score:.4f} | Score: {official:.4f}")

        if run is not None:
            run.log({
                'resattunet/epoch': epoch,
                'resattunet/train_loss': avg_train_loss,
                'resattunet/val_loss': avg_val_loss,
                'resattunet/foreground_dice': fg_dice,
                'resattunet/false_positive_score': fp_score,
                'resattunet/official_score': official,
                'resattunet/epoch_duration_sec': epoch_duration,
                'resattunet/elapsed_minutes': elapsed_min,
                'resattunet/eta_minutes': eta_min,
            })

        if official > best_score:
            best_score = official
            raw_dict = model.module.state_dict() if hasattr(model, 'module') else model.state_dict()
            save_payload = {
                'state_dict': raw_dict,
                'epoch': epoch,
                'official_score': float(best_score),
                'history': history,
                'metrics': {
                    'foreground_dice': fg_dice,
                    'false_positive_score': fp_score,
                    'official_score': official,
                }
            }
            torch.save(save_payload, 'models/best_resattunet.pth')
            del raw_dict, save_payload
            print(f"  >> Saved new best model! Score: {best_score:.4f}")

            if run is not None:
                try:
                    artifact = wandb.Artifact(
                        name='resattunet-best',
                        type='model',
                        description=f'Best ResAttentionUNet checkpoint at Epoch {epoch:02d} (Score: {best_score:.4f})',
                        metadata={'official_score': float(best_score), 'epoch': epoch, 'epochs': epochs}
                    )
                    artifact.add_file('models/best_resattunet.pth')
                    run.log_artifact(artifact)
                    print(f"  >> [W&B Artifact] Uploaded 'resattunet-best' to W&B (Epoch {epoch:02d}, Score: {best_score:.4f})")
                except Exception as e:
                    print(f"  >> [W&B Artifact] Upload notice: {e}")

        del train_pbar
        gc.collect()
        if DEVICE == 'cuda':
            torch.cuda.empty_cache()
        try:
            import ctypes
            ctypes.CDLL('libc.so.6').malloc_trim(0)
        except Exception:
            pass

    if run is not None:
        try:
            run.finish()
        except Exception:
            pass

    print(f"\nTraining complete. Best official score: {best_score:.4f}")
    del model, optimizer, scheduler, scaler
    gc.collect()
    if DEVICE == 'cuda':
        torch.cuda.empty_cache()

    return history, best_score


# Execution: Automatically detects 2 GPUs for DistributedDataParallel (DDP) or falls back to single GPU
import subprocess
import sys

num_gpus = torch.cuda.device_count() if torch.cuda.is_available() else 0
print(f"CUDA Available: {torch.cuda.is_available()} | Device Count: {num_gpus}")

if num_gpus > 1:
    print(f"\n[DDP Mode] Detected {num_gpus} GPUs. Launching 2-Process DistributedDataParallel (DDP) via torch.distributed.run...")
    print("  >> Process 0 -> GPU 0 | Process 1 -> GPU 1")
    print("  >> Host CPU RAM: Strictly flat (zero glibc arena memory leaks)")
    print("  >> Effective throughput: ~2x speedup (target: 40 epochs in ~4 hours)")
    
    # Auto-bootstrap: ensure src/train_resattunet.py exists even if notebook is run in isolation
    ddp_script = 'src/train_resattunet.py'
    if not os.path.exists(ddp_script):
        os.makedirs('src', exist_ok=True)
        with open(ddp_script, 'w', encoding='utf-8') as _f:
            _f.write('import os\nimport sys\nimport time\nimport gc\nimport ctypes\nimport numpy as np\nimport pandas as pd\nimport cv2\nimport albumentations as A\nimport torch\nimport torch.nn as nn\nimport torch.nn.functional as F\nimport torch.distributed as dist\nfrom torch.utils.data import Dataset, DataLoader\nfrom torch.utils.data.distributed import DistributedSampler\nfrom tqdm import tqdm\n\ntry:\n    import wandb\nexcept ImportError:\n    wandb = None\n\n# Apply glibc memory optimizations to prevent heap fragmentation\ntry:\n    ctypes.CDLL(\'libc.so.6\').mallopt(ctypes.c_int(-100), ctypes.c_int(0))\n    ctypes.CDLL(\'libc.so.6\').mallopt(ctypes.c_int(-101), ctypes.c_int(1))\nexcept Exception:\n    pass\n\ncv2.setNumThreads(0)\ncv2.ocl.setUseOpenCL(False)\n\n\nclass CFG:\n    SEED = 42\n    DATA_CSV = \'data/train.csv\'\n    IMAGE_DIR = \'data/train\'\n    SAMPLE_SUB_CSV = \'data/sample_submission.csv\'\n    TEST_IMAGE_DIR = \'data/test\'\n    IMG_HEIGHT = 128\n    IMG_WIDTH = 800\n    IN_CHANNELS = 3\n    NUM_CLASSES = 4\n    BATCH_SIZE = 16  # Per GPU batch size\n    EPOCHS = 40\n    LR = 3e-4\n    WEIGHT_DECAY = 1e-4\n    VAL_SIZE = 0.20\n    NUM_WORKERS = 0\n    DEFECT_CLASSES = [\'defect_a\', \'defect_b\', \'defect_c\', \'defect_d\']\n    CLASS_THRESHOLDS = {0: 0.32, 1: 0.35, 2: 0.30, 3: 0.30}\n    MIN_AREAS = {0: 60, 1: 120, 2: 350, 3: 300}\n    MAX_AREAS = {0: 25000, 1: 15000, 2: 35000, 3: 35000}\n    CLS_GATES = {0: 0.25, 1: 0.20, 2: 0.35, 3: 0.35}\n    WANDB_PROJECT = \'24F2004781-t32026\'\n    WANDB_ENTITY = \'dineshwar-developer-indian-institute-of-technology\'\n    RESUME_IF_CHECKPOINT_EXISTS = True\n\n\ndef resolve_paths():\n    csv_candidates = [\n        \'/kaggle/input/competitions/industrial-surface-defect-detection/public/train.csv\',\n        \'/kaggle/input/industrial-surface-defect-detection/public/train.csv\',\n        \'/kaggle/input/industrial-surface-defect-detection/train.csv\',\n        \'data/train.csv\'\n    ]\n    for cand in csv_candidates:\n        if os.path.exists(cand):\n            CFG.DATA_CSV = cand\n            break\n\n    img_candidates = [\n        \'/kaggle/input/competitions/industrial-surface-defect-detection/public/train\',\n        \'/kaggle/input/industrial-surface-defect-detection/public/train\',\n        \'/kaggle/input/industrial-surface-defect-detection/train\',\n        \'data/train\'\n    ]\n    for cand in img_candidates:\n        if os.path.exists(cand):\n            CFG.IMAGE_DIR = cand\n            break\n\n\ndef rle_to_mask(rle_string, height=128, width=800):\n    if not isinstance(rle_string, str) or not rle_string.strip():\n        return np.zeros((height, width), dtype=np.uint8)\n    runs = np.fromstring(rle_string, dtype=np.int32, sep=\' \')\n    starts = runs[::2] - 1\n    lengths = runs[1::2]\n    mask_flat = np.zeros(height * width, dtype=np.uint8)\n    for start, length in zip(starts, lengths):\n        mask_flat[start:start + length] = 1\n    return mask_flat.reshape((height, width), order=\'F\')\n\n\nclass SurfaceDefectDataset(Dataset):\n    def __init__(self, df, image_dir, transforms=None):\n        self.image_dir = image_dir\n        self.transforms = transforms\n        if \'defect_type\' in df.columns:\n            pivoted = df.pivot(index=\'image_id\', columns=\'defect_type\', values=\'mask_rle\').reset_index()\n        else:\n            pivoted = df.copy().reset_index(drop=True)\n        self.image_ids = pivoted[\'image_id\'].astype(str).values\n        self.rles = {\n            c: np.array([str(x) if (pd.notna(x) and str(x).strip() != \'\') else \'\' for x in pivoted[c].values], dtype=object)\n            for c in CFG.DEFECT_CLASSES\n        }\n\n    def __len__(self):\n        return len(self.image_ids)\n\n    def __getitem__(self, idx):\n        image_id = self.image_ids[idx]\n        image_path = os.path.join(self.image_dir, f"{image_id}.jpg" if not image_id.endswith(\'.jpg\') else image_id)\n        image = cv2.imread(image_path)\n        if image is None:\n            raise FileNotFoundError(f"Image not found: {image_path}")\n        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)\n        h, w, _ = image.shape\n        mask = np.zeros((h, w, 4), dtype=np.uint8)\n        for c, defect in enumerate(CFG.DEFECT_CLASSES):\n            rle_str = self.rles[defect][idx]\n            if rle_str:\n                mask[:, :, c] = rle_to_mask(rle_str, height=h, width=w)\n        if self.transforms:\n            augmented = self.transforms(image=image, mask=mask)\n            image = augmented[\'image\']\n            mask = augmented[\'mask\']\n        image_np = np.ascontiguousarray(image.transpose(2, 0, 1), dtype=np.float32) / 255.0\n        mask_np = np.ascontiguousarray(mask.transpose(2, 0, 1), dtype=np.float32)\n        return torch.from_numpy(image_np), torch.from_numpy(mask_np)\n\n\ndef get_stratified_split(df, val_size=0.20, random_state=42):\n    if \'defect_type\' in df.columns:\n        pivoted = df.pivot(index=\'image_id\', columns=\'defect_type\', values=\'mask_rle\').reset_index()\n    else:\n        pivoted = df.copy()\n\n    def get_signature(row):\n        return tuple(int(pd.notna(row[c]) and str(row[c]).strip() != \'\') for c in CFG.DEFECT_CLASSES)\n\n    pivoted[\'signature\'] = pivoted.apply(get_signature, axis=1)\n    from sklearn.model_selection import StratifiedShuffleSplit\n    strat_key = pivoted[\'signature\'].apply(lambda sig: \'empty\' if sum(sig) == 0 else f"comb_{\'\'.join(map(str, sig))}")\n    counts = strat_key.value_counts()\n    rare_classes = counts[counts < 2].index\n    strat_key_binned = strat_key.apply(lambda k: \'rare_defect\' if k in rare_classes else k)\n\n    sss = StratifiedShuffleSplit(n_splits=1, test_size=val_size, random_state=random_state)\n    train_idx, val_idx = next(sss.split(pivoted, strat_key_binned))\n    train_df = pivoted.iloc[train_idx].drop(columns=[\'signature\']).reset_index(drop=True)\n    val_df = pivoted.iloc[val_idx].drop(columns=[\'signature\']).reset_index(drop=True)\n    return train_df, val_df\n\n\ndef get_defect_aware_weights(df, clean_weight=0.2, a_weight=4.0, b_weight=12.0, c_weight=1.2, d_weight=1.5):\n    weights = np.full(len(df), clean_weight, dtype=np.float32)\n    for idx in range(len(df)):\n        row = df.iloc[idx]\n        w = clean_weight\n        if pd.notna(row.get(\'defect_c\')) and str(row.get(\'defect_c\')).strip() != \'\':\n            w = max(w, c_weight)\n        if pd.notna(row.get(\'defect_d\')) and str(row.get(\'defect_d\')).strip() != \'\':\n            w = max(w, d_weight)\n        if pd.notna(row.get(\'defect_a\')) and str(row.get(\'defect_a\')).strip() != \'\':\n            w = max(w, a_weight)\n        if pd.notna(row.get(\'defect_b\')) and str(row.get(\'defect_b\')).strip() != \'\':\n            w = max(w, b_weight)\n        weights[idx] = w\n    return torch.tensor(weights, dtype=torch.double)\n\n\n# Architecture Definitions\nclass scSEBlock(nn.Module):\n    def __init__(self, in_channels, reduction=16):\n        super().__init__()\n        self.cSE = nn.Sequential(\n            nn.AdaptiveAvgPool2d(1),\n            nn.Conv2d(in_channels, max(in_channels // reduction, 4), kernel_size=1, bias=False),\n            nn.ReLU(inplace=True),\n            nn.Conv2d(max(in_channels // reduction, 4), in_channels, kernel_size=1, bias=False),\n            nn.Sigmoid()\n        )\n        self.sSE = nn.Sequential(\n            nn.Conv2d(in_channels, 1, kernel_size=1, bias=False),\n            nn.Sigmoid()\n        )\n\n    def forward(self, x):\n        return x * self.cSE(x) + x * self.sSE(x)\n\n\nclass ResBlockWithscSE(nn.Module):\n    def __init__(self, in_channels, out_channels):\n        super().__init__()\n        self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1, bias=False)\n        self.bn1 = nn.BatchNorm2d(out_channels)\n        self.relu = nn.ReLU(inplace=True)\n        self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1, bias=False)\n        self.bn2 = nn.BatchNorm2d(out_channels)\n        self.scse = scSEBlock(out_channels)\n        self.shortcut = nn.Sequential(\n            nn.Conv2d(in_channels, out_channels, kernel_size=1, bias=False),\n            nn.BatchNorm2d(out_channels)\n        ) if in_channels != out_channels else nn.Identity()\n\n    def forward(self, x):\n        res = self.shortcut(x)\n        out = self.relu(self.bn1(self.conv1(x)))\n        out = self.bn2(self.conv2(out))\n        out = self.scse(out)\n        return self.relu(out + res)\n\n\nclass ASPPBottleneck(nn.Module):\n    def __init__(self, in_channels, out_channels, atrous_rates=(6, 12, 18)):\n        super().__init__()\n        self.b0 = nn.Sequential(\n            nn.Conv2d(in_channels, out_channels, 1, bias=False),\n            nn.BatchNorm2d(out_channels),\n            nn.ReLU(inplace=True)\n        )\n        self.b1 = nn.Sequential(\n            nn.Conv2d(in_channels, out_channels, 3, padding=atrous_rates[0], dilation=atrous_rates[0], bias=False),\n            nn.BatchNorm2d(out_channels),\n            nn.ReLU(inplace=True)\n        )\n        self.b2 = nn.Sequential(\n            nn.Conv2d(in_channels, out_channels, 3, padding=atrous_rates[1], dilation=atrous_rates[1], bias=False),\n            nn.BatchNorm2d(out_channels),\n            nn.ReLU(inplace=True)\n        )\n        self.b3 = nn.Sequential(\n            nn.Conv2d(in_channels, out_channels, 3, padding=atrous_rates[2], dilation=atrous_rates[2], bias=False),\n            nn.BatchNorm2d(out_channels),\n            nn.ReLU(inplace=True)\n        )\n        self.b4 = nn.Sequential(\n            nn.AdaptiveAvgPool2d((1, 1)),\n            nn.Conv2d(in_channels, out_channels, 1, bias=False),\n            nn.BatchNorm2d(out_channels),\n            nn.ReLU(inplace=True)\n        )\n        self.project = nn.Sequential(\n            nn.Conv2d(out_channels * 5, out_channels, 1, bias=False),\n            nn.BatchNorm2d(out_channels),\n            nn.ReLU(inplace=True),\n            nn.Dropout2d(0.3)\n        )\n\n    def forward(self, x):\n        h, w = x.shape[2], x.shape[3]\n        out4 = F.interpolate(self.b4(x), size=(h, w), mode=\'bilinear\', align_corners=True)\n        return self.project(torch.cat([self.b0(x), self.b1(x), self.b2(x), self.b3(x), out4], dim=1))\n\n\nclass AttentionBlock(nn.Module):\n    def __init__(self, F_g, F_l, F_int):\n        super().__init__()\n        self.W_g = nn.Sequential(\n            nn.Conv2d(F_g, F_int, kernel_size=1, stride=1, padding=0, bias=True),\n            nn.BatchNorm2d(F_int)\n        )\n        self.W_x = nn.Sequential(\n            nn.Conv2d(F_l, F_int, kernel_size=1, stride=1, padding=0, bias=True),\n            nn.BatchNorm2d(F_int)\n        )\n        self.psi = nn.Sequential(\n            nn.Conv2d(F_int, 1, kernel_size=1, stride=1, padding=0, bias=True),\n            nn.BatchNorm2d(1),\n            nn.Sigmoid()\n        )\n        self.relu = nn.ReLU(inplace=True)\n\n    def forward(self, g, x):\n        g1 = self.W_g(g)\n        x1 = self.W_x(x)\n        psi = self.relu(g1 + x1)\n        psi = self.psi(psi)\n        return x * psi\n\n\nclass ResAttentionUNet(nn.Module):\n    def __init__(self, in_channels=3, out_channels=4):\n        super().__init__()\n        self.enc1 = ResBlockWithscSE(in_channels, 64)\n        self.pool1 = nn.MaxPool2d(2, 2)\n        self.enc2 = ResBlockWithscSE(64, 128)\n        self.pool2 = nn.MaxPool2d(2, 2)\n        self.enc3 = ResBlockWithscSE(128, 256)\n        self.pool3 = nn.MaxPool2d(2, 2)\n        self.enc4 = ResBlockWithscSE(256, 512)\n        self.pool4 = nn.MaxPool2d(2, 2)\n\n        self.bottleneck = ASPPBottleneck(512, 512)\n\n        self.up4 = nn.ConvTranspose2d(512, 256, kernel_size=2, stride=2)\n        self.att4 = AttentionBlock(F_g=256, F_l=512, F_int=256)\n        self.dec4 = ResBlockWithscSE(512 + 256, 256)\n\n        self.up3 = nn.ConvTranspose2d(256, 128, kernel_size=2, stride=2)\n        self.att3 = AttentionBlock(F_g=128, F_l=256, F_int=128)\n        self.dec3 = ResBlockWithscSE(256 + 128, 128)\n\n        self.up2 = nn.ConvTranspose2d(128, 64, kernel_size=2, stride=2)\n        self.att2 = AttentionBlock(F_g=64, F_l=128, F_int=64)\n        self.dec2 = ResBlockWithscSE(128 + 64, 64)\n\n        self.up1 = nn.ConvTranspose2d(64, 32, kernel_size=2, stride=2)\n        self.att1 = AttentionBlock(F_g=32, F_l=64, F_int=32)\n        self.dec1 = ResBlockWithscSE(64 + 32, 32)\n\n        self.final_conv = nn.Conv2d(32, out_channels, kernel_size=1)\n        self.cls_head = nn.Sequential(\n            nn.AdaptiveAvgPool2d((1, 1)),\n            nn.Flatten(),\n            nn.Linear(512, 128),\n            nn.ReLU(inplace=True),\n            nn.Dropout(0.3),\n            nn.Linear(128, out_channels)\n        )\n\n    def forward(self, x, return_cls=False):\n        e1 = self.enc1(x)\n        e2 = self.enc2(self.pool1(e1))\n        e3 = self.enc3(self.pool2(e2))\n        e4 = self.enc4(self.pool3(e3))\n\n        b = self.bottleneck(self.pool4(e4))\n\n        d4 = self.up4(b)\n        x4 = self.att4(g=d4, x=e4)\n        d4 = self.dec4(torch.cat([d4, x4], dim=1))\n\n        d3 = self.up3(d4)\n        x3 = self.att3(g=d3, x=e3)\n        d3 = self.dec3(torch.cat([d3, x3], dim=1))\n\n        d2 = self.up2(d3)\n        x2 = self.att2(g=d2, x=e2)\n        d2 = self.dec2(torch.cat([d2, x2], dim=1))\n\n        d1 = self.up1(d2)\n        x1 = self.att1(g=d1, x=e1)\n        d1 = self.dec1(torch.cat([d1, x1], dim=1))\n\n        mask_logits = self.final_conv(d1)\n        if return_cls:\n            cls_logits = self.cls_head(b)\n            return mask_logits, cls_logits\n        return mask_logits\n\n\nclass ComboLoss(nn.Module):\n    def __init__(self, bce_weight=1.0, dice_weight=1.0, use_focal=True, gamma=2.0, alpha=(0.75, 0.85, 0.50, 0.60), channel_weights=(1.5, 2.5, 1.0, 1.2)):\n        super().__init__()\n        self.bce_weight = bce_weight\n        self.dice_weight = dice_weight\n        self.gamma = gamma\n        self.register_buffer(\'alpha\', torch.tensor(alpha, dtype=torch.float32).view(1, -1, 1, 1))\n        self.register_buffer(\'channel_weights\', torch.tensor(channel_weights, dtype=torch.float32).view(1, -1, 1, 1))\n\n    def forward(self, logits, targets):\n        probs = torch.sigmoid(logits)\n        bce = F.binary_cross_entropy_with_logits(logits, targets, reduction=\'none\')\n        p_t = probs * targets + (1.0 - probs) * (1.0 - targets)\n        alpha_t = self.alpha * targets + (1.0 - self.alpha) * (1.0 - targets)\n        focal_loss = (alpha_t * ((1.0 - p_t) ** self.gamma) * bce * self.channel_weights).mean()\n\n        intersection = (probs * targets).sum(dim=(2, 3))\n        union = probs.sum(dim=(2, 3)) + targets.sum(dim=(2, 3))\n        dice_loss = (1.0 - (2.0 * intersection + 1e-7) / (union + 1e-7)).mean()\n        return self.bce_weight * focal_loss + self.dice_weight * dice_loss\n\n\ndef train_ddp():\n    # 1. Distributed Setup\n    if "RANK" in os.environ and "WORLD_SIZE" in os.environ:\n        rank = int(os.environ["RANK"])\n        world_size = int(os.environ["WORLD_SIZE"])\n        local_rank = int(os.environ.get("LOCAL_RANK", rank))\n    else:\n        rank = 0\n        world_size = 1\n        local_rank = 0\n\n    is_distributed = world_size > 1\n    device = torch.device(f"cuda:{local_rank}" if torch.cuda.is_available() else "cpu")\n\n    if is_distributed:\n        torch.cuda.set_device(local_rank)\n        backend = "nccl" if dist.is_nccl_available() else "gloo"\n        dist.init_process_group(backend=backend, init_method="env://", rank=rank, world_size=world_size)\n\n    resolve_paths()\n\n    if rank == 0:\n        print("=" * 70)\n        print(f"  STARTING 2-GPU DISTRIBUTEDDATAPARALLEL (DDP) TRAINING (40 EPOCHS)")\n        print(f"  World Size: {world_size} GPUs | Master Rank: {rank} | Per-GPU Batch: {CFG.BATCH_SIZE}")\n        print(f"  Effective Global Batch Size: {CFG.BATCH_SIZE * world_size}")\n        print("=" * 70)\n\n    # 2. Dataset & Transforms\n    train_tf = A.Compose([\n        A.Resize(height=CFG.IMG_HEIGHT, width=CFG.IMG_WIDTH),\n        A.HorizontalFlip(p=0.5),\n        A.VerticalFlip(p=0.5),\n        A.RandomBrightnessContrast(brightness_limit=0.20, contrast_limit=0.20, p=0.4),\n    ])\n    val_tf = A.Compose([A.Resize(height=CFG.IMG_HEIGHT, width=CFG.IMG_WIDTH)])\n\n    df = pd.read_csv(CFG.DATA_CSV)\n    train_df, val_df = get_stratified_split(df, val_size=CFG.VAL_SIZE, random_state=CFG.SEED)\n\n    train_dataset = SurfaceDefectDataset(train_df, image_dir=CFG.IMAGE_DIR, transforms=train_tf)\n    val_dataset = SurfaceDefectDataset(val_df, image_dir=CFG.IMAGE_DIR, transforms=val_tf)\n\n    weights_tensor = get_defect_aware_weights(train_df)\n\n    # 3. Model, Loss, Optimizer, DDP Wrapper\n    model = ResAttentionUNet(in_channels=3, out_channels=4).to(device)\n\n    if is_distributed:\n        model = nn.parallel.DistributedDataParallel(\n            model, device_ids=[local_rank], output_device=local_rank, find_unused_parameters=False\n        )\n\n    criterion = ComboLoss().to(device)\n    optimizer = torch.optim.AdamW(model.parameters(), lr=CFG.LR, weight_decay=CFG.WEIGHT_DECAY)\n    \n    # Calculate global steps based on effective batch size\n    batches_per_epoch = len(train_df) // (CFG.BATCH_SIZE * world_size)\n    total_steps = CFG.EPOCHS * batches_per_epoch\n    scheduler = torch.optim.lr_scheduler.OneCycleLR(\n        optimizer, max_lr=CFG.LR, total_steps=total_steps,\n        pct_start=0.10, anneal_strategy=\'cos\', div_factor=10.0, final_div_factor=100.0\n    )\n    scaler = torch.amp.GradScaler(\'cuda\', enabled=(device.type == \'cuda\'))\n\n    cls_gate_tensor = torch.tensor([CFG.CLS_GATES[c] for c in range(4)], device=device).view(1, 4, 1, 1)\n    th_tensor = torch.tensor([CFG.CLASS_THRESHOLDS[c] for c in range(4)], device=device).view(1, 4, 1, 1)\n\n    os.makedirs(\'models\', exist_ok=True)\n    best_score = -1.0\n    start_epoch = 1\n\n    # Resume support\n    ckpt_path = \'models/best_resattunet.pth\'\n    if CFG.RESUME_IF_CHECKPOINT_EXISTS and os.path.exists(ckpt_path):\n        if rank == 0:\n            print(f"  >> [Rank 0] Resuming weights from existing checkpoint: {ckpt_path}")\n        try:\n            ckpt = torch.load(ckpt_path, map_location=device)\n            state_dict = ckpt[\'state_dict\'] if (isinstance(ckpt, dict) and \'state_dict\' in ckpt) else ckpt\n            state_dict = {k.replace(\'module.\', \'\'): v for k, v in state_dict.items()}\n            target_model = model.module if hasattr(model, \'module\') else model\n            target_model.load_state_dict(state_dict, strict=False)\n            if isinstance(ckpt, dict):\n                if \'epoch\' in ckpt:\n                    start_epoch = int(ckpt[\'epoch\']) + 1\n                if \'official_score\' in ckpt:\n                    best_score = float(ckpt[\'official_score\'])\n            if rank == 0:\n                print(f"  >> Resumed weights successfully! Starting from Epoch {start_epoch:02d} (Best Score: {best_score:.4f})")\n        except Exception as e:\n            if rank == 0:\n                print(f"  >> Notice: Could not resume from checkpoint ({e}). Starting fresh.")\n\n    # Only Rank 0 initializes W&B\n    run = None\n    if rank == 0 and wandb is not None:\n        try:\n            run = wandb.init(\n                project=CFG.WANDB_PROJECT,\n                entity=CFG.WANDB_ENTITY,\n                name=f"surfaceguard-resattunet-ddp-{int(time.time())}",\n                config={\n                    \'epochs\': CFG.EPOCHS,\n                    \'batch_size_per_gpu\': CFG.BATCH_SIZE,\n                    \'world_size\': world_size,\n                    \'effective_batch_size\': CFG.BATCH_SIZE * world_size,\n                    \'lr\': CFG.LR,\n                    \'architecture\': \'ResAttentionUNet\'\n                }\n            )\n            print("  >> [Rank 0] W&B run initialized successfully.")\n        except Exception as e:\n            print(f"  >> [Rank 0] W&B initialization notice: {e}")\n\n    # Validation DataLoader (Partitioned deterministically across GPUs)\n    val_sampler = DistributedSampler(val_dataset, num_replicas=world_size, rank=rank, shuffle=False) if is_distributed else None\n    val_loader = DataLoader(val_dataset, batch_size=CFG.BATCH_SIZE, sampler=val_sampler, shuffle=(val_sampler is None), num_workers=0, pin_memory=False)\n\n    history = {\'epoch\': [], \'train_loss\': [], \'val_loss\': [], \'foreground_dice\': [], \'false_positive_score\': [], \'official_score\': []}\n    run_start = time.time()\n\n    for epoch in range(start_epoch, CFG.EPOCHS + 1):\n        epoch_start = time.time()\n        model.train()\n        train_loss = 0.0\n\n        # Distributed defect-aware sampling: independent seed per rank ensures disjoint batches\n        g = torch.Generator()\n        g.manual_seed(CFG.SEED + rank + epoch * 100)\n        samples_per_gpu = len(train_df) // world_size\n        sampled_indices = torch.multinomial(weights_tensor, num_samples=samples_per_gpu, replacement=True, generator=g)\n        train_sampler = torch.utils.data.SubsetRandomSampler(sampled_indices)\n        train_loader = DataLoader(train_dataset, batch_size=CFG.BATCH_SIZE, sampler=train_sampler, num_workers=0, pin_memory=False)\n\n        train_pbar = tqdm(train_loader, desc=f"Epoch {epoch:02d}/{CFG.EPOCHS:02d} [Rank {rank} Train]", disable=(rank != 0))\n        for batch_idx, (images, masks) in enumerate(train_pbar):\n            images, masks = images.to(device), masks.to(device)\n            optimizer.zero_grad()\n            with torch.amp.autocast(\'cuda\', enabled=(device.type == \'cuda\')):\n                mask_logits, cls_logits = model(images, return_cls=True)\n                mask_loss = criterion(mask_logits, masks)\n                cls_targets = (masks.flatten(2).sum(dim=2) > 0).float()\n                cls_loss = F.binary_cross_entropy_with_logits(cls_logits, cls_targets)\n                total_loss = mask_loss + 0.3 * cls_loss\n\n            scaler.scale(total_loss).backward()\n            scaler.unscale_(optimizer)\n            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)\n            scaler.step(optimizer)\n            scaler.update()\n            scheduler.step()\n            train_loss += total_loss.item()\n\n            current_lr = optimizer.param_groups[0][\'lr\']\n            if rank == 0:\n                train_pbar.set_postfix({\'loss\': f"{total_loss.item():.4f}", \'lr\': f"{current_lr:.2e}"})\n                loss_val = total_loss.item()\n                if (batch_idx + 1) % 25 == 0 or (batch_idx + 1) == len(train_loader):\n                    if run is not None:\n                        global_step = (epoch - 1) * len(train_loader) + (batch_idx + 1)\n                        run.log({\n                            \'resattunet/batch_loss\': loss_val,\n                            \'resattunet/learning_rate\': current_lr,\n                            \'resattunet/step\': global_step\n                        }, commit=False)\n\n            del images, masks, mask_logits, cls_logits, mask_loss, cls_targets, cls_loss, total_loss\n\n        avg_train_loss = train_loss / len(train_loader)\n\n        # Validation Phase\n        model.eval()\n        val_loss = 0.0\n        dice_sum = 0.0\n        n_pos = 0.0\n        fp_sum = 0.0\n        empty_correct = 0.0\n        n_neg = 0.0\n\n        with torch.no_grad():\n            with torch.amp.autocast(\'cuda\', enabled=(device.type == \'cuda\')):\n                for images, masks in val_loader:\n                    images, masks = images.to(device), masks.to(device)\n                    mask_logits, cls_logits = model(images, return_cls=True)\n                    v_loss = criterion(mask_logits, masks)\n                    cls_targets = (masks.flatten(2).sum(dim=2) > 0).float()\n                    cls_loss_v = F.binary_cross_entropy_with_logits(cls_logits, cls_targets)\n                    val_loss += (v_loss + 0.3 * cls_loss_v).item()\n\n                    mask_probs = torch.sigmoid(mask_logits)\n                    cls_probs = torch.sigmoid(cls_logits).unsqueeze(-1).unsqueeze(-1)\n                    ratio = torch.clamp(cls_probs / (cls_gate_tensor + 1e-7), max=1.0)\n                    gated = mask_probs * (ratio ** 2)\n                    preds_bin = (gated > th_tensor).float()\n\n                    pf = preds_bin.flatten(2)\n                    mf = masks.flatten(2)\n                    inter = (pf * mf).sum(dim=2)\n                    pred_s = pf.sum(dim=2)\n                    gt_s = mf.sum(dim=2)\n\n                    pos_m = gt_s > 0\n                    neg_m = ~pos_m\n\n                    if pos_m.any():\n                        b_dice = (2.0 * inter[pos_m]) / (pred_s[pos_m] + gt_s[pos_m] + 1e-7)\n                        dice_sum += b_dice.sum().item()\n                        n_pos += pos_m.sum().item()\n\n                    if neg_m.any():\n                        b_fp = pred_s[neg_m] / (128.0 * 800.0)\n                        fp_sum += b_fp.sum().item()\n                        empty_correct += (pred_s[neg_m] == 0).sum().item()\n                        n_neg += neg_m.sum().item()\n\n                    del images, masks, mask_logits, cls_logits, v_loss, cls_targets, cls_loss_v\n                    del mask_probs, cls_probs, ratio, gated, preds_bin, pf, mf, inter, pred_s, gt_s, pos_m, neg_m\n\n        # Aggregate scalar metrics across all DDP GPUs via NCCL all_reduce\n        metrics_tensor = torch.tensor(\n            [val_loss, dice_sum, n_pos, fp_sum, empty_correct, n_neg, len(val_loader)],\n            device=device, dtype=torch.float64\n        )\n        if is_distributed:\n            dist.all_reduce(metrics_tensor, op=dist.ReduceOp.SUM)\n\n        avg_val_loss = metrics_tensor[0].item() / metrics_tensor[6].item() if metrics_tensor[6].item() > 0 else 0.0\n        fg_dice = (metrics_tensor[1].item() / metrics_tensor[2].item()) if metrics_tensor[2].item() > 0 else 0.0\n        fp_score = 1.0 - (metrics_tensor[3].item() / metrics_tensor[5].item()) if metrics_tensor[5].item() > 0 else 1.0\n        empty_acc = (metrics_tensor[4].item() / metrics_tensor[5].item()) if metrics_tensor[5].item() > 0 else 1.0\n        official = 0.70 * fg_dice + 0.20 * fp_score + 0.10 * empty_acc\n\n        epoch_duration = time.time() - epoch_start\n        elapsed_min = (time.time() - run_start) / 60.0\n        avg_epoch_sec = (time.time() - run_start) / (epoch - start_epoch + 1)\n        eta_min = ((CFG.EPOCHS - epoch) * avg_epoch_sec) / 60.0\n\n        if rank == 0:\n            history[\'epoch\'].append(epoch)\n            history[\'train_loss\'].append(avg_train_loss)\n            history[\'val_loss\'].append(avg_val_loss)\n            history[\'foreground_dice\'].append(fg_dice)\n            history[\'false_positive_score\'].append(fp_score)\n            history[\'official_score\'].append(official)\n\n            print(f"Epoch {epoch:02d}/{CFG.EPOCHS:02d} | Time: {epoch_duration:.1f}s (Elapsed: {elapsed_min:.1f}m | ETA: {eta_min:.1f}m) | "\n                  f"Train: {avg_train_loss:.4f} | Val: {avg_val_loss:.4f} | "\n                  f"Dice: {fg_dice:.4f} | FP: {fp_score:.4f} | Score: {official:.4f}")\n\n            if run is not None:\n                run.log({\n                    \'resattunet/epoch\': epoch,\n                    \'resattunet/train_loss\': avg_train_loss,\n                    \'resattunet/val_loss\': avg_val_loss,\n                    \'resattunet/foreground_dice\': fg_dice,\n                    \'resattunet/false_positive_score\': fp_score,\n                    \'resattunet/official_score\': official,\n                    \'resattunet/epoch_duration_sec\': epoch_duration,\n                    \'resattunet/elapsed_minutes\': elapsed_min,\n                    \'resattunet/eta_minutes\': eta_min,\n                })\n\n            if official > best_score:\n                best_score = official\n                raw_dict = model.module.state_dict() if hasattr(model, \'module\') else model.state_dict()\n                save_payload = {\n                    \'state_dict\': raw_dict,\n                    \'epoch\': epoch,\n                    \'official_score\': float(best_score),\n                    \'history\': history,\n                    \'metrics\': {\n                        \'foreground_dice\': fg_dice,\n                        \'false_positive_score\': fp_score,\n                        \'official_score\': official,\n                    }\n                }\n                torch.save(save_payload, \'models/best_resattunet.pth\')\n                del raw_dict, save_payload\n                print(f"  >> [Rank 0] Saved new best model! Score: {best_score:.4f}")\n\n                # Immediate in-loop W&B artifact upload\n                if run is not None:\n                    try:\n                        artifact = wandb.Artifact(\n                            name=\'resattunet-best\',\n                            type=\'model\',\n                            description=f\'Best ResAttentionUNet checkpoint at Epoch {epoch:02d} (Score: {best_score:.4f})\',\n                            metadata={\'official_score\': float(best_score), \'epoch\': epoch, \'epochs\': CFG.EPOCHS}\n                        )\n                        artifact.add_file(\'models/best_resattunet.pth\')\n                        run.log_artifact(artifact)\n                        print(f"  >> [Rank 0 Artifact] Uploaded \'resattunet-best\' to W&B (Epoch {epoch:02d}, Score: {best_score:.4f})")\n                    except Exception as e:\n                        print(f"  >> [Rank 0 Artifact] Upload notice: {e}")\n\n        del train_pbar, train_loader, train_sampler\n        gc.collect()\n        if device.type == \'cuda\':\n            torch.cuda.empty_cache()\n        try:\n            ctypes.CDLL(\'libc.so.6\').malloc_trim(0)\n        except Exception:\n            pass\n\n    if is_distributed:\n        dist.barrier()\n        dist.destroy_process_group()\n\n    if rank == 0:\n        if run is not None:\n            try:\n                run.finish()\n            except Exception:\n                pass\n        print(f"\\n[Rank 0] Training complete. Best official score: {best_score:.4f}")\n\n\nif __name__ == \'__main__\':\n    train_ddp()\n')
        print(f"  >> [Standalone Mode] Auto-generated '{ddp_script}' for DDP execution.")

    cmd = [
        sys.executable, "-m", "torch.distributed.run",
        f"--nproc_per_node={num_gpus}",
        ddp_script
    ]
    subprocess.run(cmd, check=True)

    # Load resulting checkpoint history and score into notebook scope
    if os.path.exists('models/best_resattunet.pth'):
        ckpt = torch.load('models/best_resattunet.pth', map_location=DEVICE)
        history_m2 = ckpt.get('history', {})
        best_m2 = ckpt.get('official_score', 0.0)
        print(f"\n[OK] 2-GPU DDP Training Finished! Best Checkpoint Score: {best_m2:.4f}")
    else:
        history_m2, best_m2 = {}, 0.0
else:
    print(f"\n[Single-GPU Mode] Running ResAttentionUNet on primary device: {DEVICE}...")
    history_m2, best_m2 = train_res_attention_unet(train_loader, val_loader, epochs=CFG.EPOCHS, lr=CFG.LR)


### 11.3 ResAttentionUNet Training Visualization

In [ ]:
if 'history_m2' in locals() and len(history_m2.get('epoch', [])) > 0:
    plot_training_history(history_m2, 'Model 2: ResAttentionUNet')
else:
    print("No training history for ResAttentionUNet. (Set CFG.FORCE_RETRAIN = True to train).")


---

## 12. Model Architecture 3: Two-Stage PretrainedResNet34 (SurfaceDefectClassifier -> PretrainedResNet34UNet)

### 12.1 Architecture Definition

A transfer learning approach using an ImageNet-pretrained ResNet-34 backbone as the encoder,
combined with a custom symmetrical U-Net decoder. This model uses 2-phase progressive fine-tuning:

**Phase 1 (Warmup):** Encoder frozen, only decoder and classification head are trained at a higher
learning rate. This prevents catastrophic forgetting of pre-trained visual features.

**Phase 2 (Differential Fine-Tuning):** Encoder unfrozen with discriminative learning rates --
the pre-trained backbone uses a gentle 1e-5 learning rate while the custom decoder uses 1e-4.

| Skip | Source | Channels | Resolution |
|------|--------|----------|------------|
| Skip 0 | Stem Conv | 64 | 64 x 400 |
| Skip 1 | ResNet Layer 1 | 64 | 32 x 200 |
| Skip 2 | ResNet Layer 2 | 128 | 16 x 100 |
| Skip 3 | ResNet Layer 3 | 256 | 8 x 50 |
| Latent | ResNet Layer 4 | 512 | 4 x 25 |

In [ ]:
# class ResNetDecoderBlock(nn.Module):
#     """Symmetrical U-Net Decoder Block for ResNet backbones.
# 
#     Upsamples coarse latent features by 2x via ConvTranspose2d, concatenates with
#     fine-grained encoder skip connection, and refines via double convolution.
#     Includes spatial guard against 1-pixel rounding discrepancies.
#     """
#     def __init__(self, in_channels, skip_channels, out_channels):
#         super().__init__()
#         self.up = nn.ConvTranspose2d(in_channels, in_channels // 2, kernel_size=2, stride=2)
#         self.conv = nn.Sequential(
#             nn.Conv2d(in_channels // 2 + skip_channels, out_channels, 3, padding=1, bias=False),
#             nn.BatchNorm2d(out_channels), nn.ReLU(True),
#             nn.Conv2d(out_channels, out_channels, 3, padding=1, bias=False),
#             nn.BatchNorm2d(out_channels), nn.ReLU(True))
# 
#     def forward(self, x, skip):
#         x = self.up(x)
#         if x.size(2) != skip.size(2) or x.size(3) != skip.size(3):
#             x = F.interpolate(x, size=(skip.size(2), skip.size(3)), mode='bilinear', align_corners=False)
#         return self.conv(torch.cat([x, skip], dim=1))
# 
# 
# class PretrainedResNet34UNet(nn.Module):
#     """Pretrained ResNet-34 U-Net for industrial steel surface defect segmentation.
# 
#     Uses ImageNet-pretrained ResNet-34 as the encoder backbone with a custom symmetrical
#     decoder, multi-task auxiliary classification head, and built-in progressive fine-tuning
#     controls (freeze_encoder, unfreeze_encoder, get_parameter_groups).
#     """
#     def __init__(self, in_channels=3, out_channels=4, pretrained=True, with_classifier=True):
#         super().__init__()
#         self.with_classifier = with_classifier
#         self.with_attention = False
# 
#         # Load pretrained ResNet-34
#         try:
#             from torchvision.models import resnet34, ResNet34_Weights
#             weights = ResNet34_Weights.DEFAULT if pretrained else None
#             base = resnet34(weights=weights)
#         except (ImportError, AttributeError):
#             import torchvision.models as models
#             base = models.resnet34(pretrained=pretrained)
# 
#         # Encoder: reuse pretrained layers
#         self.conv1 = base.conv1 if in_channels == 3 else nn.Conv2d(in_channels, 64, 7, stride=2, padding=3, bias=False)
#         self.bn1 = base.bn1
#         self.relu = base.relu
#         self.maxpool = base.maxpool
#         self.layer1 = base.layer1
#         self.layer2 = base.layer2
#         self.layer3 = base.layer3
#         self.layer4 = base.layer4
# 
#         # Auxiliary classification head
#         if with_classifier:
#             self.avg_pool = nn.AdaptiveAvgPool2d(1)
#             self.cls_head = nn.Sequential(
#                 nn.Linear(512, 256), nn.BatchNorm1d(256), nn.ReLU(True),
#                 nn.Dropout(0.4), nn.Linear(256, out_channels))
# 
#         # Symmetrical U-Net Decoder
#         self.dec4 = ResNetDecoderBlock(512, 256, 256)
#         self.dec3 = ResNetDecoderBlock(256, 128, 128)
#         self.dec2 = ResNetDecoderBlock(128, 64, 64)
#         self.dec1 = ResNetDecoderBlock(64, 64, 64)
# 
#         # Final upsampling head
#         self.final_conv = nn.Sequential(
#             nn.Upsample(scale_factor=2, mode='bilinear', align_corners=False),
#             nn.Conv2d(64, 32, 3, padding=1, bias=False), nn.BatchNorm2d(32), nn.ReLU(True),
#             nn.Conv2d(32, out_channels, 1))
# 
#     def forward(self, x, return_cls=False):
#         s0 = self.relu(self.bn1(self.conv1(x)))
#         s1 = self.layer1(self.maxpool(s0))
#         s2 = self.layer2(s1)
#         s3 = self.layer3(s2)
#         b = self.layer4(s3)
# 
#         cls_logits = None
#         if self.with_classifier:
#             cls_logits = self.cls_head(self.avg_pool(b).flatten(1))
# 
#         d4 = self.dec4(b, s3)
#         d3 = self.dec3(d4, s2)
#         d2 = self.dec2(d3, s1)
#         d1 = self.dec1(d2, s0)
#         mask_logits = self.final_conv(d1)
# 
#         if return_cls or (self.training and cls_logits is not None):
#             return mask_logits, cls_logits
#         return mask_logits
# 
#     def freeze_encoder(self):
#         """Freeze pre-trained encoder weights for Phase 1 warmup."""
#         for module in [self.conv1, self.bn1, self.layer1, self.layer2, self.layer3, self.layer4]:
#             for param in module.parameters():
#                 param.requires_grad = False
#         print("[PretrainedResNet34UNet] Encoder frozen for Phase 1 warmup.")
# 
#     def unfreeze_encoder(self):
#         """Unfreeze encoder weights for Phase 2 differential fine-tuning."""
#         for module in [self.conv1, self.bn1, self.layer1, self.layer2, self.layer3, self.layer4]:
#             for param in module.parameters():
#                 param.requires_grad = True
#         print("[PretrainedResNet34UNet] Encoder unfrozen for Phase 2 fine-tuning.")
# 
#     def get_parameter_groups(self, lr_encoder=1e-5, lr_decoder=1e-4, weight_decay=1e-4):
#         """Construct parameter groups with discriminative learning rates."""
#         encoder_params = (
#             list(self.conv1.parameters()) + list(self.bn1.parameters()) +
#             list(self.layer1.parameters()) + list(self.layer2.parameters()) +
#             list(self.layer3.parameters()) + list(self.layer4.parameters()))
#         decoder_params = (
#             list(self.dec4.parameters()) + list(self.dec3.parameters()) +
#             list(self.dec2.parameters()) + list(self.dec1.parameters()) +
#             list(self.final_conv.parameters()) +
#             (list(self.cls_head.parameters()) if self.with_classifier else []))
#         return [
#             {"params": encoder_params, "lr": lr_encoder, "weight_decay": weight_decay},
#             {"params": decoder_params, "lr": lr_decoder, "weight_decay": weight_decay}]
# 
## Display architecture summary
# model_3 = PretrainedResNet34UNet(in_channels=3, out_channels=4, pretrained=True).to(DEVICE)
# total_params_3 = sum(p.numel() for p in model_3.parameters())
# trainable_params_3 = sum(p.numel() for p in model_3.parameters() if p.requires_grad)
# print(f"Model 3: PretrainedResNet34UNet")
# print(f"  Total parameters: {total_params_3:,}")
# print(f"  Trainable parameters: {trainable_params_3:,}")
# print(f"  Backbone: ResNet-34 (ImageNet pretrained)")
# print(f"  Fine-tuning: 2-Phase Progressive (Freeze -> Differential LR)")
# 
# with torch.no_grad():
#     dummy = torch.randn(2, 3, 128, 800).to(DEVICE)
#     mask_out, cls_out = model_3(dummy, return_cls=True)
#     print(f"  Output mask shape: {mask_out.shape}")
#     print(f"  Output cls shape: {cls_out.shape}")
# del model_3, dummy, mask_out, cls_out
# gc.collect()
# if torch.cuda.is_available():
#     torch.cuda.empty_cache()

### 12.2 PretrainedResNet34UNet 2-Phase Training Loop

In [ ]:
# def train_pretrained_resnet34(train_loader, val_loader, epochs=CFG.EPOCHS, warmup_epochs=5,
#                              lr_warmup=5e-4, lr_encoder=1e-5, lr_decoder=1e-4):
#     """Train the PretrainedResNet34UNet with 2-phase progressive fine-tuning and memory-safe streaming metrics.
# 
#     Phase 1 (warmup_epochs): Encoder frozen, trains decoder at lr_warmup.
#     Phase 2 (remaining): Encoder unfrozen with discriminative LRs (lr_encoder, lr_decoder).
#     """
#     print("=" * 70)
#     print("  STARTING MODEL 3: PRETRAINED RESNET-34 U-NET 2-PHASE FINE-TUNING")
#     print(f"  Device: {DEVICE} | Total Epochs: {epochs} | Warmup Epochs: {warmup_epochs}")
#     print(f"  LR Warmup: {lr_warmup} | LR Encoder: {lr_encoder} | LR Decoder: {lr_decoder}")
#     print("=" * 70)
# 
#     model = PretrainedResNet34UNet(in_channels=3, out_channels=4, pretrained=True, with_classifier=True).to(DEVICE)
# 
#     is_parallel = False
#     if CFG.USE_DATA_PARALLEL and torch.cuda.is_available() and torch.cuda.device_count() > 1:
#         print(f"Utilizing {torch.cuda.device_count()} GPUs with DataParallel!")
#         model = nn.DataParallel(model)
#         is_parallel = True
# 
#     def get_raw():
#         return model.module if is_parallel else model
# 
#     run = init_wandb(run_name='surfaceguard-pretrained-resnet34')
# 
#     seg_criterion = ComboLoss(bce_weight=1.0, dice_weight=1.0, use_focal=True, gamma=2.0,
#                                alpha=(0.75, 0.85, 0.50, 0.60),
#                                channel_weights=(1.5, 2.5, 1.0, 1.2))
#     pos_weights = torch.tensor([3.0, 8.0, 1.2, 1.8], device=DEVICE)
#     cls_criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weights)
#     scaler = torch.amp.GradScaler('cuda', enabled=(DEVICE == 'cuda'))
# 
#     cls_gate_tensor = torch.tensor([CFG.CLS_GATES[c] for c in range(4)], device=DEVICE).view(1, 4, 1, 1)
#     th_tensor = torch.tensor([CFG.CLASS_THRESHOLDS[c] for c in range(4)], device=DEVICE).view(1, 4, 1, 1)
# 
#     os.makedirs('models', exist_ok=True)
#     best_score = -1.0
#     history = {'epoch': [], 'train_loss': [], 'val_loss': [],
#                'foreground_dice': [], 'false_positive_score': [], 'official_score': [], 'phase': []}
# 
#     optimizer = None
#     scheduler = None
#     current_phase = 0
# 
#     run_start = time.time()
#     for epoch in range(1, epochs + 1):
#         epoch_start = time.time()
#         # Phase management
#         if epoch <= warmup_epochs and current_phase != 1:
#             current_phase = 1
#             print(f"\n>>> ENTERING PHASE 1: WARMUP (Epochs 1-{warmup_epochs})")
#             print("    Encoder FROZEN. Training only Decoder and Heads.")
#             get_raw().freeze_encoder()
#             trainable_params = [p for p in model.parameters() if p.requires_grad]
#             optimizer = torch.optim.AdamW(trainable_params, lr=lr_warmup, weight_decay=1e-4)
#             scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=warmup_epochs, eta_min=1e-5)
# 
#         elif epoch > warmup_epochs and current_phase != 2:
#             current_phase = 2
#             print(f"\n>>> ENTERING PHASE 2: DIFFERENTIAL FINE-TUNING (Epochs {warmup_epochs+1}-{epochs})")
#             print(f"    Encoder UNFROZEN. Backbone LR: {lr_encoder} | Decoder LR: {lr_decoder}")
#             get_raw().unfreeze_encoder()
#             param_groups = get_raw().get_parameter_groups(lr_encoder=lr_encoder, lr_decoder=lr_decoder)
#             optimizer = torch.optim.AdamW(param_groups)
#             scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
#                 optimizer, T_max=max(1, epochs - warmup_epochs), eta_min=1e-6)
# 
#         # Training step
#         model.train()
#         train_loss = 0.0
#         train_pbar = tqdm(train_loader, desc=f"Epoch {epoch:02d}/{epochs:02d} [ResNet34 Train P{current_phase}]")
#         for batch_idx, (images, masks) in enumerate(train_pbar):
#             images, masks = images.to(DEVICE), masks.to(DEVICE)
#             gt_cls = (masks.sum(dim=(2, 3)) > 0).float()
#             optimizer.zero_grad(set_to_none=True)
# 
#             with torch.amp.autocast('cuda', enabled=(DEVICE == 'cuda')):
#                 out = model(images, return_cls=True)
#                 mask_logits, cls_logits = out if isinstance(out, tuple) else (out, None)
#                 seg_loss = seg_criterion(mask_logits, masks)
#                 cls_loss = cls_criterion(cls_logits, gt_cls) if cls_logits is not None else torch.tensor(0.0)
#                 loss = seg_loss + 0.3 * cls_loss
# 
#             scaler.scale(loss).backward()
#             scaler.unscale_(optimizer)
#             torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=2.0)
#             scaler.step(optimizer)
#             scaler.update()
#             train_loss += loss.item()
# 
#             current_lr = optimizer.param_groups[0]['lr']
#             train_pbar.set_postfix({'loss': f"{loss.item():.4f}", 'lr': f"{current_lr:.2e}"})
# 
#             # In-epoch live W&B progress logging every 25 batches
#             loss_val = loss.item()
#             if (batch_idx + 1) % 25 == 0 or (batch_idx + 1) == len(train_loader):
#                 if run is not None:
#                     global_step = (epoch - 1) * len(train_loader) + (batch_idx + 1)
#                     epoch_progress = (epoch - 1) + (batch_idx + 1) / len(train_loader)
#                     run.log({
#                         'pretrained_resnet34/batch_loss': loss_val,
#                         'pretrained_resnet34/learning_rate': current_lr,
#                         'pretrained_resnet34/epoch_progress': epoch_progress,
#                         'pretrained_resnet34/step': global_step,
#                     }, commit=False)
# 
#             # CRITICAL: Free training batch tensors immediately
#             del images, masks, gt_cls, mask_logits, cls_logits, seg_loss, cls_loss, loss
# 
#         scheduler.step()
#         avg_train_loss = train_loss / len(train_loader)
# 
#         # Validation phase with scalar streaming metrics (zero heap allocations)
#         model.eval()
#         val_loss = 0.0
#         dice_sum = 0.0
#         n_pos = 0
#         fp_sum = 0.0
#         empty_correct = 0
#         n_neg = 0
# 
#         with torch.no_grad():
#             for images, masks in tqdm(val_loader, desc=f"Epoch {epoch:02d}/{epochs:02d} [ResNet34 Val]"):
#                 images, masks = images.to(DEVICE), masks.to(DEVICE)
#                 with torch.amp.autocast('cuda', enabled=(DEVICE == 'cuda')):
#                     out = model(images, return_cls=True)
#                     mask_logits, cls_logits = out if isinstance(out, tuple) else (out, None)
#                     val_loss += seg_criterion(mask_logits, masks).item()
# 
#                 mask_p = torch.sigmoid(mask_logits)
#                 if cls_logits is not None:
#                     cls_p = torch.sigmoid(cls_logits).unsqueeze(-1).unsqueeze(-1)
#                     ratio = torch.clamp(cls_p / (cls_gate_tensor + 1e-7), max=1.0)
#                     mask_p = mask_p * (ratio ** 2)
#                 preds_bin = (mask_p > th_tensor).float()
# 
#                 pf = preds_bin.flatten(2)
#                 mf = masks.flatten(2)
#                 inter = (pf * mf).sum(dim=2)
#                 pred_s = pf.sum(dim=2)
#                 gt_s = mf.sum(dim=2)
# 
#                 pos_m = gt_s > 0
#                 neg_m = ~pos_m
# 
#                 if pos_m.any():
#                     b_dice = (2.0 * inter[pos_m]) / (pred_s[pos_m] + gt_s[pos_m] + 1e-7)
#                     dice_sum += b_dice.sum().item()
#                     n_pos += pos_m.sum().item()
# 
#                 if neg_m.any():
#                     b_fp = pred_s[neg_m] / (128.0 * 800.0)
#                     fp_sum += b_fp.sum().item()
#                     empty_correct += (pred_s[neg_m] == 0).sum().item()
#                     n_neg += neg_m.sum().item()
# 
#                 # CRITICAL: Free validation batch tensors immediately
#                 del images, masks, out, mask_logits, cls_logits, mask_p, preds_bin, pf, mf, inter, pred_s, gt_s, pos_m, neg_m
#                 if cls_logits is not None:
#                     del cls_p, ratio
# 
#         avg_val_loss = val_loss / len(val_loader)
#         fg_dice = (dice_sum / n_pos) if n_pos > 0 else 0.0
#         fp_score = 1.0 - (fp_sum / n_neg) if n_neg > 0 else 1.0
#         empty_acc = (empty_correct / n_neg) if n_neg > 0 else 1.0
#         official = 0.70 * fg_dice + 0.20 * fp_score + 0.10 * empty_acc
# 
#         metrics = {
#             'foreground_dice': fg_dice,
#             'false_positive_score': fp_score,
#             'empty_accuracy': empty_acc,
#             'official_score': official,
#         }
# 
#         epoch_duration = time.time() - epoch_start
#         elapsed_min = (time.time() - run_start) / 60.0
#         avg_epoch_sec = (time.time() - run_start) / epoch
#         eta_min = ((epochs - epoch) * avg_epoch_sec) / 60.0
# 
#         history['epoch'].append(epoch)
#         history['train_loss'].append(avg_train_loss)
#         history['val_loss'].append(avg_val_loss)
#         history['foreground_dice'].append(metrics['foreground_dice'])
#         history['false_positive_score'].append(metrics['false_positive_score'])
#         history['official_score'].append(metrics['official_score'])
#         history['phase'].append(current_phase)
# 
#         print(f"Epoch {epoch:02d}/{epochs:02d} (P{current_phase}) | Time: {epoch_duration:.1f}s (Elapsed: {elapsed_min:.1f}m | ETA: {eta_min:.1f}m) | "
#               f"Train: {avg_train_loss:.4f} | Val: {avg_val_loss:.4f} | "
#               f"Dice: {metrics['foreground_dice']:.5f} | FP: {metrics['false_positive_score']:.5f} | "
#               f"Score: {metrics['official_score']:.5f}")
# 
#         if run is not None:
#             run.log({
#                 'pretrained_resnet34/epoch': epoch,
#                 'pretrained_resnet34/phase': current_phase,
#                 'pretrained_resnet34/train_loss': avg_train_loss,
#                 'pretrained_resnet34/val_loss': avg_val_loss,
#                 'pretrained_resnet34/epoch_duration_sec': epoch_duration,
#                 'pretrained_resnet34/elapsed_minutes': elapsed_min,
#                 'pretrained_resnet34/eta_minutes': eta_min,
#                 **{f"pretrained_resnet34/{k}": v for k, v in metrics.items()}
#             })
# 
#         if metrics['official_score'] > best_score:
#             best_score = metrics['official_score']
#             save_dict = get_raw().state_dict()
#             torch.save({'state_dict': save_dict, 'metrics': metrics, 'epoch': epoch}, 'models/best_pretrained_resnet34.pth')
#             del save_dict
#             print(f"  >> Saved new best model! Score: {best_score:.5f}")
# 
#         del train_pbar
#         gc.collect()
#         if DEVICE == 'cuda':
#             torch.cuda.empty_cache()
#         try:
#             import ctypes
#             ctypes.CDLL('libc.so.6').malloc_trim(0)
#         except Exception:
#             pass
# 
#     if run is not None:
#         if os.path.exists('models/best_pretrained_resnet34.pth'):
#             try:
#                 artifact = wandb.Artifact(name='pretrained-resnet34-best', type='model',
#                                           description='Best PretrainedResNet34UNet checkpoint',
#                                           metadata={'official_score': best_score, 'epochs': epochs})
#                 artifact.add_file('models/best_pretrained_resnet34.pth')
#                 run.log_artifact(artifact)
#                 print("Logged artifact 'pretrained-resnet34-best' to W&B.")
#             except Exception as e:
#                 print(f"Notice: Could not log artifact to W&B: {e}")
#         run.finish()
# 
#     print(f"\nTraining complete. Best official score: {best_score:.5f}")
# 
#     # Explicit memory cleanup
#     del model, optimizer, scheduler, scaler
#     gc.collect()
#     if DEVICE == 'cuda':
#         torch.cuda.empty_cache()
# 
#     return history, best_score
# 
## Execute PretrainedResNet34 training if FORCE_RETRAIN or checkpoint missing
# if CFG.FORCE_RETRAIN or not os.path.exists('models/best_pretrained_resnet34.pth'):
#     history_m3, best_m3 = train_pretrained_resnet34(
#         train_loader, val_loader, epochs=CFG.EPOCHS, warmup_epochs=5,
#         lr_warmup=5e-4, lr_encoder=1e-5, lr_decoder=1e-4)
# else:
#     print("PretrainedResNet34 checkpoint already exists. Set CFG.FORCE_RETRAIN = True to retrain.")
#     history_m3, best_m3 = {'epoch': []}, None


### 12.3 PretrainedResNet34UNet Training Visualization

In [ ]:
# if 'history_m3' in locals() and len(history_m3.get('epoch', [])) > 0:
#     plot_training_history(history_m3, 'Model 3: PretrainedResNet34UNet')
# else:
#     print("No training history for PretrainedResNet34UNet. (Set CFG.FORCE_RETRAIN = True to train).")


---

## 13. Comparative Analysis: All Three Two-Stage Architectures

### 13.1 Side-by-Side Comparison of the Three Two-Stage Combinations

A rigorous side-by-side comparison of all three architectures across all competition metrics,
parameter counts, and training characteristics.

In [ ]:
# def load_and_evaluate_model(model_name, model_class, model_init_kwargs,
#                             artifact_name, version='latest', local_path=None,
#                             fallback_artifacts=None, val_loader=None, classifier=None):
#     '''Download the model artifact from W&B (or use local fallback) and evaluate on validation split.
# 
#     Args:
#         model_name: Display name (e.g., 'ScratchUNet').
#         model_class: Model class type.
#         model_init_kwargs: Dict of constructor arguments.
#         artifact_name: W&B artifact name (e.g., 'scratch-unet-best').
#         version: W&B version tag (e.g., 'latest', 'v0', 'v1'). Defaults to 'latest'.
#         local_path: Local fallback file path if W&B is offline.
#         fallback_artifacts: Secondary artifact names if primary is not found.
#         val_loader: Validation DataLoader.
# 
#     Returns:
#         Dictionary of evaluated competition metrics and metadata.
#     '''
#     print("=" * 75)
#     print(f"  EVALUATING MODEL: {model_name}")
#     print(f"  W&B Artifact: {artifact_name}:{version}")
#     print("=" * 75)
# 
#     ckpt_path = None
#     # 1. Attempt to download specified version from W&B
#     if wandb is not None:
#         try:
#             ckpt_path = fetch_model_from_wandb(
#                 artifact_name=artifact_name,
#                 version=version,
#                 fallback_artifacts=fallback_artifacts
#             )
#         except Exception as e:
#             print(f"[W&B] Download notice: {e}")
# 
#     # 2. Local fallback if W&B was not reachable
#     if (ckpt_path is None or not os.path.exists(ckpt_path)) and local_path:
#         if os.path.exists(local_path):
#             print(f"[Local Fallback] Using existing checkpoint: {local_path}")
#             ckpt_path = local_path
#         else:
#             filename = os.path.basename(local_path)
#             for kdir in [f"/kaggle/input/{filename}", f"/kaggle/input/surfaceguard-checkpoints/{filename}"]:
#                 if os.path.exists(kdir):
#                     print(f"[Kaggle Input] Found checkpoint at: {kdir}")
#                     ckpt_path = kdir
#                     break
# 
#     if ckpt_path is None or not os.path.exists(ckpt_path):
#         raise FileNotFoundError(f"Checkpoint for '{model_name}' could not be downloaded from W&B ({artifact_name}:{version}) or found locally.")
# 
#     # Instantiate architecture and load state
#     model = model_class(**model_init_kwargs).to(DEVICE)
#     checkpoint = torch.load(ckpt_path, map_location=DEVICE)
#     state = checkpoint['state_dict'] if isinstance(checkpoint, dict) and 'state_dict' in checkpoint else checkpoint
#     state = {k.replace('module.', ''): v for k, v in state.items()}
#     model.load_state_dict(state, strict=False)
# 
#     total_params = sum(p.numel() for p in model.parameters())
# 
#     # Standardized evaluation on validation set
#     metrics = evaluate_model(model, val_loader, classifier=classifier, model_name=model_name)
#     metrics['model_name'] = model_name
#     metrics['parameters'] = total_params
#     metrics['artifact'] = f"{artifact_name}:{version}"
#     metrics['checkpoint_path'] = ckpt_path
#     return metrics
# 
# 
# def compare_models_from_wandb(val_loader, model_versions=None):
#     '''Compares all three models by downloading their best checkpoints from W&B.
# 
#     Args:
#         val_loader: Validation DataLoader.
#         model_versions: Optional dict mapping model names to specific artifact versions.
#                        Example: {'ScratchUNet': 'v0', 'ResAttentionUNet': 'latest', 'PretrainedResNet34UNet': 'latest'}
#                        If not specified, all models load ':latest' by default.
# 
#     Returns:
#         DataFrame containing comparative metrics across all three architectures.
#     '''
#     if model_versions is None:
#         model_versions = {}
# 
#     # Load shared Stage 1 SurfaceDefectClassifier
#     clf_path = None
#     if wandb is not None:
#         try:
#             clf_path = fetch_model_from_wandb(
#                 artifact_name=CFG.MODEL_ARTIFACTS['SurfaceDefectClassifier']['artifact'],
#                 version=model_versions.get('SurfaceDefectClassifier', CFG.MODEL_ARTIFACTS['SurfaceDefectClassifier']['default_version']),
#                 fallback_artifacts=CFG.MODEL_ARTIFACTS['SurfaceDefectClassifier']['fallback_artifacts']
#             )
#         except Exception as e:
#             print(f"[W&B] Classifier fetch notice: {e}")
#     if clf_path is None or not os.path.exists(clf_path):
#         clf_path = CFG.MODEL_ARTIFACTS['SurfaceDefectClassifier']['local_fallback']
#     classifier = None
#     if clf_path and os.path.exists(clf_path):
#         print(f"Loaded shared Stage 1 SurfaceDefectClassifier from: {clf_path}")
#         classifier = SurfaceDefectClassifier(in_channels=3, num_classes=4).to(DEVICE)
#         ckpt = torch.load(clf_path, map_location=DEVICE)
#         state = ckpt['state_dict'] if isinstance(ckpt, dict) and 'state_dict' in ckpt else ckpt
#         state = {k.replace('module.', ''): v for k, v in state.items()}
#         classifier.load_state_dict(state, strict=False)
#         classifier.eval()
# 
#     model_configs = [
#         {
#             'model_name': 'TwoStage-ScratchUNet (Classifier -> ScratchUNet)',
#             'model_class': ScratchUNet,
#             'model_init_kwargs': {'in_channels': 3, 'out_channels': 4},
#             'artifact_name': CFG.MODEL_ARTIFACTS['ScratchUNet']['artifact'],
#             'version': model_versions.get('ScratchUNet', CFG.MODEL_ARTIFACTS['ScratchUNet']['default_version']),
#             'local_path': CFG.MODEL_ARTIFACTS['ScratchUNet']['local_fallback'],
#             'fallback_artifacts': CFG.MODEL_ARTIFACTS['ScratchUNet']['fallback_artifacts'],
#         },
#         {
#             'model_name': 'TwoStage-ResAttentionUNet (Classifier -> ResAttentionUNet)',
#             'model_class': ResAttentionUNet,
#             'model_init_kwargs': {'in_channels': 3, 'out_channels': 4},
#             'artifact_name': CFG.MODEL_ARTIFACTS['ResAttentionUNet']['artifact'],
#             'version': model_versions.get('ResAttentionUNet', CFG.MODEL_ARTIFACTS['ResAttentionUNet']['default_version']),
#             'local_path': CFG.MODEL_ARTIFACTS['ResAttentionUNet']['local_fallback'],
#             'fallback_artifacts': CFG.MODEL_ARTIFACTS['ResAttentionUNet']['fallback_artifacts'],
#         },
#         {
#             'model_name': 'TwoStage-PretrainedResNet34 (Classifier -> PretrainedResNet34)',
#             'model_class': PretrainedResNet34UNet,
#             'model_init_kwargs': {'in_channels': 3, 'out_channels': 4, 'pretrained': False, 'with_classifier': True},
#             'artifact_name': CFG.MODEL_ARTIFACTS['PretrainedResNet34UNet']['artifact'],
#             'version': model_versions.get('PretrainedResNet34UNet', CFG.MODEL_ARTIFACTS['PretrainedResNet34UNet']['default_version']),
#             'local_path': CFG.MODEL_ARTIFACTS['PretrainedResNet34UNet']['local_fallback'],
#             'fallback_artifacts': CFG.MODEL_ARTIFACTS['PretrainedResNet34UNet']['fallback_artifacts'],
#         },
#     ]
# 
#     results = []
#     for cfg in model_configs:
#         try:
#             res = load_and_evaluate_model(
#                 model_name=cfg['model_name'],
#                 model_class=cfg['model_class'],
#                 model_init_kwargs=cfg['model_init_kwargs'],
#                 artifact_name=cfg['artifact_name'],
#                 version=cfg['version'],
#                 local_path=cfg['local_path'],
#                 fallback_artifacts=cfg['fallback_artifacts'],
#                 val_loader=val_loader,
#                 classifier=classifier
#             )
#             results.append(res)
#             # Memory cleanup after each model evaluation
#             gc.collect()
#             if DEVICE == 'cuda':
#                 torch.cuda.empty_cache()
#         except Exception as e:
#             print(f"Error evaluating {cfg['model_name']}: {e}")
# 
#     if not results:
#         print("No models could be evaluated from W&B or local checkpoints.")
#         return None
# 
#     # Comparative DataFrame
#     df_comp = pd.DataFrame([
#         {
#             'Model': r['model_name'],
#             'W&B Artifact': r['artifact'],
#             'Parameters': f"{r['parameters'] / 1e6:.1f} M",
#             'Official Score': r['official_score'],
#             'Foreground Dice (70%)': r['foreground_dice'],
#             'FP Silence Score (20%)': r['false_positive_score'],
#             'Empty Accuracy (10%)': r['empty_accuracy'],
#         }
#         for r in results
#     ])
# 
#     print("")
#     print("=" * 80)
#     print("  CROSS-MODEL PERFORMANCE COMPARISON (W&B ARTIFACT BENCHMARK)")
#     print("=" * 80)
#     print(df_comp.to_string(index=False))
#     print("=" * 80)
# 
#     # Visualization: Side-by-side metric comparison
#     fig, axes = plt.subplots(1, 3, figsize=(18, 6))
#     model_labels = [r['model_name'] for r in results]
#     x = np.arange(len(model_labels))
#     colors = ['#3498db', '#e74c3c', '#2ecc71']
# 
#     # Official Score
#     bars0 = axes[0].bar(x, [r['official_score'] for r in results], color=colors[:len(results)], width=0.55)
#     axes[0].set_title('Official Competition Score', fontsize=12, fontweight='bold')
#     axes[0].set_ylabel('Score')
#     axes[0].set_xticks(x)
#     axes[0].set_xticklabels(model_labels, fontsize=10)
#     for bar in bars0:
#         axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
#                      f"{bar.get_height():.5f}", ha='center', fontsize=10, fontweight='bold')
# 
#     # Foreground Dice
#     bars1 = axes[1].bar(x, [r['foreground_dice'] for r in results], color=colors[:len(results)], width=0.55)
#     axes[1].set_title('Foreground Dice (Defective Sheets)', fontsize=12, fontweight='bold')
#     axes[1].set_ylabel('Dice')
#     axes[1].set_xticks(x)
#     axes[1].set_xticklabels(model_labels, fontsize=10)
#     for bar in bars1:
#         axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
#                      f"{bar.get_height():.5f}", ha='center', fontsize=10, fontweight='bold')
# 
#     # FP Silence Score
#     bars2 = axes[2].bar(x, [r['false_positive_score'] for r in results], color=colors[:len(results)], width=0.55)
#     axes[2].set_title('False-Positive Silence Score', fontsize=12, fontweight='bold')
#     axes[2].set_ylabel('FP Score')
#     axes[2].set_xticks(x)
#     axes[2].set_xticklabels(model_labels, fontsize=10)
#     for bar in bars2:
#         axes[2].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
#                      f"{bar.get_height():.5f}", ha='center', fontsize=10, fontweight='bold')
# 
#     plt.suptitle('Comparative Analysis Across All Three Architectures (W&B Benchmarks)',
#                  fontsize=14, fontweight='bold')
#     plt.tight_layout()
#     plt.show()
# 
#     return df_comp
# 
# comparison_df = compare_models_from_wandb(val_loader)
# 


### 13.2 Training Curve Overlay

In [ ]:
## Overlay training curves for all three models
# has_full_curves = any(len(h.get('epoch', [])) > 1 for h in [history_m1, history_m2, history_m3])
# 
# if has_full_curves:
#     fig, axes = plt.subplots(1, 3, figsize=(18, 5))
# 
#     # Loss Curves
#     if len(history_m1['epoch']) > 1:
#         axes[0].plot(history_m1['epoch'], history_m1['train_loss'], 'b-', label='ScratchUNet Train', alpha=0.7)
#         axes[0].plot(history_m1['epoch'], history_m1['val_loss'], 'b--', label='ScratchUNet Val', alpha=0.7)
#     if len(history_m2['epoch']) > 1:
#         axes[0].plot(history_m2['epoch'], history_m2['train_loss'], 'r-', label='ResAttUNet Train', alpha=0.7)
#         axes[0].plot(history_m2['epoch'], history_m2['val_loss'], 'r--', label='ResAttUNet Val', alpha=0.7)
#     if len(history_m3['epoch']) > 1:
#         axes[0].plot(history_m3['epoch'], history_m3['train_loss'], 'g-', label='Pretrained Train', alpha=0.7)
#         axes[0].plot(history_m3['epoch'], history_m3['val_loss'], 'g--', label='Pretrained Val', alpha=0.7)
#     axes[0].set_title('Loss Convergence Comparison', fontsize=12, fontweight='bold')
#     axes[0].set_xlabel('Epoch')
#     axes[0].set_ylabel('Loss')
#     axes[0].legend(fontsize=8)
#     axes[0].grid(True, alpha=0.3)
# 
#     # Foreground Dice
#     if len(history_m1['epoch']) > 1:
#         axes[1].plot(history_m1['epoch'], history_m1['foreground_dice'], 'b-o', markersize=3, label='ScratchUNet')
#     if len(history_m2['epoch']) > 1:
#         axes[1].plot(history_m2['epoch'], history_m2['foreground_dice'], 'r-o', markersize=3, label='ResAttUNet')
#     if len(history_m3['epoch']) > 1:
#         axes[1].plot(history_m3['epoch'], history_m3['foreground_dice'], 'g-o', markersize=3, label='Pretrained ResNet34')
#     axes[1].set_title('Foreground Dice Progression', fontsize=12, fontweight='bold')
#     axes[1].set_xlabel('Epoch')
#     axes[1].set_ylabel('Dice Score')
#     axes[1].legend(fontsize=9)
#     axes[1].grid(True, alpha=0.3)
# 
#     # Official Score
#     if len(history_m1['epoch']) > 1:
#         axes[2].plot(history_m1['epoch'], history_m1['official_score'], 'b-o', markersize=3, label='ScratchUNet')
#     if len(history_m2['epoch']) > 1:
#         axes[2].plot(history_m2['epoch'], history_m2['official_score'], 'r-o', markersize=3, label='ResAttUNet')
#     if len(history_m3['epoch']) > 1:
#         axes[2].plot(history_m3['epoch'], history_m3['official_score'], 'g-o', markersize=3, label='Pretrained ResNet34')
#     axes[2].set_title('Official Score Progression', fontsize=12, fontweight='bold')
#     axes[2].set_xlabel('Epoch')
#     axes[2].set_ylabel('Official Score')
#     axes[2].legend(fontsize=9)
#     axes[2].grid(True, alpha=0.3)
# 
#     plt.suptitle('Cross-Model Training Dynamics Comparison', fontsize=14, fontweight='bold')
#     plt.tight_layout()
#     plt.show()
# else:
#     print("All three models were evaluated directly from existing checkpoints.")
#     print("Cross-architecture performance comparison table and bar charts displayed above.")

---

## 14. Competition Submission (Two-Stage ResAttentionUNet Pipeline)

### 14.1 Test Dataset and Inference Pipeline

The submission pipeline loads the best ResAttentionUNet checkpoint, applies 4-way Test-Time
Augmentation (Original, Horizontal Flip, Vertical Flip, Both), soft quadratic classifier gating,
per-channel thresholding, and morphological post-processing to generate the final submission CSV.

In [ ]:
class TestDefectDataset(Dataset):
    """Loads test images without ground-truth masks for inference."""
    def __init__(self, image_ids, image_dir):
        self.image_ids = list(image_ids)
        self.image_dir = image_dir

    def __len__(self):
        return len(self.image_ids)

    def __getitem__(self, idx):
        img_id = self.image_ids[idx]
        img_path = os.path.join(self.image_dir, f"{img_id}.jpg")
        image = cv2.imread(img_path)
        if image is None:
            raise FileNotFoundError(f"Image not found: {img_path}")
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        image = cv2.resize(image, (CFG.IMG_WIDTH, CFG.IMG_HEIGHT))
        image_tensor = torch.tensor(image, dtype=torch.float32).permute(2, 0, 1) / 255.0
        return image_tensor, img_id


def predict_with_tta(segmenter, classifier, images, cls_gate_tensor):
    """Run calibrated 4-way TTA inference with two-stage consensus gating.

    Solves the False Positive Explosion (41.31% -> ~12.5%):
      1. 4-Way TTA on ResAttentionUNet (Original, H-Flip, V-Flip, Both).
      2. 4-Way TTA on ResAttentionUNet Auxiliary Head (Trained with unweighted BCE, peak 0.6792).
      3. Standalone Stage 1 Classifier Gating.
      4. Two-Stage Consensus Gating: Uses ResAttentionUNet's own calibrated head as primary gate
         and enforces consensus (torch.minimum) with the Stage 1 Classifier to ensure clean sheets
         are never falsely inflated by positive-weight bias.
      5. Soft Quadratic Gating + Hard Clean-Sheet Cutoff (< 0.25 confidence -> zero mask).
    """
    img_h = torch.flip(images, dims=[3])
    img_v = torch.flip(images, dims=[2])
    img_hv = torch.flip(images, dims=[2, 3])

    # 1. 4-Way TTA on Segmenter
    out1 = segmenter(images, return_cls=True) if hasattr(segmenter, 'cls_head') else segmenter(images)
    out2 = segmenter(img_h, return_cls=True) if hasattr(segmenter, 'cls_head') else segmenter(img_h)
    out3 = segmenter(img_v, return_cls=True) if hasattr(segmenter, 'cls_head') else segmenter(img_v)
    out4 = segmenter(img_hv, return_cls=True) if hasattr(segmenter, 'cls_head') else segmenter(img_hv)

    m1 = out1[0] if isinstance(out1, tuple) else out1
    m2 = out2[0] if isinstance(out2, tuple) else out2
    m3 = out3[0] if isinstance(out3, tuple) else out3
    m4 = out4[0] if isinstance(out4, tuple) else out4

    p1 = torch.sigmoid(m1)
    p2 = torch.flip(torch.sigmoid(m2), dims=[3])
    p3 = torch.flip(torch.sigmoid(m3), dims=[2])
    p4 = torch.flip(torch.sigmoid(m4), dims=[2, 3])
    mask_probs = 0.25 * (p1 + p2 + p3 + p4)

    # Segmenter auxiliary head TTA (Calibrated with unweighted BCE, 0.6792 peak checkpoint)
    if isinstance(out1, tuple):
        c1 = torch.sigmoid(out1[1])
        c2 = torch.sigmoid(out2[1])
        c3 = torch.sigmoid(out3[1])
        c4 = torch.sigmoid(out4[1])
        seg_cls_p = 0.25 * (c1 + c2 + c3 + c4)
    else:
        seg_cls_p = None

    # Standalone Stage 1 Classifier 4-Way TTA
    if classifier is not None:
        clf1 = torch.sigmoid(classifier(images))
        clf2 = torch.sigmoid(classifier(img_h))
        clf3 = torch.sigmoid(classifier(img_v))
        clf4 = torch.sigmoid(classifier(img_hv))
        clf_p = 0.25 * (clf1 + clf2 + clf3 + clf4)
    else:
        clf_p = None

    # Two-Stage Consensus Gating:
    # If both models are present, use the conservative minimum so that a clean sheet
    # (seg_cls_p < 0.05) is NEVER falsely inflated by positive-weight bias.
    if seg_cls_p is not None and clf_p is not None:
        cls_p = torch.minimum(seg_cls_p, clf_p)
    elif seg_cls_p is not None:
        cls_p = seg_cls_p
    else:
        cls_p = clf_p

    # Soft Quadratic Gating + Hard Clean-Sheet Suppression
    if cls_p is not None and cls_gate_tensor is not None:
        cls_p_4d = cls_p.unsqueeze(-1).unsqueeze(-1)
        # Soft quadratic scaling
        ratio = torch.clamp(cls_p_4d / (cls_gate_tensor + 1e-7), max=1.0)
        mask_probs = mask_probs * (ratio ** 2)
        # Hard cutoff: if classifier confidence is below 0.25, zero out mask completely
        clean_cutoff = (cls_p_4d >= 0.25).float()
        mask_probs = mask_probs * clean_cutoff

    return mask_probs


### 14.2 Generate Competition Submission

In [ ]:
def generate_submission():
    """Generate the competition submission CSV using the calibrated Two-Stage Pipeline.

    Uses ResAttentionUNet (0.6792 peak checkpoint / newly trained 40-epoch DDP model)
    + Stage 1 SurfaceDefectClassifier (auto-downloaded from W&B if not local)
    with two-stage consensus gating and calibrated area thresholds.
    """
    print("=" * 70)
    print("  GENERATING COMPETITION SUBMISSION (CALIBRATED TWO-STAGE PIPELINE)")
    print("  Segmenter:  ResAttentionUNet (Stage 2 Segmenter)")
    print("  Classifier: SurfaceDefectClassifier (Stage 1 Consensus Gating)")
    print("  Post-Proc:  Calibrated Area Bounds & 4-Way TTA")
    print("=" * 70)

    # 1. Load submission template
    sub_df = pd.read_csv(CFG.SAMPLE_SUB_CSV)
    unique_ids = sub_df['image_id'].unique()
    print(f"Total test images: {len(unique_ids)}")

    # 2. Resolve ResAttentionUNet Checkpoint
    checkpoint_path = None
    local_candidates = [
        'models/best_resattunet.pth',
        'best_resattunet_score_0.6792.pth',
        'models/segmenter/best_res_attention_unet.pth',
        '/kaggle/input/surfaceguard-checkpoints/best_resattunet.pth',
        '/kaggle/input/surfaceguard-checkpoints/best_res_attention_unet.pth',
    ]
    for cand in local_candidates:
        if cand and os.path.exists(cand):
            checkpoint_path = cand
            print(f"[Local Checkpoint] Found segmenter weights at: {checkpoint_path}")
            break

    if checkpoint_path is None and wandb is not None:
        try:
            print("[W&B] Fetching ResAttentionUNet checkpoint from W&B...")
            checkpoint_path = fetch_model_from_wandb(
                artifact_name=CFG.MODEL_ARTIFACTS['ResAttentionUNet']['artifact'],
                version=CFG.MODEL_ARTIFACTS['ResAttentionUNet']['default_version'],
                fallback_artifacts=CFG.MODEL_ARTIFACTS['ResAttentionUNet']['fallback_artifacts']
            )
        except Exception as e:
            print(f"[W&B] Segmenter download notice: {e}")

    if checkpoint_path is None or not os.path.exists(checkpoint_path):
        raise FileNotFoundError("ResAttentionUNet checkpoint could not be found locally or on W&B.")

    # 3. Instantiate and load ResAttentionUNet
    model = ResAttentionUNet(in_channels=3, out_channels=4).to(DEVICE)
    print(f"Loading ResAttentionUNet weights from: {checkpoint_path}")
    checkpoint = torch.load(checkpoint_path, map_location=DEVICE)
    if isinstance(checkpoint, dict) and 'state_dict' in checkpoint:
        checkpoint = checkpoint['state_dict']
    checkpoint = {k.replace('module.', ''): v for k, v in checkpoint.items()}
    model.load_state_dict(checkpoint, strict=False)
    model.eval()
    print("[OK] ResAttentionUNet loaded successfully.")

    # 4. Resolve and load Stage 1 SurfaceDefectClassifier (Local candidates -> W&B Download)
    classifier = None
    clf_candidates = [
        'models/best_classifier.pth',
        'models/classifier/best_classifier.pth',
        '/kaggle/input/surfaceguard-checkpoints/best_classifier.pth',
    ]
    clf_path = None
    for cand in clf_candidates:
        if cand and os.path.exists(cand):
            clf_path = cand
            print(f"[Local Checkpoint] Found classifier weights at: {clf_path}")
            break

    # If not found locally, query W&B for best_classifier artifact
    if clf_path is None and wandb is not None:
        try:
            print("[W&B] Classifier not found locally. Downloading best classifier from W&B...")
            clf_path = fetch_model_from_wandb(
                artifact_name=CFG.MODEL_ARTIFACTS['SurfaceDefectClassifier']['artifact'],
                version=CFG.MODEL_ARTIFACTS['SurfaceDefectClassifier']['default_version'],
                fallback_artifacts=CFG.MODEL_ARTIFACTS['SurfaceDefectClassifier']['fallback_artifacts']
            )
        except Exception as e:
            print(f"[W&B] Classifier download notice: {e}")

    if clf_path and os.path.exists(clf_path) and 'SurfaceDefectClassifier' in globals():
        try:
            classifier = SurfaceDefectClassifier(in_channels=3, num_classes=4).to(DEVICE)
            c_ckpt = torch.load(clf_path, map_location=DEVICE)
            if isinstance(c_ckpt, dict) and 'state_dict' in c_ckpt:
                c_ckpt = c_ckpt['state_dict']
            c_ckpt = {k.replace('module.', ''): v for k, v in c_ckpt.items()}
            classifier.load_state_dict(c_ckpt, strict=False)
            classifier.eval()
            print(f"[OK] Stage 1 SurfaceDefectClassifier loaded successfully from: {clf_path}")
        except Exception as e:
            print(f"[Notice] Could not load classifier weights ({e}). Falling back to segmenter auxiliary head gating.")
            classifier = None
    else:
        classifier = None
        print("[Notice] Standalone classifier not found. Proceeding with segmenter auxiliary head gating.")

    # 5. Calibrated Area & Threshold Bounds
    calibrated_min_areas = {0: 300, 1: 400, 2: 600, 3: 800}
    th_tensor = torch.tensor([CFG.CLASS_THRESHOLDS[c] for c in range(4)], device=DEVICE).view(1, 4, 1, 1)
    cls_gate_tensor = torch.tensor([0.35, 0.35, 0.35, 0.35], device=DEVICE).view(1, 4, 1, 1)

    # 6. Test DataLoader
    test_ds = TestDefectDataset(unique_ids, CFG.TEST_IMAGE_DIR)
    test_loader = DataLoader(test_ds, batch_size=CFG.BATCH_SIZE, shuffle=False, num_workers=0)

    # 7. Run 4-Way TTA Inference
    predictions = {}
    with torch.no_grad():
        for images, img_ids in tqdm(test_loader, desc="Generating Predictions (TTA)"):
            images = images.to(DEVICE)
            probs = predict_with_tta(
                segmenter=model,
                classifier=classifier,
                images=images,
                cls_gate_tensor=cls_gate_tensor
            )
            preds_bin = (probs > th_tensor).cpu().numpy().astype(np.uint8)

            for i, img_id in enumerate(img_ids):
                cleaned = post_process_mask(
                    preds_bin[i],
                    min_areas=calibrated_min_areas,
                    max_areas=CFG.MAX_AREAS,
                    filter_geometry=True,
                    use_closing=True
                )
                for c, defect_name in enumerate(CFG.DEFECT_CLASSES):
                    channel_mask = cleaned[c]
                    rle_str = mask_to_rle(channel_mask) if channel_mask.sum() > 0 else " "
                    predictions[(img_id, defect_name)] = rle_str

    # 8. Format Submission CSV
    sub_df['mask_rle'] = sub_df.apply(
        lambda row: predictions.get((row['image_id'], row['defect_type']), " "), axis=1)

    sub_df.to_csv('submission.csv', index=False)

    non_empty = (sub_df['mask_rle'].str.strip() != "").sum()
    print("\n" + "=" * 70)
    print("  FINAL SUBMISSION GENERATION SUMMARY")
    print("=" * 70)
    print(f"  Submission file:       submission.csv")
    print(f"  Total prediction rows: {len(sub_df)}")
    print(f"  Detected defect rows:  {non_empty} / {len(sub_df)} ({non_empty/len(sub_df)*100:.2f}%)")
    print(f"  Clean sheet rows:      {len(sub_df) - non_empty} ({(len(sub_df)-non_empty)/len(sub_df)*100:.2f}%)")
    print("=" * 70)

generate_submission()


---

## 15. Conclusion and Summary

This notebook presented a complete pipeline for industrial steel surface defect detection,
comparing three progressively sophisticated segmentation architectures:

1. **ScratchUNet** -- Baseline establishing the U-Net approach for multi-label segmentation
2. **ResAttentionUNet** -- Advanced architecture incorporating residual scSE attention,
   ASPP with anisotropic strip convolutions, and additive attention gates
3. **PretrainedResNet34UNet** -- Transfer learning benchmark with 2-phase progressive fine-tuning

Key engineering decisions:
- **Focal + Dice ComboLoss** addresses extreme class imbalance (>50% clean sheets)
- **Defect-aware weighted sampling** ensures rare defects (B: 12x, A: 4x) appear in every batch
- **Soft quadratic classifier gating** suppresses false positives on clean sheets
- **Morphological post-processing** with per-class area and geometry constraints

The ResAttentionUNet model, trained 100% from scratch, serves as the primary submission model
with 4-way TTA and full post-processing pipeline.

---